# RSNA Knee — complément 320 (séries manquantes)

**Shard 1/2** `t1` (CPU, internet off). **320×320** uint8.

Les séries fluid 320 sont déjà le clone du 224 (mêmes UID, même crop, mêmes coupes). Ce kernel ne les refait pas.

UID **figés** depuis `manifest_t1_part` du crop 130 mm 224 (1 série non-fluide extra par étude, hors des 3 picks fluid). Même recette spatiale :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 % si spacing absent
2. bande profondeur **20–80 %**
3. resample **24** coupes, resize **320×320**, uint8, percentiles 1–99

Sortie dans `/kaggle/working/` :
- `volumes_crop130_320_t1_part1.zip`
- `manifest_t1_part1.csv`
- `RECIPE.json`

A déposer dans `data/crop130_320/` à côté des zip fluid.

In [ ]:
SHARD_ID = 1
N_SHARDS = 2

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        # debug local : CSV seulement, pas de DICOM
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
# Kaggle : train_series/<StudyUID>/<SeriesUID>/*.dcm
# (parfois a la racine COMP_DIR, parfois un cran plus bas)
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764","series":"1.2.826.0.1.3680043.8.498.12035938794439579756583835150081778633","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10021179701366615563817491713738306606","series":"1.2.826.0.1.3680043.8.498.36575741184530379556865901754021617448","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938","series":"1.2.826.0.1.3680043.8.498.61060692464231753798040762152369852425","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10025765742726180988537955593315007559","series":"1.2.826.0.1.3680043.8.498.12296387404690065979747592907647620416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585","series":"1.2.826.0.1.3680043.8.498.27845969797953115400959653384511165237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10039686090738454718354181337513293981","series":"1.2.826.0.1.3680043.8.498.34261736157116391641290714895482792113","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10061120822929703004201627784850458401","series":"1.2.826.0.1.3680043.8.498.51676946161186468941650577339931945923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922","series":"1.2.826.0.1.3680043.8.498.11587327685249364808271419937082956182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339","series":"1.2.826.0.1.3680043.8.498.83614010998801212494242431788612349272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10077803906105306230229342168184970097","series":"1.2.826.0.1.3680043.8.498.35461930354143940974642150297761639127","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10085187975213640798228717119613397941","series":"1.2.826.0.1.3680043.8.498.56139651590917417326146433456594976281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866","series":"1.2.826.0.1.3680043.8.498.33788855095027519865615604807327323172","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10090136633727522725960302994815319097","series":"1.2.826.0.1.3680043.8.498.10033217636189774155483719825495508042","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149","series":"1.2.826.0.1.3680043.8.498.10791858932231443712736629517371931282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10088120558600684332737135714636269365","series":"1.2.826.0.1.3680043.8.498.18366537086670924043033803636085833012","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10100718559177860518999368876392117659","series":"1.2.826.0.1.3680043.8.498.48773532541265941245766532225595532276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10096216489317602387766193675918767557","series":"1.2.826.0.1.3680043.8.498.46174887318465327763817364318575843548","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100","series":"1.2.826.0.1.3680043.8.498.17355150797297345667861720960301187303","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10121558854084271154265851933273260339","series":"1.2.826.0.1.3680043.8.498.11045479064503184057854236314537486044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10128954897472795975323353442305671586","series":"1.2.826.0.1.3680043.8.498.18250552347912320116850890825534074677","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10124063607403942330156450768015985834","series":"1.2.826.0.1.3680043.8.498.69652236847532170406852813979663373735","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10132169498032330070492330913783717864","series":"1.2.826.0.1.3680043.8.498.69214460339246844021907171343113321393","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586","series":"1.2.826.0.1.3680043.8.498.14991345816284084660769375156189038685","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314","series":"1.2.826.0.1.3680043.8.498.29750195135795939693896442301999293994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10148037169326559689992758180208780221","series":"1.2.826.0.1.3680043.8.498.70944668429307975193901939863950885929","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320","series":"1.2.826.0.1.3680043.8.498.11081869777466496980946756456652430317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10149967031679082348901055550415519790","series":"1.2.826.0.1.3680043.8.498.13428571725466320588810484325028997911","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104","series":"1.2.826.0.1.3680043.8.498.53787212012756349650425557865396727826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10158178668962379857908493434842864484","series":"1.2.826.0.1.3680043.8.498.23665757498452344201495732325817484986","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269","series":"1.2.826.0.1.3680043.8.498.39687308593816556791491064151179747304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10161011217774350674534600439166319161","series":"1.2.826.0.1.3680043.8.498.12461975233583204950838091362146598280","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10162720374291180196973998234794414415","series":"1.2.826.0.1.3680043.8.498.13304763566771283298203682646565039471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10181291283712008544475048457589664787","series":"1.2.826.0.1.3680043.8.498.45394881720475122092726559449343687345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10197565004992937722841731691775578390","series":"1.2.826.0.1.3680043.8.498.12654827639333114479747396372948670861","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10190506315646895064803936408630113507","series":"1.2.826.0.1.3680043.8.498.32512483619641874454128441636792579533","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10201197020583611347810070328854760801","series":"1.2.826.0.1.3680043.8.498.48950480256477480101177877164709371717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10209387047693448282446870911381660585","series":"1.2.826.0.1.3680043.8.498.11529309520105279537501270800913260335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10205592316907266136938526795826751183","series":"1.2.826.0.1.3680043.8.498.95536512734425776602520192097860235501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10183939841890895037058579455663880292","series":"1.2.826.0.1.3680043.8.498.38265444076831201006431018541125176189","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10211815027850731240126403602239595062","series":"1.2.826.0.1.3680043.8.498.44076700519249293923232310445902812533","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10239418182819270860819533058081041576","series":"1.2.826.0.1.3680043.8.498.22903464588641532341502679189979795734","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10218458664774126578027318307405130821","series":"1.2.826.0.1.3680043.8.498.43752233978238519514683268098957735692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10240237086492917349933491458808329326","series":"1.2.826.0.1.3680043.8.498.12807815284855746799876002867743236501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10252121617380434672930525780360219531","series":"1.2.826.0.1.3680043.8.498.74524777621297243054896376044173013085","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10256441455107860474283716992657597873","series":"1.2.826.0.1.3680043.8.498.12279222319660760130623733767347406207","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10254828365033984708960944586564394286","series":"1.2.826.0.1.3680043.8.498.12427381584228298885293688963574323648","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10262024906324456863812446480254600405","series":"1.2.826.0.1.3680043.8.498.85985794862127095585192707633430420116","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10271567891603078238024189515805908561","series":"1.2.826.0.1.3680043.8.498.88773960893971809432349013133060938098","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10275265831651276229168539541982478639","series":"1.2.826.0.1.3680043.8.498.10641081243353851892021905881692814843","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10243300429607935614655219482457289307","series":"1.2.826.0.1.3680043.8.498.73782959974044202287034378839154139690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029","series":"1.2.826.0.1.3680043.8.498.77766976116670551979942629232553368416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10301089872627496769249366231947887097","series":"1.2.826.0.1.3680043.8.498.93495652587822368992224521369968101882","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10284446538566255572004044331831825072","series":"1.2.826.0.1.3680043.8.498.68594777197272736846799007294149623402","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10299396087685174806129850512856451005","series":"1.2.826.0.1.3680043.8.498.98733876567133671733705168596969228500","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052","series":"1.2.826.0.1.3680043.8.498.55427564251500418323672579441288025575","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10321461243762453573005633063753741725","series":"1.2.826.0.1.3680043.8.498.99382992121129708528636999373528210689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10310831394568337664174691281092566272","series":"1.2.826.0.1.3680043.8.498.65258925798656967203843242611950695729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10314240078685804110739316196999571466","series":"1.2.826.0.1.3680043.8.498.63747656240463144163860613478456312396","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10321702681363532511953946967334431952","series":"1.2.826.0.1.3680043.8.498.40605939406486221410792371410610189705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10336221220793312503236159680373829726","series":"1.2.826.0.1.3680043.8.498.10355624694647950956997008891539958632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10339654093023509742663983374586998753","series":"1.2.826.0.1.3680043.8.498.41872765926524756990122192389840354782","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10322853084122920408377349136830993432","series":"1.2.826.0.1.3680043.8.498.17697711597859086507295300312348615087","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10349368332722015923160611226877741306","series":"1.2.826.0.1.3680043.8.498.13323518992038409474498100312023197493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10352044313309892076985642485759368197","series":"1.2.826.0.1.3680043.8.498.64422891086330237435156873804773169359","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10351460911387106879135924409514835385","series":"1.2.826.0.1.3680043.8.498.10005047919793760894599773106036876356","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10327608571317884761792196888985963332","series":"1.2.826.0.1.3680043.8.498.39506465627529543765264712561750329245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10378535286770237477615181844634903700","series":"1.2.826.0.1.3680043.8.498.69029661178389771772074972317839685178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10386894500329189457070808193345170500","series":"1.2.826.0.1.3680043.8.498.10843417049434653564450890800518831128","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10371162371589200772388569623043753617","series":"1.2.826.0.1.3680043.8.498.12819425291674675672062168009589326029","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387383212921524066516723428665512341","series":"1.2.826.0.1.3680043.8.498.24366922563288150531040273712070930123","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10361030765327974510262894062047216433","series":"1.2.826.0.1.3680043.8.498.17148003617765316760015327235903013723","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10405028141768633100332386157519916334","series":"1.2.826.0.1.3680043.8.498.11220000050750131038029343935480763721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10401813253995823397883215400000450724","series":"1.2.826.0.1.3680043.8.498.59679591899172017478992721044886020146","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434","series":"1.2.826.0.1.3680043.8.498.61055644564394168698042963913913650719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10417042894702157309437002434676100667","series":"1.2.826.0.1.3680043.8.498.38473866058891960033923007115035930761","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10419958821528108176389641942168280311","series":"1.2.826.0.1.3680043.8.498.37879332041830925434341172854741834145","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10427893579790102216163565692202919399","series":"1.2.826.0.1.3680043.8.498.75497736695040760603407848125295518500","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10391178814963564201551003358372148419","series":"1.2.826.0.1.3680043.8.498.40422448144321049235911137816098490107","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10423586531262405323219596291234698626","series":"1.2.826.0.1.3680043.8.498.27368888445605106720428289247143067283","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10432568348356281020515782178064869805","series":"1.2.826.0.1.3680043.8.498.12862771526670808608904605078864426170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10440858363080910322920496514558347561","series":"1.2.826.0.1.3680043.8.498.46166395554568668511323177656838697107","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10422130685453254363700365908336246863","series":"1.2.826.0.1.3680043.8.498.39936343059064158196914564351933422105","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10445893053733106322562086357169418409","series":"1.2.826.0.1.3680043.8.498.64362202088851071176361650586336727570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10454875772176439944597553400963295397","series":"1.2.826.0.1.3680043.8.498.88863725628634972901064023858499017410","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10491012711700179118010977263805828326","series":"1.2.826.0.1.3680043.8.498.86569521473200743674638383158989750021","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10470282959537049979906878778884086884","series":"1.2.826.0.1.3680043.8.498.10055871012601176668266024881216265472","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10475397432011699698577978597417345091","series":"1.2.826.0.1.3680043.8.498.79689803747379857208090951595847632334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10495281520660483987888839682017074584","series":"1.2.826.0.1.3680043.8.498.60937579538433294438916040161180815564","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10459044164648238749799867924228983674","series":"1.2.826.0.1.3680043.8.498.68262482937445256099604837922904885569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10502440129091559437492859441158882078","series":"1.2.826.0.1.3680043.8.498.92854885655383840904925782352020836991","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10519480654942067141173172019311858213","series":"1.2.826.0.1.3680043.8.498.11802124024554872991296416161742277614","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10499265498716562276605625682538326716","series":"1.2.826.0.1.3680043.8.498.83867494661825167554590143045762580834","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10524164196949381411165626176732559196","series":"1.2.826.0.1.3680043.8.498.32589572285444891635349935440720629929","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10521372086358356219840179458708706015","series":"1.2.826.0.1.3680043.8.498.53050132866424386883355609746475534307","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10511712617129216774539441890488742560","series":"1.2.826.0.1.3680043.8.498.60541181263171472456819580794227537189","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10532708800468111850607752497432656694","series":"1.2.826.0.1.3680043.8.498.44112310880101671509001224863586298551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10534260101860670685224894870019533364","series":"1.2.826.0.1.3680043.8.498.50581877347630891477845107780255805465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10541029960718591409318399557079055927","series":"1.2.826.0.1.3680043.8.498.92679627845520691792217237021162883135","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10537503354035115750949427337500909327","series":"1.2.826.0.1.3680043.8.498.11162278440235061156649335972703209840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10541227283184832783436242773084724216","series":"1.2.826.0.1.3680043.8.498.12537729854295691260415214247516064738","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10540882690103231159064894952851205573","series":"1.2.826.0.1.3680043.8.498.71756190113907904121735845334132837824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736","series":"1.2.826.0.1.3680043.8.498.10637925259235402880926451594821918687","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10546313646515875953943156661172963953","series":"1.2.826.0.1.3680043.8.498.79194086061555794406311099605816715714","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10587511485561679190576188107774736852","series":"1.2.826.0.1.3680043.8.498.55328504129708534609935594295450814413","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10580260109117947709069163110100632085","series":"1.2.826.0.1.3680043.8.498.36989948936004747849648637376162205239","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10591507262178959553431147886647585916","series":"1.2.826.0.1.3680043.8.498.11898480838154899814371688171626543958","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10574706940531650593705137704922433109","series":"1.2.826.0.1.3680043.8.498.35085809899550840803765051471609153495","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10590148272001480024086117269799629816","series":"1.2.826.0.1.3680043.8.498.58258467705489100594537722261484038632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10601062881015669567737602031338578146","series":"1.2.826.0.1.3680043.8.498.46635940681606147009204301777014526298","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196","series":"1.2.826.0.1.3680043.8.498.47432904807867882894652932038747744662","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10614508096225944703643482200835890514","series":"1.2.826.0.1.3680043.8.498.90632270480377325950680478185169525482","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10624417936475721846835761375334601839","series":"1.2.826.0.1.3680043.8.498.50344353668926430475044815731509082000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10626411974489938664325062396760010912","series":"1.2.826.0.1.3680043.8.498.66386136914980875728127427361173345893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10627795990821506536342369227971125477","series":"1.2.826.0.1.3680043.8.498.24872275347880480636927394543888122406","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10632177122947017620482277062226288392","series":"1.2.826.0.1.3680043.8.498.80699338809022799642244818179504752720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10619486897279334206912764495067756449","series":"1.2.826.0.1.3680043.8.498.50176703731215284144470430416209475585","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10599128554628698518723223547497007137","series":"1.2.826.0.1.3680043.8.498.10457722103437597353351370290626803648","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10653432159060774337155662807644609967","series":"1.2.826.0.1.3680043.8.498.57733963522183468140271255185222670234","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10657506540109137829814058249895525831","series":"1.2.826.0.1.3680043.8.498.58360552985708904581527567828634759720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10652685418462611085839874321267337079","series":"1.2.826.0.1.3680043.8.498.79982534358896588607326098685753566714","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10675139840690240606562530926349987331","series":"1.2.826.0.1.3680043.8.498.19987097303914992766164853215772846169","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10673300644364559354643106316993112731","series":"1.2.826.0.1.3680043.8.498.23934316670432021861006355081092376905","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10659781998801845781246073928620795082","series":"1.2.826.0.1.3680043.8.498.18030423512659336927032241144123924558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10684020894033412667820478260168097071","series":"1.2.826.0.1.3680043.8.498.20363333326596567038579935247878308902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10702284568670902901846767166679293654","series":"1.2.826.0.1.3680043.8.498.10096488075186477560498359249801245146","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10648346793256473462304051626410900340","series":"1.2.826.0.1.3680043.8.498.12592854543953428015999239918349110334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10713285016170190922777298047146228811","series":"1.2.826.0.1.3680043.8.498.13231842701790643576556126227684228451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10724604223190030113012801427752979879","series":"1.2.826.0.1.3680043.8.498.98353629474736025133984065113480552032","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10688670034893539180306173203903350286","series":"1.2.826.0.1.3680043.8.498.11227158002995688802511103888794034414","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10728448245470003618625208276233778895","series":"1.2.826.0.1.3680043.8.498.93748893051701273966508030840509082255","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10731612202208916303411280624156291171","series":"1.2.826.0.1.3680043.8.498.84047126631088959343658030351228929200","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10738640862162795323010965765443420384","series":"1.2.826.0.1.3680043.8.498.33682882029868486459397572658883391198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10740857795874468664161171813136103519","series":"1.2.826.0.1.3680043.8.498.41552951872646988864818439719132778517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10750145216793948201921667945805555093","series":"1.2.826.0.1.3680043.8.498.80578660557040452814256943827961916646","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10761658878615069270304502752374858610","series":"1.2.826.0.1.3680043.8.498.88089865919457051504930941581723273203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10765617676329232046176300833850901740","series":"1.2.826.0.1.3680043.8.498.70040316994257134083028919382038280463","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10751624265599812767108610284996090628","series":"1.2.826.0.1.3680043.8.498.11238910502477208988271836351516439940","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10774147135673365265606645781662088745","series":"1.2.826.0.1.3680043.8.498.65964041770453362610397494114365233570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334","series":"1.2.826.0.1.3680043.8.498.10719231877551676984182272268301940778","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10759288702627176691392048851622294054","series":"1.2.826.0.1.3680043.8.498.10490278258578942118557141257163634182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10790689419760453870458170966064168002","series":"1.2.826.0.1.3680043.8.498.43757905335322113351103990342243600218","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10769126730281890056368854871900324448","series":"1.2.826.0.1.3680043.8.498.62898453840703005058126143528997470412","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10803283287658556463524649793955055077","series":"1.2.826.0.1.3680043.8.498.26614509798408297901598406965975999411","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10824420630965162625242888916805538597","series":"1.2.826.0.1.3680043.8.498.26265764259282212122888165424371516517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10812894672739619566834776054635588406","series":"1.2.826.0.1.3680043.8.498.90519725310611999715844918954073082345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10839126901366636822676830560774505865","series":"1.2.826.0.1.3680043.8.498.10634009940138707006767243395628345306","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10830288389793651695251406408666084306","series":"1.2.826.0.1.3680043.8.498.77381608009576639152885941120157769648","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10851562687641986545442408470139368318","series":"1.2.826.0.1.3680043.8.498.26084376788783968382054780494707921750","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839","series":"1.2.826.0.1.3680043.8.498.11095855317733507937819924915747155492","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10848350333764008140459854470293737650","series":"1.2.826.0.1.3680043.8.498.19804498360205813332100165708113470938","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10855290714402117092990651362697310270","series":"1.2.826.0.1.3680043.8.498.94978801886429346268220177480964043900","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10857771747813858957718058322384884551","series":"1.2.826.0.1.3680043.8.498.27538996055361001853140012053959794921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10867622488467375444427908425709367385","series":"1.2.826.0.1.3680043.8.498.67787477230257805958675985329793735963","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10854012268828835845581966634225522931","series":"1.2.826.0.1.3680043.8.498.50018448175582266704642805250088437895","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10880553082640412137979450100928382021","series":"1.2.826.0.1.3680043.8.498.80118566037581465106578031317627946407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10895521387894821238259709507311341758","series":"1.2.826.0.1.3680043.8.498.30032732698875993911917722072464285441","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10902166579916526176697193295886992412","series":"1.2.826.0.1.3680043.8.498.49977660763019599396861418472683409743","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10918580785296934228715514366408269433","series":"1.2.826.0.1.3680043.8.498.12608533145340432493646814911000771508","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10917389623891555306584069980815338466","series":"1.2.826.0.1.3680043.8.498.23504633479098554642941085243835025076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10920269679387171659375091645187772303","series":"1.2.826.0.1.3680043.8.498.92455600591402675895558733986191140913","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10930374192038896613513882597332665403","series":"1.2.826.0.1.3680043.8.498.57102998217286118624861770119728259811","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10924995713871537837233985073507509566","series":"1.2.826.0.1.3680043.8.498.61788493550079348013362814757966573607","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10886583574130697346788286647281863932","series":"1.2.826.0.1.3680043.8.498.10199464076352134980970281334377107311","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10937181824204907065071760564745305722","series":"1.2.826.0.1.3680043.8.498.23065195904724610449900822761164486278","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10937585701850306334458420860276069606","series":"1.2.826.0.1.3680043.8.498.62394328567191720608628909884960910997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10959153852077371752490787490374555514","series":"1.2.826.0.1.3680043.8.498.13303376136598938661136320384488159058","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10939964887585312209861601414669494409","series":"1.2.826.0.1.3680043.8.498.17092201598597623706892346405907008969","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10964287649604862436796341368993301245","series":"1.2.826.0.1.3680043.8.498.11831194092603685651864824383584453938","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10932818400202426453525918723737011737","series":"1.2.826.0.1.3680043.8.498.10932325670157464624260915640014162134","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10975347305662629858093628672419901528","series":"1.2.826.0.1.3680043.8.498.10917762304353522942663331343716035242","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10981062990502278795793220223076241424","series":"1.2.826.0.1.3680043.8.498.52585316187335441610377926514859853325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994848378662155711176898659205016445","series":"1.2.826.0.1.3680043.8.498.38840528433942468998998383863038214460","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994899269366718796438389394954844979","series":"1.2.826.0.1.3680043.8.498.19948375643720027039216035207659162686","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10992875360571368325079907908221899699","series":"1.2.826.0.1.3680043.8.498.11193344262215846482320197273402127392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10998144118962608870602180672090998054","series":"1.2.826.0.1.3680043.8.498.10140887816944112044742717004984233349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11014091506864318694851337553197477906","series":"1.2.826.0.1.3680043.8.498.11214645231035703782350271821706221616","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11010004366885474301784150582039137111","series":"1.2.826.0.1.3680043.8.498.53586115095368197321625265173708372517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11041694833778404465582610810664391088","series":"1.2.826.0.1.3680043.8.498.11739682885624380260977612130921260965","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11006241238894862275805535376070111510","series":"1.2.826.0.1.3680043.8.498.90615948084786827528167792650334160680","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11052720536194493057619720620119095739","series":"1.2.826.0.1.3680043.8.498.70011032679948221644257986251535029294","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11043543420859531726426521315399258528","series":"1.2.826.0.1.3680043.8.498.83483003583826357781897718028453144065","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11060834081463491262807709963877480026","series":"1.2.826.0.1.3680043.8.498.12751804616396515154865178948161070577","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11067200656721609390997819562401839124","series":"1.2.826.0.1.3680043.8.498.77174146563970283436603471770263283736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11050604958275820465127935558416408008","series":"1.2.826.0.1.3680043.8.498.12069770320340355266813912499570590170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11077406555624926104628200252710914930","series":"1.2.826.0.1.3680043.8.498.14296237954597402119768864897549965304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11045459770905584149950259201414074545","series":"1.2.826.0.1.3680043.8.498.25242199583610751323398221132559743173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11090884548990390279582832205294943635","series":"1.2.826.0.1.3680043.8.498.33924476870655602036880022285772217520","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11113442398569287228475456000446876768","series":"1.2.826.0.1.3680043.8.498.21599274942516200255395086865270640569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11100224438877567316062152508871079233","series":"1.2.826.0.1.3680043.8.498.15309294241026255275263161991632901951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11080651962904815250736183346281427445","series":"1.2.826.0.1.3680043.8.498.10741632889324923303259748193086923516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11110575656494405149601490027444484170","series":"1.2.826.0.1.3680043.8.498.44120208823716066059423069825729660236","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11122485005317809892694992204053124213","series":"1.2.826.0.1.3680043.8.498.54290044862559459108971022194049427882","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11128099896284546522122813576483001027","series":"1.2.826.0.1.3680043.8.498.61808734669982649590623048420737073303","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11125718814639515292482773418372267821","series":"1.2.826.0.1.3680043.8.498.62895631316143783707883133619871572680","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11131210140372132238166068091728877219","series":"1.2.826.0.1.3680043.8.498.31751564370124535591795950564107486369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11150193626240150189232364321568347937","series":"1.2.826.0.1.3680043.8.498.11167260759049759079709904667320379831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11150751959089942633754454715290410758","series":"1.2.826.0.1.3680043.8.498.56310913022297572205441545603814825893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11154238555579687338157479416363307605","series":"1.2.826.0.1.3680043.8.498.40513931798213795972848116507640201531","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11148939654362681630811787929668902046","series":"1.2.826.0.1.3680043.8.498.99848626726594457093705367272619998372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11159546221032123495135103648570690656","series":"1.2.826.0.1.3680043.8.498.87386226856144144002989951755158731329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11180457573346204459956051956213528997","series":"1.2.826.0.1.3680043.8.498.46427302339464780388090657542662081088","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11184968672176462852561645279750399747","series":"1.2.826.0.1.3680043.8.498.49022340225674703108931531297813342349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286","series":"1.2.826.0.1.3680043.8.498.12929930691966163180656740586659349690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11207013225631251718312189933025529824","series":"1.2.826.0.1.3680043.8.498.12972299354368510426914890245705104304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11191015483427903713800997920940844550","series":"1.2.826.0.1.3680043.8.498.30980863675935941008105037879776019329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11201589885283493748455257854104771618","series":"1.2.826.0.1.3680043.8.498.42898491770454578451164744293383945596","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11218782547268547937641080283066300678","series":"1.2.826.0.1.3680043.8.498.67734248481701094706781140259865716869","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11208663314213048426907645330143931419","series":"1.2.826.0.1.3680043.8.498.75994728789601964816909657126630474498","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11214158717183172156320144105486247892","series":"1.2.826.0.1.3680043.8.498.78314978491876402312622528533451128022","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11211598263863419448236436640277320409","series":"1.2.826.0.1.3680043.8.498.24764255313329471182002636264973414185","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11221318524515817857172161854667955115","series":"1.2.826.0.1.3680043.8.498.81940377421637831450024997832980055932","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11231822445970527276203191047278744341","series":"1.2.826.0.1.3680043.8.498.54486856205337405948582883309205896310","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11257869433929994221899994675448655047","series":"1.2.826.0.1.3680043.8.498.84838559811084146139953356627934540579","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11254890841796835340329521299102281678","series":"1.2.826.0.1.3680043.8.498.72872783340662865226615382588525834289","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11244115315611630362929576340631670750","series":"1.2.826.0.1.3680043.8.498.94702337448697331337350752611637995507","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11268635423021790546063903457498005204","series":"1.2.826.0.1.3680043.8.498.12598169077787511889922564887005305814","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11262944211104069231500934582412003273","series":"1.2.826.0.1.3680043.8.498.28769030233927332109579624359642442703","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11276732020240247755618413418003401291","series":"1.2.826.0.1.3680043.8.498.32013233467326527147343878483254663222","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11226582925516370364253861721606580867","series":"1.2.826.0.1.3680043.8.498.31202608765349538559685394062266061309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11282706861378379965012380644125662529","series":"1.2.826.0.1.3680043.8.498.37671375170361167645864598416838388843","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11275711698468745807758222802302685022","series":"1.2.826.0.1.3680043.8.498.81755020382869351143096306877279704971","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580","series":"1.2.826.0.1.3680043.8.498.14912947740172635156351119156260949957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11277278232377185347750793496260885546","series":"1.2.826.0.1.3680043.8.498.76793040700794000836585769118948148556","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11294885816514924203337081832122619605","series":"1.2.826.0.1.3680043.8.498.45977835813484597706832123660553037583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11312798102268182343541500463824266929","series":"1.2.826.0.1.3680043.8.498.83914092883219099799906239909489985421","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11302750434639976191333636496428935632","series":"1.2.826.0.1.3680043.8.498.11115904669609656163554793356410837754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11296867854260286881418279496022384461","series":"1.2.826.0.1.3680043.8.498.35670558401211449970309107264580233401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11331824020096159502562313322049966712","series":"1.2.826.0.1.3680043.8.498.13201132079462947912734182972220430265","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11334776960319816844935217444817103072","series":"1.2.826.0.1.3680043.8.498.12616636289723947946455686235985091594","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11315871586936178043350206344448851055","series":"1.2.826.0.1.3680043.8.498.54646240049519448387798356560719469069","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11345369648571407541082013397591193473","series":"1.2.826.0.1.3680043.8.498.10294048115798777353764436235282788705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11338962847958062668991874160027887372","series":"1.2.826.0.1.3680043.8.498.10284219920313233626530492412896345428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11371400503360468962117579439852064607","series":"1.2.826.0.1.3680043.8.498.71824218488830583118310862947161725016","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11362547519125634779422991321337283241","series":"1.2.826.0.1.3680043.8.498.13397418997381548880920459500334445458","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11383056981606331444758741778014516431","series":"1.2.826.0.1.3680043.8.498.10448829801014088904393484365639216481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11394861774586578112386386342670500749","series":"1.2.826.0.1.3680043.8.498.39216590551819379672243675532955966772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11386074316342852649653530230563922899","series":"1.2.826.0.1.3680043.8.498.47396621144976202956667392069364847338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11352722814008170661344291337061231555","series":"1.2.826.0.1.3680043.8.498.43793062896885921272768660228349449704","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11413427362266862411525989114324631512","series":"1.2.826.0.1.3680043.8.498.66220482246531160063008538719287136315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11406304618685790128709325242250648417","series":"1.2.826.0.1.3680043.8.498.29499482019790891630430839775873619039","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11425683153901568541658945026861110110","series":"1.2.826.0.1.3680043.8.498.10427581481017218690762910333730454263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11432815311029789072253610823554308183","series":"1.2.826.0.1.3680043.8.498.73622998013540223372038585897105183121","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11420670844338686032030860787688858057","series":"1.2.826.0.1.3680043.8.498.52018822497651119443124728003808777853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11396451979734570993257791565721573849","series":"1.2.826.0.1.3680043.8.498.87559177507132843964549129602664265002","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11440217845998767853986099058965644463","series":"1.2.826.0.1.3680043.8.498.83689987005188717255706495839875078685","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11448865770537436004548718998804636234","series":"1.2.826.0.1.3680043.8.498.23854356002166046342050769882483902981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11446264194433135282793848490754001623","series":"1.2.826.0.1.3680043.8.498.13477821137105966308865850388927075306","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11451981256762378203455481812714370193","series":"1.2.826.0.1.3680043.8.498.98209832394374463992561722036409650099","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11458488293461896930750018852127694875","series":"1.2.826.0.1.3680043.8.498.77413543786187024242220774935652913136","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11465586454035959390661499120751232686","series":"1.2.826.0.1.3680043.8.498.35275627587044578602950791719732295150","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11472441320199233244785757534656853558","series":"1.2.826.0.1.3680043.8.498.12875961949937140860970436891232467648","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11468812519969607938527116557104410121","series":"1.2.826.0.1.3680043.8.498.18570506313772761907578099172720027844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11487799220518102363643849548556099719","series":"1.2.826.0.1.3680043.8.498.39214302777137833295623618076483815000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11495133447993934376048707763538781434","series":"1.2.826.0.1.3680043.8.498.86985551635887653095455398165431767796","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11498239309891927151456629915927889165","series":"1.2.826.0.1.3680043.8.498.12014881867631165354703273230911409710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11504079342934513987748646905909982588","series":"1.2.826.0.1.3680043.8.498.83661754057304113572066158751101730432","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11498845504486569595587386245872511981","series":"1.2.826.0.1.3680043.8.498.90924773257875016151787718474536008332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11477583622065195478886525904745372063","series":"1.2.826.0.1.3680043.8.498.50382017273511013345838568800924359882","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11515987024418648685676369132061127664","series":"1.2.826.0.1.3680043.8.498.84682505725425327811157919620337162834","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11506856887791286395737465774061395929","series":"1.2.826.0.1.3680043.8.498.64986803803447779055032805486812482195","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11516842786271998149681118182571061813","series":"1.2.826.0.1.3680043.8.498.76091201572891198793136559565129502226","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11521971289581057766555206729274639736","series":"1.2.826.0.1.3680043.8.498.11944848771632898890165082868184614896","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11543622448213658856759993374276266341","series":"1.2.826.0.1.3680043.8.498.46494884964239416726457356700536210957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11533971444023236216745801026152089075","series":"1.2.826.0.1.3680043.8.498.91779232878306879025700337854014792298","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11546127923715190914128277599849591183","series":"1.2.826.0.1.3680043.8.498.79348145937688099854927598677633897221","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935","series":"1.2.826.0.1.3680043.8.498.54680010975301762365724523486752133418","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11530446425918247469882351709851883983","series":"1.2.826.0.1.3680043.8.498.30834998619975301005268800293577539235","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840","series":"1.2.826.0.1.3680043.8.498.74490634866067804318496348509978724860","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11569355546217846610131708883985793793","series":"1.2.826.0.1.3680043.8.498.79240019274187933792578935356797109245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11571057273590444914079281238002921868","series":"1.2.826.0.1.3680043.8.498.19061733740401589340572749195072157689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11574667600340420151274680172753922967","series":"1.2.826.0.1.3680043.8.498.12553924478922948226789962457454342732","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11586616391541918134110642641868079265","series":"1.2.826.0.1.3680043.8.498.16238345368341750914865489782399196737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11551347885357242022697196552822735795","series":"1.2.826.0.1.3680043.8.498.11457115155245525023726804254801073093","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11579571696748051185342754881791715610","series":"1.2.826.0.1.3680043.8.498.78334325675611422740930744592516526023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11590096889491409910425456896187293515","series":"1.2.826.0.1.3680043.8.498.57958159116471122325226234815888549683","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11592838054859712596515534341207971997","series":"1.2.826.0.1.3680043.8.498.10788714070459964231154884978901427160","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11598606533748611505510113033734692457","series":"1.2.826.0.1.3680043.8.498.12364031433876658179598449149032650447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11596665780617412984766822528862741584","series":"1.2.826.0.1.3680043.8.498.11169158928779880798108509329821926091","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11604138460812393624691789275263090511","series":"1.2.826.0.1.3680043.8.498.12977043085895407399669296659191222802","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11610585947047502123824835405421129552","series":"1.2.826.0.1.3680043.8.498.11159120959471023026993735239702011942","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11619883329923252065839671636992770849","series":"1.2.826.0.1.3680043.8.498.12103311282035389519321944384948209864","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11629659616217352895144469980856259097","series":"1.2.826.0.1.3680043.8.498.61301729143628841277262615707112508439","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11642678020345468499556230871411308014","series":"1.2.826.0.1.3680043.8.498.57180692012960681862417646036764525166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11641866550826314411250696478180861612","series":"1.2.826.0.1.3680043.8.498.47200298337002943087005842312118552120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11652570369213315569275956969935617843","series":"1.2.826.0.1.3680043.8.498.96832375822372017531066290669279596022","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11647595587705638372290862475518195181","series":"1.2.826.0.1.3680043.8.498.62162141211097040410337096498348835310","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11605266731872772432220415835172557961","series":"1.2.826.0.1.3680043.8.498.48672777476739796105883375818515637609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11661333712307504012375232163708358493","series":"1.2.826.0.1.3680043.8.498.23504403474054134514787919548193116501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11679981444785137570894759919380932611","series":"1.2.826.0.1.3680043.8.498.82155997852397353959734062953503131803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11685904538570360040472064238371379681","series":"1.2.826.0.1.3680043.8.498.46401983179415953510877934290216116335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11691245281415120685024733512803575288","series":"1.2.826.0.1.3680043.8.498.42601900544539152801710048303482572988","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11674925658184069242322832524274428408","series":"1.2.826.0.1.3680043.8.498.10631379905248886281584110672360292337","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11703195077261884009442719578918319962","series":"1.2.826.0.1.3680043.8.498.29855462953845010031276834438087976321","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11709909941833245598246010905322129256","series":"1.2.826.0.1.3680043.8.498.82632948695916629786257411046038972274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11720535027022778971695130536590302635","series":"1.2.826.0.1.3680043.8.498.35976323934752293141311648322076736464","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11716730437469973434747875401501334429","series":"1.2.826.0.1.3680043.8.498.44456811837595069089378687127077492422","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11655692245049710970881619360721342623","series":"1.2.826.0.1.3680043.8.498.11844412364254694200687885348234817830","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11722185671504770687850569082348083612","series":"1.2.826.0.1.3680043.8.498.40436189400403161513802840351544666595","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11723945299462627737346907048285143063","series":"1.2.826.0.1.3680043.8.498.39794942415832934927320150633900484674","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11726452645519338341002251323463736083","series":"1.2.826.0.1.3680043.8.498.31500892862123086648272319156523315490","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11753067309373104365558887398441364695","series":"1.2.826.0.1.3680043.8.498.52924740404777047342270508862630082827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11751697254880722008245447197446027607","series":"1.2.826.0.1.3680043.8.498.72934572820627323226039528008194829833","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11762851510866726051342964961256680025","series":"1.2.826.0.1.3680043.8.498.91603083526048744728588853038189255972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11776647434143676427722616249587614803","series":"1.2.826.0.1.3680043.8.498.22793554490195421179202308721676214800","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11783597160745641720078978406968189488","series":"1.2.826.0.1.3680043.8.498.51205293320304541716324027135430691247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756","series":"1.2.826.0.1.3680043.8.498.38661357433276987043969996027193861223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11786296135006283515679011075884245663","series":"1.2.826.0.1.3680043.8.498.12142869680335098355590540911198762458","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11791028039098776993164396657149877483","series":"1.2.826.0.1.3680043.8.498.73942837544727705561373165608566783524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11798183157109435753679723596499976762","series":"1.2.826.0.1.3680043.8.498.11141658821914679354926570992468612604","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11805131119810457174456565128522037991","series":"1.2.826.0.1.3680043.8.498.12944903542719919260187384437623728064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11812607848520072826145426247813670653","series":"1.2.826.0.1.3680043.8.498.71069389332890648418954738424823877658","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11823120042104793954689778719229868108","series":"1.2.826.0.1.3680043.8.498.30193094320355160032572764489152992149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11794753699425693678056056340551238033","series":"1.2.826.0.1.3680043.8.498.83252241597109034586001613706950136062","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11836929166973549727078374306175992474","series":"1.2.826.0.1.3680043.8.498.57374985740985872350719251839877660233","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11803762390999881908101130422708589537","series":"1.2.826.0.1.3680043.8.498.95880688320323997496484959392641735778","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11851285389689341495961233129719325417","series":"1.2.826.0.1.3680043.8.498.92014766167558307381787224272820673730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11867198902144602395671904591496233129","series":"1.2.826.0.1.3680043.8.498.11669354503138898895029043481526361694","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11860731961337988611847638299761007498","series":"1.2.826.0.1.3680043.8.498.79056744740732815205562666540652384692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11878847808072119549790690649256090471","series":"1.2.826.0.1.3680043.8.498.14666079282551054617472153640359581994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11892908237449514945544380023820626319","series":"1.2.826.0.1.3680043.8.498.11931728732211899159980707076717933544","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11868389442143514705330687746274512632","series":"1.2.826.0.1.3680043.8.498.12710759321130369987008694694031826133","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11881735311125899170335963782017197184","series":"1.2.826.0.1.3680043.8.498.18149999371805410090347135949350227588","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11900355716044995266104285198121445744","series":"1.2.826.0.1.3680043.8.498.11694107495552930251406682182652982806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11912276256810863127132294167870461707","series":"1.2.826.0.1.3680043.8.498.56882887225880303182999051775799892429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11902464730418099436837163594625541861","series":"1.2.826.0.1.3680043.8.498.90176560490161071301004846406840733489","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11905397531490569218559534495408776672","series":"1.2.826.0.1.3680043.8.498.20813084676967773222372087048677780952","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11909779134152823550896593167929611759","series":"1.2.826.0.1.3680043.8.498.84982877557446192128274481285804561487","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11919420571588879483419578078755803351","series":"1.2.826.0.1.3680043.8.498.72250655594631242975455830500410497136","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11920531053482540304184657218030242597","series":"1.2.826.0.1.3680043.8.498.36215673402950310587789228112501361784","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11927352926669104658144933891780551483","series":"1.2.826.0.1.3680043.8.498.12170508358775324589702715935884393497","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11915933561045570822185908784072057715","series":"1.2.826.0.1.3680043.8.498.85262385357980228699742034305097551865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11935116791150327546384719051023540185","series":"1.2.826.0.1.3680043.8.498.50713587121784410562449689091388553163","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11924999889245769866426422681253524920","series":"1.2.826.0.1.3680043.8.498.35474234697460412322408273670781642868","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11937911933281010146897685679862930140","series":"1.2.826.0.1.3680043.8.498.91351276369320632625074185755681234272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11958117172819754862618612170999909480","series":"1.2.826.0.1.3680043.8.498.86363903045677187039144356099110792369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11956771495610267604936834092317960600","series":"1.2.826.0.1.3680043.8.498.18429758990058807579207652351620353713","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317","series":"1.2.826.0.1.3680043.8.498.37360736441830873870609190110962299923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11979034558288454336812613626594510186","series":"1.2.826.0.1.3680043.8.498.66854834900828072887707848024232279019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11980187124986749407584543247586384443","series":"1.2.826.0.1.3680043.8.498.12111954425606970772275976696587540070","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11991512352474459753052051071717622771","series":"1.2.826.0.1.3680043.8.498.87376458110164264621648119429843477417","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11944762815616453054528091721553484857","series":"1.2.826.0.1.3680043.8.498.33455730924801165535054190674478537259","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11999008448480965933785988718780864948","series":"1.2.826.0.1.3680043.8.498.85820928530872230625756769173447722815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11981170645605720774509861722988500673","series":"1.2.826.0.1.3680043.8.498.18013628462059561022182129836739311700","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12009816058271421855131593296751372723","series":"1.2.826.0.1.3680043.8.498.12803797679917596161780541170187114263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12006027138204983547873134458107233878","series":"1.2.826.0.1.3680043.8.498.36766376930240933666381230276913958998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12014959202771141042815134335571931814","series":"1.2.826.0.1.3680043.8.498.89731509730013970115786688955399256740","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12026457191304650188547548517205757122","series":"1.2.826.0.1.3680043.8.498.74614829868048968708574793698665879077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12041392088066705782960819598258792338","series":"1.2.826.0.1.3680043.8.498.34342562388357782561214029231076586481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12045300645580015230470991493790324857","series":"1.2.826.0.1.3680043.8.498.16405010790840929644412572845533986393","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12051169109587120248372991913002670813","series":"1.2.826.0.1.3680043.8.498.28635309647289187724760413207014964296","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12034579945815402722591687609951730205","series":"1.2.826.0.1.3680043.8.498.48569222386746885184434102922614029325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12067625141402105065337444349201280606","series":"1.2.826.0.1.3680043.8.498.12131463982304095566743497174967527646","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12071998470084955703027522152040042364","series":"1.2.826.0.1.3680043.8.498.47383175405691734351481093045064377382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12092744899746753313905432642371563786","series":"1.2.826.0.1.3680043.8.498.75004159226612438577724181459551206643","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12063742398421783626420335095128808816","series":"1.2.826.0.1.3680043.8.498.10966318154445739523966073417426318408","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12107680313133383587522762626929218051","series":"1.2.826.0.1.3680043.8.498.27624758338950476960122592322842340686","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12130868132537772155475981935868588761","series":"1.2.826.0.1.3680043.8.498.83817125439473147280383559232008298358","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12104799278874963688461235330096856415","series":"1.2.826.0.1.3680043.8.498.10308496537329574421594979493853151721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12117377043894182951345966247899775612","series":"1.2.826.0.1.3680043.8.498.89008962186447512916259559263913665066","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12132814192991105841939278718608095213","series":"1.2.826.0.1.3680043.8.498.78353769800336848659566653123564882296","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12137739162059836759686062882514824614","series":"1.2.826.0.1.3680043.8.498.12420828072849503611383902941929334326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12098718763390783455090805975786780205","series":"1.2.826.0.1.3680043.8.498.61158539488521846136697060297019310269","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12153755823296803843819361055386265595","series":"1.2.826.0.1.3680043.8.498.58313871254257520649846427940724728250","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12158630368692962312115489365152319457","series":"1.2.826.0.1.3680043.8.498.83056656597690907204422999839709990053","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12148236440353424874546037143134438088","series":"1.2.826.0.1.3680043.8.498.50752975457603099805094616172123092062","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12165893434332227221101266719291133169","series":"1.2.826.0.1.3680043.8.498.18195973771803375595858903364098369549","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12170753305862973517190756418865414984","series":"1.2.826.0.1.3680043.8.498.12710383538607339848876475758619547457","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12171790864123519428649395041090102217","series":"1.2.826.0.1.3680043.8.498.11699331641641211612918205292073800768","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12166235678892893545054546864889743251","series":"1.2.826.0.1.3680043.8.498.22021519347988056551757909695530014604","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12176469158911896593955662590695392044","series":"1.2.826.0.1.3680043.8.498.33836612732028809598006095320453843161","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12181591358572249491885248089245858515","series":"1.2.826.0.1.3680043.8.498.70022295668067974367224959502226966008","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12185615973620890770158935333033805789","series":"1.2.826.0.1.3680043.8.498.44963717367832755137043256222984073592","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12189863023888657759738596583562992408","series":"1.2.826.0.1.3680043.8.498.27630079257751312816068733933059642988","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12200672323103829956274910258854933266","series":"1.2.826.0.1.3680043.8.498.11498720533774178571608478355214348583","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12195007606427082866449558290311905939","series":"1.2.826.0.1.3680043.8.498.90029090025868443971245692919092822444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12219658385745696203053079500246399481","series":"1.2.826.0.1.3680043.8.498.96794451467110213405036027057321768155","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12208986088767216690584621773389997968","series":"1.2.826.0.1.3680043.8.498.75505038835088719854070125792753644754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12238865510201989224648662550299925120","series":"1.2.826.0.1.3680043.8.498.42782954653948084836336282462770128697","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12232127448545053821926845887026738988","series":"1.2.826.0.1.3680043.8.498.12851049362466615188390693961781425112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12225975251304101975065271521710735735","series":"1.2.826.0.1.3680043.8.498.85108259807438464048088702612273549839","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12202865491511254130359509254373393681","series":"1.2.826.0.1.3680043.8.498.45132915063551105004998917933094082357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12248983667596865960950671947508678392","series":"1.2.826.0.1.3680043.8.498.11803270268589733625953139722953206099","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12245760386130805526409111049874664817","series":"1.2.826.0.1.3680043.8.498.14893021698863922043122973077571541107","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12257119852150088212071954292747252243","series":"1.2.826.0.1.3680043.8.498.10808937242141629594849465610716680679","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12259896940707176389739553110492954399","series":"1.2.826.0.1.3680043.8.498.58758735759979515645166498422291012198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12261175520056712414700866621816882336","series":"1.2.826.0.1.3680043.8.498.52523327222602745749596859523976496310","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12242172267695588655664456078003635837","series":"1.2.826.0.1.3680043.8.498.71768483873778980204482305312786949840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12266644932420958618394322959047940542","series":"1.2.826.0.1.3680043.8.498.17557822948704586176709830886669173973","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12271616365050313084031931009756086606","series":"1.2.826.0.1.3680043.8.498.79071153667766966109864825287944670631","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12283662652762881783845275662580546217","series":"1.2.826.0.1.3680043.8.498.15497568644816449893299629007711660138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12273490180961458167930039242826008249","series":"1.2.826.0.1.3680043.8.498.11116380381696290455585051338120673645","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12275265458583331749472987758046974406","series":"1.2.826.0.1.3680043.8.498.11106124948934734884254073369546115424","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12266054872802015665014176014323258615","series":"1.2.826.0.1.3680043.8.498.21752742427477317663612614499304714771","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12285469859900617016859868410323043734","series":"1.2.826.0.1.3680043.8.498.10577123123268523071425054660363343055","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12294073960417956629256115458135748872","series":"1.2.826.0.1.3680043.8.498.23554049788978726106174763069941081076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12295443776523723913480216844110882937","series":"1.2.826.0.1.3680043.8.498.43197919527922948511906976437541088375","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12303161232396569915032951761728884199","series":"1.2.826.0.1.3680043.8.498.89738555185818812928997374017451342360","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12300749089664733313051543117302244508","series":"1.2.826.0.1.3680043.8.498.11188250230313040187691714505466669859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12305733049746414359748178404911549573","series":"1.2.826.0.1.3680043.8.498.63325345005266608168681716085332282519","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12322760603170246518010233325038530097","series":"1.2.826.0.1.3680043.8.498.76615661984378663889588753827208881376","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12328313245396930020974053819810033945","series":"1.2.826.0.1.3680043.8.498.68217436167769410424406041962945255351","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12335573784261147388984372283481919933","series":"1.2.826.0.1.3680043.8.498.28483336484136695326549223982384249050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12332727488899501354972751168284397151","series":"1.2.826.0.1.3680043.8.498.40960424643598953356797520228556418180","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12354685390005531691919754048141542648","series":"1.2.826.0.1.3680043.8.498.98042832657197173260675705753484134689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12348875283993064999589090039392140219","series":"1.2.826.0.1.3680043.8.498.10904463693784431009629136736698250528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12349656752404066719917550972333542508","series":"1.2.826.0.1.3680043.8.498.95193839623005132269966440289165875440","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12365033663152692616611548717152384269","series":"1.2.826.0.1.3680043.8.498.60537943203933025457771303854435658122","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12347007036329783363431172840583325937","series":"1.2.826.0.1.3680043.8.498.23300706187624852232483230507770195252","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12356611369549236159671896247094661767","series":"1.2.826.0.1.3680043.8.498.57190103821310215773100073637699681774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12373923338033469765189220784089350380","series":"1.2.826.0.1.3680043.8.498.20463391854947025180034027215961084865","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12383716855639375235119728470042467411","series":"1.2.826.0.1.3680043.8.498.60593596220273975925029217159774337400","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12387075993162655781372091860026675548","series":"1.2.826.0.1.3680043.8.498.89830604077505552892399562445881659574","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12377852033407536077897684568488365785","series":"1.2.826.0.1.3680043.8.498.14253193255435491519402570857556386223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12401147012438969595202076588471168894","series":"1.2.826.0.1.3680043.8.498.75158129661235577023930763597325498729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12381325071746339154385415220972069782","series":"1.2.826.0.1.3680043.8.498.25615944480855946447418088853558132822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12387667155732116394489547491673280062","series":"1.2.826.0.1.3680043.8.498.27088710662583687902059306438696843494","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408933790879905193765468926022393696","series":"1.2.826.0.1.3680043.8.498.11852485293457601081821731783158498131","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408186961945791320086301369036829266","series":"1.2.826.0.1.3680043.8.498.52218847211899176691436330101998401555","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12414869188691502885668778427120766321","series":"1.2.826.0.1.3680043.8.498.85074586939520270917933476837961932763","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12431272351851438889289190308550697909","series":"1.2.826.0.1.3680043.8.498.11707477725285854260104992917545210062","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12435506274406010562396858512839065232","series":"1.2.826.0.1.3680043.8.498.13200788756898278404298226002219251156","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12411340904876460326022004747253138831","series":"1.2.826.0.1.3680043.8.498.89278388209024300982797595647212028416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12438369033802887819851319347622787876","series":"1.2.826.0.1.3680043.8.498.24507293469595917340226506206208074358","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12462498156569316578044317782899399656","series":"1.2.826.0.1.3680043.8.498.11116380736584609872366629559581714670","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12473755232610784318190599222252893949","series":"1.2.826.0.1.3680043.8.498.59651370198825515685097710979565648848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12471165461877621368840037945454256612","series":"1.2.826.0.1.3680043.8.498.90630146114950114859255975664301311490","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12487362549907644519947672428024977201","series":"1.2.826.0.1.3680043.8.498.37944601299947582797330443165032229064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945","series":"1.2.826.0.1.3680043.8.498.30578400972104371422156040149306494449","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12482482491088588051076210158062455084","series":"1.2.826.0.1.3680043.8.498.14852634347378027776462319542357336061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12490034912643658601552092574630438642","series":"1.2.826.0.1.3680043.8.498.29045864045527320667789797407326686658","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491747080684524905643593142141672951","series":"1.2.826.0.1.3680043.8.498.34968571761542232265504487551235916080","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12492431885298164225672926785397151525","series":"1.2.826.0.1.3680043.8.498.98088158106162517036349306062117146158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12499208731138180053966892599139021038","series":"1.2.826.0.1.3680043.8.498.20059995939072300652312755832382885577","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819","series":"1.2.826.0.1.3680043.8.498.10058037489628716700816688496042880245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12510856755988712246013743789183840748","series":"1.2.826.0.1.3680043.8.498.66048255369028049462265704562139493801","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12515322634403509077334846234426636136","series":"1.2.826.0.1.3680043.8.498.11256127730205537607690552032136076344","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12520203592477518476890010185037272575","series":"1.2.826.0.1.3680043.8.498.24767517106929769570762134192599994311","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12526590714362529606046044104426807158","series":"1.2.826.0.1.3680043.8.498.38353516881417388658951800382745896452","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12531442175319757740745664049001806494","series":"1.2.826.0.1.3680043.8.498.10323344273035797249715247927321330888","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12523676732876902778577970655429060858","series":"1.2.826.0.1.3680043.8.498.58211784878373746752777325606501620572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12548176380550230498070609559486512004","series":"1.2.826.0.1.3680043.8.498.10645385620204048290942200086497488562","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12509825237087140646629745447753158673","series":"1.2.826.0.1.3680043.8.498.93408627930291627087023617912128870500","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12534628800787651008876070514080610643","series":"1.2.826.0.1.3680043.8.498.22322674601261697612789169004998225302","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12554824497579383319980187351290254727","series":"1.2.826.0.1.3680043.8.498.86406904388991553240777089011229728023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12563940497360871791097093821926086616","series":"1.2.826.0.1.3680043.8.498.12941291450358803743309760704355507238","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12553010776519233077752150342664160826","series":"1.2.826.0.1.3680043.8.498.36049192824612715891457913550078488855","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12569762036474456642024618586300354737","series":"1.2.826.0.1.3680043.8.498.11564530057987457184395918847941248621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12573518342767030025851169814440863785","series":"1.2.826.0.1.3680043.8.498.60804107113192247457385607302909820498","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12579738538063674380235672702328412431","series":"1.2.826.0.1.3680043.8.498.80583742170789066543079553992055808228","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12606136253498337506682547306933254691","series":"1.2.826.0.1.3680043.8.498.97990363686186146579545673010033986066","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12583872770399266463538638801797827468","series":"1.2.826.0.1.3680043.8.498.87674189798986780550750885797184545833","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12607203615180268117341271176820357242","series":"1.2.826.0.1.3680043.8.498.13084875690073690187107322574241334371","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12616376042564380071463081913949255574","series":"1.2.826.0.1.3680043.8.498.10119877605610345723650124821808676412","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12607959449008110729566106776800628990","series":"1.2.826.0.1.3680043.8.498.25051043385554726329960468479103584806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12620718517255799151283445360332799702","series":"1.2.826.0.1.3680043.8.498.10007282253941170661178728530472447162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12630423349626505319792444053874011718","series":"1.2.826.0.1.3680043.8.498.84763003080304144344972352148816574143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12576055921678075813718486704978381993","series":"1.2.826.0.1.3680043.8.498.82674395897986868671375564375095409997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12632129640405661298623288977632682289","series":"1.2.826.0.1.3680043.8.498.10346030122694910196746254300805454660","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12626383124142146001532197239671686442","series":"1.2.826.0.1.3680043.8.498.11805943007256167834632366535666601945","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12642641661988102031826128876502303807","series":"1.2.826.0.1.3680043.8.498.60268876120540555021221043617575546198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12655324218817524683138594937984186560","series":"1.2.826.0.1.3680043.8.498.10265726252128223255109996657958690764","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12660901572462562819575002681831361427","series":"1.2.826.0.1.3680043.8.498.79958422089386099850296479042738504137","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12638205949145637289277744267473965719","series":"1.2.826.0.1.3680043.8.498.38578562449317389342714384747827158272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12647791319443598318497312837147189300","series":"1.2.826.0.1.3680043.8.498.44838574759714801507236702251841684624","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12673561338227595483297605409292531492","series":"1.2.826.0.1.3680043.8.498.11553405159777863689670566736306559429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12674930973831274527829261576087154644","series":"1.2.826.0.1.3680043.8.498.53674084167489037735940693436304262443","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12690080124553846006638565600585469969","series":"1.2.826.0.1.3680043.8.498.88762965092087700134563504422734589056","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12692579161952587215276483587686562922","series":"1.2.826.0.1.3680043.8.498.29998840657737203171910520380027546904","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12699751030439008569144814819343161608","series":"1.2.826.0.1.3680043.8.498.11757422602512922682814824296249946191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12700981063762577397919340901063297803","series":"1.2.826.0.1.3680043.8.498.68712823976891537890413893927920279302","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12715067947657150279232238280712066972","series":"1.2.826.0.1.3680043.8.498.95498040600327761359769867228162340150","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12704063145595847725421427571860589471","series":"1.2.826.0.1.3680043.8.498.85007776385504109998299696021269426288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12715402081094788880283826571710131940","series":"1.2.826.0.1.3680043.8.498.77698072758721900503087517142148652294","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12688954493547892047021455671738310186","series":"1.2.826.0.1.3680043.8.498.35593770069961139464945530424863643598","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12719455674338091089495020475033082405","series":"1.2.826.0.1.3680043.8.498.89154160764207938031123586597974798772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12718966639501121738042125605376714231","series":"1.2.826.0.1.3680043.8.498.58341292142232844791855133891362456476","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12728397615513347615111391662798012012","series":"1.2.826.0.1.3680043.8.498.12547673966754789121163080333171237489","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12725832336445568031849006413563892122","series":"1.2.826.0.1.3680043.8.498.44474088976405934873947480937472128372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12748840878706905309050091576449488666","series":"1.2.826.0.1.3680043.8.498.12883203121343673998318129032817153828","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12759704450656741777120047828560448622","series":"1.2.826.0.1.3680043.8.498.10368646287856025912294032513214210218","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12775050278338123442507219047972133270","series":"1.2.826.0.1.3680043.8.498.68922966204841356674942365290104518493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12765434812409174436720333864278447770","series":"1.2.826.0.1.3680043.8.498.44448275905632095457013605600817493866","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12769887673556938425710414718385118283","series":"1.2.826.0.1.3680043.8.498.11638307273095811068168585347612592288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12762076104929080571464647825161185643","series":"1.2.826.0.1.3680043.8.498.78854009429610586495924046174821320866","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12792431109288033212787919118858331455","series":"1.2.826.0.1.3680043.8.498.10069820596235696038340553494738206298","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12785377697188564760194569072434828180","series":"1.2.826.0.1.3680043.8.498.58585973062424329989836859688003798765","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12785792890626338008419452097492192088","series":"1.2.826.0.1.3680043.8.498.79417746325238434806767453608883753312","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12775769262869064592411499834441472333","series":"1.2.826.0.1.3680043.8.498.49716610474678231160428660302354280584","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12799849540231315910448269889175806934","series":"1.2.826.0.1.3680043.8.498.45888427689753496781764745850677340653","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492","series":"1.2.826.0.1.3680043.8.498.12840952989395411061636411902524593000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12817753008281742621758605755308036035","series":"1.2.826.0.1.3680043.8.498.79894314464418892431286722016372638972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12806792356922865929896103306611295151","series":"1.2.826.0.1.3680043.8.498.36496566596122950147384665893767254818","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12819437506700585448407291050085063174","series":"1.2.826.0.1.3680043.8.498.49216817851303353787463557006764927659","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12827120140904847104302376671076396695","series":"1.2.826.0.1.3680043.8.498.79733546647407765422835954383620187109","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12824438254096860470348581093672563751","series":"1.2.826.0.1.3680043.8.498.22423481667587071732768233817141384558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12839498544183083639548298117995964432","series":"1.2.826.0.1.3680043.8.498.61675475947746645267096236307560563625","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12887544771002132349592859410979448655","series":"1.2.826.0.1.3680043.8.498.12472317614004155557451315476841826754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12814598731776085957663058326979908652","series":"1.2.826.0.1.3680043.8.498.11144291148465692321143657130081142384","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12836702174475748025129787471026627810","series":"1.2.826.0.1.3680043.8.498.20740918137165374791865004347146891949","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12899995703999073968090810947564075608","series":"1.2.826.0.1.3680043.8.498.13072921948442217174166627157789359690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12864336642218457767597053831458716404","series":"1.2.826.0.1.3680043.8.498.13337975248392262079830617706473600663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12907457183137517669505067507754394039","series":"1.2.826.0.1.3680043.8.498.66844494028159914271256894733831386418","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12936296973456045538701035923586653502","series":"1.2.826.0.1.3680043.8.498.11799162913698887642924984283363237911","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12942677240800343332600884989364651588","series":"1.2.826.0.1.3680043.8.498.52734518761580241740756826521065197095","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12919303621342250989179219420795277559","series":"1.2.826.0.1.3680043.8.498.11051728213528410962644621606043711689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12946789387115203718210904958275579544","series":"1.2.826.0.1.3680043.8.498.54827871237190403618308006316676553690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12950639892345812884896649629409199101","series":"1.2.826.0.1.3680043.8.498.81590976414734528793855904341814537604","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12958999546496283565938189274771483491","series":"1.2.826.0.1.3680043.8.498.91100498664516884010603124951938695830","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12957252711278416684461267069498115130","series":"1.2.826.0.1.3680043.8.498.91352731948469543678781833008196289406","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12914677763148740041755977075799271057","series":"1.2.826.0.1.3680043.8.498.38323515606723741883011868293088583318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12965787822216495014894819598440431626","series":"1.2.826.0.1.3680043.8.498.28967198394453768027381264979888648956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803","series":"1.2.826.0.1.3680043.8.498.10259924209097075418304938918872474391","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12981413356524881858859912294078498486","series":"1.2.826.0.1.3680043.8.498.10677991600335293242218149077223298651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12985683732818911488433522743768046254","series":"1.2.826.0.1.3680043.8.498.10197406329378895802824089362268502213","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12992013816638129896738209015383985569","series":"1.2.826.0.1.3680043.8.498.35076325044576605294591097388552356125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13004048880335015585285100185910148808","series":"1.2.826.0.1.3680043.8.498.11554810793861594003937899459491320096","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12986517636774565418070546261705439006","series":"1.2.826.0.1.3680043.8.498.12048244575051588554459454894402900151","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13017199074506191657941905498813292773","series":"1.2.826.0.1.3680043.8.498.10160295509327359226939822946595207401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13015502057686077045104167339480215528","series":"1.2.826.0.1.3680043.8.498.43084125425105237760465000827878912725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12970315889871740202726744767179863179","series":"1.2.826.0.1.3680043.8.498.61439685417271915872562631717507972886","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13022585036130454941964190427558363602","series":"1.2.826.0.1.3680043.8.498.98524479193308359522351034545883379357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13027502247317650159940595684214701589","series":"1.2.826.0.1.3680043.8.498.46127653962961269010827143317373397777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13024625496981667037628275235696732200","series":"1.2.826.0.1.3680043.8.498.73362252124384090071247843516452804866","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13030662056306690969977140713179461363","series":"1.2.826.0.1.3680043.8.498.12708310131711134223325269050452147528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13026463185033527579856819607371375594","series":"1.2.826.0.1.3680043.8.498.53030436723593184909169682178946181111","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13035632998746261521316395720102873141","series":"1.2.826.0.1.3680043.8.498.57690554036415847168399266572017846579","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13054766954092919774634499780241210835","series":"1.2.826.0.1.3680043.8.498.57752310288124421386474484828807573670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13062978180647549151946243290102173508","series":"1.2.826.0.1.3680043.8.498.16830740143443080800100376868682786168","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13068696303639759696458130704554709416","series":"1.2.826.0.1.3680043.8.498.30575049566215406867923012056226315801","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13077357855769903335728068420884303993","series":"1.2.826.0.1.3680043.8.498.48746333099374436564239915696943156282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13078551660783737720660607567713996703","series":"1.2.826.0.1.3680043.8.498.24914982801670673329832440631049331090","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13072908567275370866813748886351904593","series":"1.2.826.0.1.3680043.8.498.82440198300326652209569712434524754111","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13079170182386765602129976624023222742","series":"1.2.826.0.1.3680043.8.498.77593935918251886725799992273080832065","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13087770868099843948960419968576882974","series":"1.2.826.0.1.3680043.8.498.71186539001072945874100044017211175600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13093751810869039645959119469231437894","series":"1.2.826.0.1.3680043.8.498.11367577998233338936433376708171166274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13092094949394905924922534008771583745","series":"1.2.826.0.1.3680043.8.498.62128077643613414118895966733104799919","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13113623966675790946816351311283701141","series":"1.2.826.0.1.3680043.8.498.12196534403245116397293824221263360378","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13114693814781737495494700902699619138","series":"1.2.826.0.1.3680043.8.498.91754552986738371359878537098765503346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13118325043138522704531128725514354285","series":"1.2.826.0.1.3680043.8.498.24527335630913571117790563428892537632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13099702881219862576749493772702888064","series":"1.2.826.0.1.3680043.8.498.54922188686709564692752743599095426941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13103513835731355705707295729643672786","series":"1.2.826.0.1.3680043.8.498.38206718081253987079114371154876589380","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13125910783461518266627225743585018143","series":"1.2.826.0.1.3680043.8.498.69931043307600336442241554180195145469","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13126380916005729420163981055214890048","series":"1.2.826.0.1.3680043.8.498.14790502976473759816387037568704562098","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13130534736322715018024786213131944607","series":"1.2.826.0.1.3680043.8.498.12248874840318611320070800123390080196","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13128506994126383651503283021834372142","series":"1.2.826.0.1.3680043.8.498.93936625794427892814176199525324792691","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13139684164906828131777150061532685584","series":"1.2.826.0.1.3680043.8.498.10411616519664381572519590389315562281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13137510961466386836387239510287919268","series":"1.2.826.0.1.3680043.8.498.47928757967785952235003776534844723699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13136367866260644484250529322937554927","series":"1.2.826.0.1.3680043.8.498.24154903485090332509949937608704262504","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13148805278538458401210722493048566510","series":"1.2.826.0.1.3680043.8.498.10423664825586402680118020498387884844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13150794288945922409613136250303663502","series":"1.2.826.0.1.3680043.8.498.12230536477872701916852281189275395581","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13157776420553094741176082047621957775","series":"1.2.826.0.1.3680043.8.498.13155927792880093973460855227342868450","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13152690996267297597178587721720788387","series":"1.2.826.0.1.3680043.8.498.11030437094490549605690633300363200147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13133755791254469039849289186598412660","series":"1.2.826.0.1.3680043.8.498.10568066561746571915886159307182011459","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13158644154518335426512578922874968296","series":"1.2.826.0.1.3680043.8.498.27911153680377862749738355539772051491","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13162479706670045075896993021883562680","series":"1.2.826.0.1.3680043.8.498.11777554071923272151102781925826062751","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13165770394160595162984722917452661205","series":"1.2.826.0.1.3680043.8.498.45998951916696391696369295663503360519","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13146466304138609448483938844487385775","series":"1.2.826.0.1.3680043.8.498.56192160105769036462806696590556890208","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13173781757294564816839795307893228067","series":"1.2.826.0.1.3680043.8.498.13342305582597349550306755106645372295","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13176146790594027813190776550419533937","series":"1.2.826.0.1.3680043.8.498.11992041556412338412217691019113032237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13172295193446466338347273836250944090","series":"1.2.826.0.1.3680043.8.498.11349304102117647036731531173344315180","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13203246712933982189680087013566455075","series":"1.2.826.0.1.3680043.8.498.31234508549985473661145915004921714538","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13210084125872700841685814144002495145","series":"1.2.826.0.1.3680043.8.498.96341698057613649514276891423679995276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13199635020145400856987271814458645896","series":"1.2.826.0.1.3680043.8.498.72333966867713249024279617834259396317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13179829911816163852815103052496326166","series":"1.2.826.0.1.3680043.8.498.87554432478120366960666620736043775632","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13212280182729018182073794487313115605","series":"1.2.826.0.1.3680043.8.498.72845501398471386536221221540643977538","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13226961394403558608712498872213808226","series":"1.2.826.0.1.3680043.8.498.19566030987366285589377360169186789895","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13222515831888962408632292783773537312","series":"1.2.826.0.1.3680043.8.498.12828070682257087394513659544802098446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13240749152416621711688053465035013133","series":"1.2.826.0.1.3680043.8.498.44500152077279558352537801097941270374","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13256173302993031630968402868348974108","series":"1.2.826.0.1.3680043.8.498.71821696405509259217113512165047183360","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13232501202175905676335982676386953813","series":"1.2.826.0.1.3680043.8.498.11613676766489417256977430554190698003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13245409855393847268063261203253424169","series":"1.2.826.0.1.3680043.8.498.70464342305686760752045592811159650423","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13262053361139017457941697384315610025","series":"1.2.826.0.1.3680043.8.498.21082499439875857466670337482188300852","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13262583253812134425581922368942846684","series":"1.2.826.0.1.3680043.8.498.12073488699793416503424867112402691454","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13267708980958529833316446819156908381","series":"1.2.826.0.1.3680043.8.498.41800231954367010670800444196246828267","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13280017707177438711374726939683941320","series":"1.2.826.0.1.3680043.8.498.46434421769699986428788451607164497907","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13284123714012951608998434802430211634","series":"1.2.826.0.1.3680043.8.498.27486575750905262294770025418222186006","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13269841239774089690227054060975903636","series":"1.2.826.0.1.3680043.8.498.83950279556675315393700306013850947534","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13288244567806030859347115222642155186","series":"1.2.826.0.1.3680043.8.498.42360166116406021638543810830600580104","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13291943370279119939499960846843879156","series":"1.2.826.0.1.3680043.8.498.40324125648836921987748628020869650813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13295633458001436592005068637719195786","series":"1.2.826.0.1.3680043.8.498.11420335219113190173506712994798288185","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13314276151860609710431717811242659651","series":"1.2.826.0.1.3680043.8.498.55825492374169297448181192331354103440","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13309299049346932814529962458932580753","series":"1.2.826.0.1.3680043.8.498.38217738737327077731992406456795800810","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13321746459276191365724686301267421531","series":"1.2.826.0.1.3680043.8.498.78016941133550245818994967941761157315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13328029743559327467301971917273397645","series":"1.2.826.0.1.3680043.8.498.30300733304618293199468674541144393263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13335786652653204791740906118815827207","series":"1.2.826.0.1.3680043.8.498.18570878536523898215744519073510451173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13353249891620291438488727876474802879","series":"1.2.826.0.1.3680043.8.498.57788344109796400009097996455510220328","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13332521365069604440230185026865466121","series":"1.2.826.0.1.3680043.8.498.12055367955612629516072456427213720380","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13353966209257569249620562863006006441","series":"1.2.826.0.1.3680043.8.498.62709726707722651409431357501698502996","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13358702230966891064267033765095124604","series":"1.2.826.0.1.3680043.8.498.49906054911215250054815883889478324340","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13368881691140330168876825436602977388","series":"1.2.826.0.1.3680043.8.498.43426900580028778889651289888586372157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13367691581890020433564928882602648123","series":"1.2.826.0.1.3680043.8.498.57895204327994107937752461267726725563","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13371001997511542011416510364359588752","series":"1.2.826.0.1.3680043.8.498.11583333594100531919260369520101232171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13363562142644508091380459785810556567","series":"1.2.826.0.1.3680043.8.498.83407990508130202259819544763061269191","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13380735447520796902327272696579348141","series":"1.2.826.0.1.3680043.8.498.56807194747701013272480306696551349931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13393268272072264469105890769640363680","series":"1.2.826.0.1.3680043.8.498.48349189156207473747487763070922619591","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13430522388784978790587514142693914965","series":"1.2.826.0.1.3680043.8.498.10202970152092055869296214177741879055","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13400273795954308944359576841942016981","series":"1.2.826.0.1.3680043.8.498.11410789112243666224749206113271745180","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13455212888114594010716499984611216646","series":"1.2.826.0.1.3680043.8.498.96757050548145816683859965530887219391","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13460078631594032326006554321483799928","series":"1.2.826.0.1.3680043.8.498.10218141502823588443302657361190660596","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13461073016327661159616957998589287085","series":"1.2.826.0.1.3680043.8.498.12259525001313740293640090535376215051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13390551515761269433524925969561757962","series":"1.2.826.0.1.3680043.8.498.12547595413571787607329459500953293891","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13597463452741791628023738080583139636","series":"1.2.826.0.1.3680043.8.498.25575595483528409536832138380838781291","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13660439785395658826946769630921591235","series":"1.2.826.0.1.3680043.8.498.41622895621583194365007614865080549580","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13505671694940677161128783863912245059","series":"1.2.826.0.1.3680043.8.498.12074473792091085913487088138840229315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13737588520456064856233576393747389054","series":"1.2.826.0.1.3680043.8.498.86419664226394341912550063578150639520","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13879163301346984697246311226647565712","series":"1.2.826.0.1.3680043.8.498.72983662659814230231263866943509588041","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13958765761660666865188384031150827763","series":"1.2.826.0.1.3680043.8.498.13003221383272112675989892449572901629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14066537811671722419109228137841995847","series":"1.2.826.0.1.3680043.8.498.31810851891310164077942481246933231611","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13975027824128345697756313869165838055","series":"1.2.826.0.1.3680043.8.498.85603523949360654211616471181459881138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14032903253348079919799627243690484791","series":"1.2.826.0.1.3680043.8.498.68222111798068082084655654690078121587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13829111847660458141056077151781118782","series":"1.2.826.0.1.3680043.8.498.91110349508417193341687863185902407765","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14218113768651197803980865495163189825","series":"1.2.826.0.1.3680043.8.498.12897606105317706334371603741726663338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14137431039800236510749401303688858447","series":"1.2.826.0.1.3680043.8.498.72211981036528367939653311881383961372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14236519693915284584208989450687591759","series":"1.2.826.0.1.3680043.8.498.12744329721348118012531615448809784040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14453847272947043746856632410254555895","series":"1.2.826.0.1.3680043.8.498.47201246267967477995804848774346376436","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14325424371811230497993759959862738838","series":"1.2.826.0.1.3680043.8.498.77575566010160706556329046683513958251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14377917653321813299237274871837695586","series":"1.2.826.0.1.3680043.8.498.82413676475276021802651997866155203641","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14490539215064287912254286565666891557","series":"1.2.826.0.1.3680043.8.498.29674178691985111585439769605811194972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14511862461248267537051973802039959211","series":"1.2.826.0.1.3680043.8.498.10100773453049450747398125062254974591","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14477978091360159774052709992034078166","series":"1.2.826.0.1.3680043.8.498.86770234425589870353234055663856389801","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14744714658078107007743063095375646143","series":"1.2.826.0.1.3680043.8.498.70168202783191847789761127287427273606","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14776241668061173271058268311868302014","series":"1.2.826.0.1.3680043.8.498.60873830037045327494189352948588014839","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14679140388328648617386311272361396555","series":"1.2.826.0.1.3680043.8.498.51834226559068471500245103788911169978","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14959732862523218997149041503831001633","series":"1.2.826.0.1.3680043.8.498.10043334987354567111812324187317075137","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15024602758408626305833210011882949904","series":"1.2.826.0.1.3680043.8.498.41640913449995578444734468136396516720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15065556607080211420503043544482923211","series":"1.2.826.0.1.3680043.8.498.59118158434456430404358980287790490243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15219410096632334470101257478715035071","series":"1.2.826.0.1.3680043.8.498.40268379449122194927357444795432787615","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14845527334849156639306067473358712770","series":"1.2.826.0.1.3680043.8.498.99132457810631721249766263252874046183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14958560978176121375035839707004667832","series":"1.2.826.0.1.3680043.8.498.86515148329505604806140173492723960705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15297444019822621504849851092382560980","series":"1.2.826.0.1.3680043.8.498.79128882588017263807200318335646818226","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15103255966832759925770334719703425755","series":"1.2.826.0.1.3680043.8.498.93615777377908278429736137037566217022","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15376535778246978844757121776560020722","series":"1.2.826.0.1.3680043.8.498.12835800391210044470191409671735754238","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15337067581747365729183643142512558411","series":"1.2.826.0.1.3680043.8.498.10519981490359920507573190826392793941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15468179600399954478612037548577772269","series":"1.2.826.0.1.3680043.8.498.14302568917510290107849177794535792261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15428051518761311558410606976607852351","series":"1.2.826.0.1.3680043.8.498.78275189251283420275694319852040815054","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15522600887743817464976568872439594397","series":"1.2.826.0.1.3680043.8.498.53902919065122512906120077458438676448","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15599161312748548709240395406977042093","series":"1.2.826.0.1.3680043.8.498.12763577002809043465286302208785360429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15851463019383290210760064201595021029","series":"1.2.826.0.1.3680043.8.498.35886100504151525419895774834438901216","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16051798246004436152597344963060310360","series":"1.2.826.0.1.3680043.8.498.10071731081841776926911954285395913583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15709232000390534671340322807442769184","series":"1.2.826.0.1.3680043.8.498.99983505777663770150961778920287449750","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15969838281154273030237391394632849764","series":"1.2.826.0.1.3680043.8.498.48358295993244154099904080822884748651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309","series":"1.2.826.0.1.3680043.8.498.12836209270253860624540818250354794761","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16164261456077269604929662134371603081","series":"1.2.826.0.1.3680043.8.498.22072278473564513750067749852745225215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16101435445990180482665354366724730305","series":"1.2.826.0.1.3680043.8.498.87370103630704457688898504502516242526","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16258190393569006141285550986533848849","series":"1.2.826.0.1.3680043.8.498.93217343571696987318151044834832297631","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16370303687278282143772106415664171358","series":"1.2.826.0.1.3680043.8.498.37580213447774736059976716750365000149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16469256144078887607437578587242258846","series":"1.2.826.0.1.3680043.8.498.36272623006124416735109362687518773692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16200518336931321995379999168737787921","series":"1.2.826.0.1.3680043.8.498.15584982064278417309946996204089363433","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16538738179840493095338324009830673843","series":"1.2.826.0.1.3680043.8.498.94721182141979060988934510438021920762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16581426215518472805200227992550139190","series":"1.2.826.0.1.3680043.8.498.10671381805957692968234212715895062125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16657553974040122719239506008948241297","series":"1.2.826.0.1.3680043.8.498.25429459589078213600300517572376407064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16338296537380784459604897543069625390","series":"1.2.826.0.1.3680043.8.498.13896473674574658278943027776305934254","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16674237773422474998796191258459643061","series":"1.2.826.0.1.3680043.8.498.30465011790426290777205563914395279826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16709565360569239280032743619008187278","series":"1.2.826.0.1.3680043.8.498.18052084683465540778671374071599477570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16774450022563283092653506225603012645","series":"1.2.826.0.1.3680043.8.498.54705918668756234568745906287331257410","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16718788321951483456195507236392554704","series":"1.2.826.0.1.3680043.8.498.28419133188262548966924081904831990215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16843381581429596234606436682284624187","series":"1.2.826.0.1.3680043.8.498.57902422303173429217982567541619634023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16862596867845295946732698348903803700","series":"1.2.826.0.1.3680043.8.498.79004110790832233680641230988168636551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16527747624688737126118546487228060848","series":"1.2.826.0.1.3680043.8.498.13162274249699423303585473531401907820","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17074746747536999918446136673966046653","series":"1.2.826.0.1.3680043.8.498.95230094684626752362274439647459907921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16886658450293145802897654768506439494","series":"1.2.826.0.1.3680043.8.498.17050129296690074611565245342312193745","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17056997047488992455301411194938263817","series":"1.2.826.0.1.3680043.8.498.21657113338682682494573349498787597548","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17093025979426350838724553020541643719","series":"1.2.826.0.1.3680043.8.498.61836003381059395166907159610175855902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17265093113586420061761042467476063876","series":"1.2.826.0.1.3680043.8.498.12606525274418256069024505226408789814","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17160610480070701553118850947260201608","series":"1.2.826.0.1.3680043.8.498.51892990661212985818484235887327769830","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17123244662958741977525806700986256210","series":"1.2.826.0.1.3680043.8.498.38761531310852667777172658059990743758","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17336223563862025807066700226971845206","series":"1.2.826.0.1.3680043.8.498.55510287565913733887592857920014293456","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17547781127981007962008163248942714330","series":"1.2.826.0.1.3680043.8.498.11777344048327463289186253547865378798","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17618734998753079190687719289815512319","series":"1.2.826.0.1.3680043.8.498.58434720010080671651531566651388473046","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17663005209256897892831319144686413853","series":"1.2.826.0.1.3680043.8.498.84216940881041294908338306330925043840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17741907690913585163443565102293192244","series":"1.2.826.0.1.3680043.8.498.12953148585222632459882066976644636129","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17427902052389033416295416299697276519","series":"1.2.826.0.1.3680043.8.498.76234020749678031049385438694578717513","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17769942698581399165275319502954100325","series":"1.2.826.0.1.3680043.8.498.72457850081559011575332841348131640584","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17415936504684075438800895036740822145","series":"1.2.826.0.1.3680043.8.498.63995757305452373523445052239997720872","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17856297841306582866782295284505406916","series":"1.2.826.0.1.3680043.8.498.60968447237385063834900406878517503018","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17909590711893200514107254988453660857","series":"1.2.826.0.1.3680043.8.498.82604192246447561319302873925465289540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17967978373588651520812399064902037467","series":"1.2.826.0.1.3680043.8.498.77154390264686577448531898169503992373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18092766733837308394607532608319746043","series":"1.2.826.0.1.3680043.8.498.12634367675822811374273849655485884755","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18080511167840608448833484000937072258","series":"1.2.826.0.1.3680043.8.498.18885597732916044100685289158733339751","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17836815413824722543229791044312174885","series":"1.2.826.0.1.3680043.8.498.88272503889536046439528715136485625313","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18114567737638963627123562895002602285","series":"1.2.826.0.1.3680043.8.498.92270962245482103345527330259843730105","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18176586296602067352845573876750324761","series":"1.2.826.0.1.3680043.8.498.11614254232170219813739471373531403350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18230847745299902998425838090653825745","series":"1.2.826.0.1.3680043.8.498.60733372717579061395537203941467678670","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18350185433331570523543207086298834067","series":"1.2.826.0.1.3680043.8.498.54217611392100165970381736916901886853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18315708641533862849721595991941979080","series":"1.2.826.0.1.3680043.8.498.11955157714217822375089857712542496758","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378","series":"1.2.826.0.1.3680043.8.498.47417363768929649365837124498379339352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18529656610752887218162968403101035390","series":"1.2.826.0.1.3680043.8.498.10544235781347869990881403249137838640","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18653838887148864162027930218548124748","series":"1.2.826.0.1.3680043.8.498.11372382960242173938833799008133506369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18614704377104425868545800677445972261","series":"1.2.826.0.1.3680043.8.498.81106614348542406824720893749903658326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18680059761479598675335345641208680006","series":"1.2.826.0.1.3680043.8.498.61190862248721184768331652223336601885","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18721466442792788108431096912844719622","series":"1.2.826.0.1.3680043.8.498.72305076918177036262433435055275064148","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18509793799833391995987656714798904411","series":"1.2.826.0.1.3680043.8.498.88281243551285912863067863925152743339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18792568294433896490648198192877693850","series":"1.2.826.0.1.3680043.8.498.98102326384022384988135161740715554081","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18879049235810379709186936621085221483","series":"1.2.826.0.1.3680043.8.498.25758184767199224870864227713561303301","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18931107448953327254073392032141677797","series":"1.2.826.0.1.3680043.8.498.64818543578864967222301785886581197546","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18993960105596277135383396226525760874","series":"1.2.826.0.1.3680043.8.498.14434123796539886253952186691414701140","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19104350033985745058396432133828748480","series":"1.2.826.0.1.3680043.8.498.10064844036614751208433717781482724643","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19178214677492694135853596920291058397","series":"1.2.826.0.1.3680043.8.498.11699190701474131419644117844472408321","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19198152070740205434848422424805073748","series":"1.2.826.0.1.3680043.8.498.30798504638467265143850048721886968522","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19194090255958016414082788444327981898","series":"1.2.826.0.1.3680043.8.498.12940304391711345415329946242012199841","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19342628541833173544083802327412975225","series":"1.2.826.0.1.3680043.8.498.13261728522305402844964071111650502529","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18825393854436144614202928260751418496","series":"1.2.826.0.1.3680043.8.498.61167626063546758010972524508257646087","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19419878169415950073207624537163068568","series":"1.2.826.0.1.3680043.8.498.31646005803661887922716781897523358649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19512754570664577075249836633718382506","series":"1.2.826.0.1.3680043.8.498.13102475076252897167624194737035541646","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19573195389434178432956976770560221462","series":"1.2.826.0.1.3680043.8.498.84777295564595939149219048771120995161","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19706775737718777290660344285046025310","series":"1.2.826.0.1.3680043.8.498.11902899604612320626549760552488574645","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19601428113073026637547887698342670296","series":"1.2.826.0.1.3680043.8.498.69602439266530826431206442120752978810","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19486520578867272329833751210055639515","series":"1.2.826.0.1.3680043.8.498.55556918162081522878381246711154691288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19821498229702528191367700655079805824","series":"1.2.826.0.1.3680043.8.498.11863647166406143175140124991383502785","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19832308119617081676615236082965891208","series":"1.2.826.0.1.3680043.8.498.43900428940670302294366505113320708945","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19788612491138642031531719942303619648","series":"1.2.826.0.1.3680043.8.498.62337200027316955091925223027834174230","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19871496221665809577186118817075295760","series":"1.2.826.0.1.3680043.8.498.61358459936414246387457682094924330891","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19895710326297085463251295629948896948","series":"1.2.826.0.1.3680043.8.498.12544385607029900595265488105715562050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19965044457637713843792494926652048462","series":"1.2.826.0.1.3680043.8.498.90586230071070700171633293127472610003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20023559426196341254845485444587607391","series":"1.2.826.0.1.3680043.8.498.37241997076364900439820638362928047328","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20156361590688072955579948030116579099","series":"1.2.826.0.1.3680043.8.498.21960239809086643753176287423873422112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20204907530808218314127264658715054041","series":"1.2.826.0.1.3680043.8.498.71582789112069255276248231278643755020","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20123813962665544845986028832159432587","series":"1.2.826.0.1.3680043.8.498.12687860397380878968117022462647254996","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20266523169651039493799975212435349766","series":"1.2.826.0.1.3680043.8.498.50647908153013537700151417449987949262","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20192024727432864451309151144591371229","series":"1.2.826.0.1.3680043.8.498.30805291642060106124242691031472812791","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20301796413163444642416918108994411350","series":"1.2.826.0.1.3680043.8.498.22734719783990648495819260669365401671","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20472525526345683475357475395677168372","series":"1.2.826.0.1.3680043.8.498.11925328941814705548649877113404219773","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20522330861615570894210641293344468905","series":"1.2.826.0.1.3680043.8.498.46782749961303965962548940093204599690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20411068889491114730824265411248922690","series":"1.2.826.0.1.3680043.8.498.86635623387879622843869710734899687318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886","series":"1.2.826.0.1.3680043.8.498.71776347363552533436056527036005244084","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20600318722531772866761132262772105640","series":"1.2.826.0.1.3680043.8.498.55794037841936151337308200808312628464","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20690227395174139480327772367312879333","series":"1.2.826.0.1.3680043.8.498.11996628930150670517107669261857912544","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20781255814255169306555078952156148784","series":"1.2.826.0.1.3680043.8.498.16837416019620271134500354070176970678","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20850785960533583051456342090773230767","series":"1.2.826.0.1.3680043.8.498.10059771288496613521644569510237655897","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20859518005209632163032213488405671537","series":"1.2.826.0.1.3680043.8.498.73593863203374157535717281135637617510","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21024377993947321291280779867493407199","series":"1.2.826.0.1.3680043.8.498.99756865494595141142824145518281660651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20969353615978236040448556356026349615","series":"1.2.826.0.1.3680043.8.498.51351735311004166283507441542721694730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20947841686457891570351579406919873254","series":"1.2.826.0.1.3680043.8.498.64841169896055868373134089332658177355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21116046804798247223618072736212944615","series":"1.2.826.0.1.3680043.8.498.34794618411863081838860997718241080980","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21294732642571417618432023007693943849","series":"1.2.826.0.1.3680043.8.498.18298481211045244308266992633384326725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21142867813782396036287673290033987859","series":"1.2.826.0.1.3680043.8.498.43049019037221033147829380352862563560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21322638458116180116030396109651929596","series":"1.2.826.0.1.3680043.8.498.31052903228558120438870663627173693471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21403015320378798778538823075316902483","series":"1.2.826.0.1.3680043.8.498.36252964793350042124970026984052916664","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21449012857520712901884874638467532958","series":"1.2.826.0.1.3680043.8.498.67840665032607043299627393462017340437","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21605506484276750075895228202564072044","series":"1.2.826.0.1.3680043.8.498.43138885607690036363029502611030494413","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21520719119040558609068952147312604007","series":"1.2.826.0.1.3680043.8.498.81386786135438175200303028554043057252","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21634792491972658759276025394685307053","series":"1.2.826.0.1.3680043.8.498.86471309283202983932030505791954142461","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21726746202429506433532477857404854137","series":"1.2.826.0.1.3680043.8.498.37324849636647662864138201051696470282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21798342483930597563659425189400765869","series":"1.2.826.0.1.3680043.8.498.44820694480342764715632352215502856831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21558570608416990974711670673431746511","series":"1.2.826.0.1.3680043.8.498.27841570152025048691101791820344894911","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21868734450232158209414868901539475548","series":"1.2.826.0.1.3680043.8.498.53719300505946698075151357454327376426","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21904421326144238681996713485932410596","series":"1.2.826.0.1.3680043.8.498.12319844349576654616228864926711919952","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22027914323057444746912555313058666950","series":"1.2.826.0.1.3680043.8.498.37701523017165849723682681168729420937","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22007253197523827371600763959266047295","series":"1.2.826.0.1.3680043.8.498.89091739085814481696795551144656138780","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21954482915702797371609582020715955762","series":"1.2.826.0.1.3680043.8.498.13342761481170638701155471613070779961","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404","series":"1.2.826.0.1.3680043.8.498.52627968695314286677230767737762639225","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22057596860528658818833903685827585805","series":"1.2.826.0.1.3680043.8.498.44940057266941549625663772742466711244","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22137222939487357840166329770642627189","series":"1.2.826.0.1.3680043.8.498.24144330983205859585213267101773123114","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22263857526032448247006468310072414129","series":"1.2.826.0.1.3680043.8.498.12467908070990617384352154517671611120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22126824950877890715014783370071292997","series":"1.2.826.0.1.3680043.8.498.11708393900969262433918234369635905893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22394208178597637441149549895512290013","series":"1.2.826.0.1.3680043.8.498.73164823067389043495794059636850558720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22398731807962276404889164827238996005","series":"1.2.826.0.1.3680043.8.498.13111656775499769414585826572445331952","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22426390571898968732573184721547490379","series":"1.2.826.0.1.3680043.8.498.20034393726740695827868628178375684459","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22480218016816209183384996714060953859","series":"1.2.826.0.1.3680043.8.498.38385958290037302131631259071754721121","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22338400095583099753286935111647807038","series":"1.2.826.0.1.3680043.8.498.12021257254805965301620368299847746130","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22583985355672251619550173090948161714","series":"1.2.826.0.1.3680043.8.498.91758844599580325032197314958890734576","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22617494262793136174349268379191631799","series":"1.2.826.0.1.3680043.8.498.95549952506945202658164742141451282944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22594742481654038247097269358877852869","series":"1.2.826.0.1.3680043.8.498.35351827342679928781284462582776260340","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22656222254273662937226093008714426490","series":"1.2.826.0.1.3680043.8.498.33563144347046850616229494586496407515","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22938598188283919466040132624303635817","series":"1.2.826.0.1.3680043.8.498.82455263833927974735763308466906170953","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22684505799921376181913295621462268931","series":"1.2.826.0.1.3680043.8.498.88072976603142953231258525259898350682","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22990786875064680564672188701015740170","series":"1.2.826.0.1.3680043.8.498.45428601233787588940719158936408241733","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23046618786812839762801821364158313134","series":"1.2.826.0.1.3680043.8.498.61253592500888755458885642191150711706","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23065868481290440008286663193810913414","series":"1.2.826.0.1.3680043.8.498.10193421103817217173710546319909670101","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23081294963277525274551516232134221416","series":"1.2.826.0.1.3680043.8.498.95721951658360140578677457978901346662","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23107903961514228607587963030481481744","series":"1.2.826.0.1.3680043.8.498.10875220221313346496926201774349889539","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22867776500261605752195206933939172136","series":"1.2.826.0.1.3680043.8.498.83970457906240706183142121956305461640","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23422918444346962998731376196027420200","series":"1.2.826.0.1.3680043.8.498.10598348235067725994506140798807934807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23182005205627494484490011351673355568","series":"1.2.826.0.1.3680043.8.498.94858703494067451594142450362968933451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23518857399095666263158269685495375084","series":"1.2.826.0.1.3680043.8.498.64920530860400150884785215149984342392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23677916081296740815989149501889343663","series":"1.2.826.0.1.3680043.8.498.61295311345824665267078009784940978423","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23394601328148358087542766577727876635","series":"1.2.826.0.1.3680043.8.498.59535846654970852606997949668189710200","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23757007145576852496599816072431075057","series":"1.2.826.0.1.3680043.8.498.58645774884044070062120436089823556525","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23719982492392272262684855675016475265","series":"1.2.826.0.1.3680043.8.498.66451332882238713279048443464918006364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23703817737293327679300330167234268929","series":"1.2.826.0.1.3680043.8.498.72244206582468511055201973555490859607","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23911658810234305013984976739924353704","series":"1.2.826.0.1.3680043.8.498.73660640502280898255822238217879749471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24012076387083114283116476459425015413","series":"1.2.826.0.1.3680043.8.498.70698958745501737055918043326456226859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23833933822461809237193998300076847210","series":"1.2.826.0.1.3680043.8.498.58831835034125080065323823604480138601","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24101792822866991033671006059283560916","series":"1.2.826.0.1.3680043.8.498.11046858953591803686991180687662050663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24220732394554303343522053195582322788","series":"1.2.826.0.1.3680043.8.498.34686995036245794859476286697956883772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24209643225939781032698894992032106781","series":"1.2.826.0.1.3680043.8.498.71434245025293911169161210286156018615","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23956169649865635269920565218680494366","series":"1.2.826.0.1.3680043.8.498.72449199995920253342372245914209719760","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24283484945988825507450239904606754405","series":"1.2.826.0.1.3680043.8.498.27423679132125243988246560269212833511","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24302889622350917085868032747474929504","series":"1.2.826.0.1.3680043.8.498.10155922706019224296169209173124101364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24478195415117310702720951368429189290","series":"1.2.826.0.1.3680043.8.498.40944868985820883186888578097766259045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24529300482982370922479679056174585978","series":"1.2.826.0.1.3680043.8.498.10097114342487245269502915278041526907","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24679905857546029995247839380830345841","series":"1.2.826.0.1.3680043.8.498.12181467542559493681973969554785015233","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24635589616467409099788838058892886889","series":"1.2.826.0.1.3680043.8.498.79483590643870138748376315678284207403","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24369332907731805266450307536793220148","series":"1.2.826.0.1.3680043.8.498.62640311124283986072962619363965269491","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24714029982431301486050388345836960910","series":"1.2.826.0.1.3680043.8.498.99863158819314010420145665518660469887","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24563801799946310644914912532825629118","series":"1.2.826.0.1.3680043.8.498.92711891672711597725800074199449311473","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24875052319550288230112434151792125187","series":"1.2.826.0.1.3680043.8.498.13904289038537780714382264138089248722","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24808246532191535554375304081795021306","series":"1.2.826.0.1.3680043.8.498.36489849908781911893005045369053611649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24866943620580666216571030343221534720","series":"1.2.826.0.1.3680043.8.498.41778198489689697090185735136597299205","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25006652485159423287561656594171884310","series":"1.2.826.0.1.3680043.8.498.10325477107615557050655272451438345040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25156207169523051276979595262558652072","series":"1.2.826.0.1.3680043.8.498.11703901486351431573638385778664719744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25107837907285885353845034555062540189","series":"1.2.826.0.1.3680043.8.498.52266696985006802553995766778684779890","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25088333117273619121936521384053709674","series":"1.2.826.0.1.3680043.8.498.36610755000129381543973307033277748301","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25259588756496584739814764896903786709","series":"1.2.826.0.1.3680043.8.498.19601609373697687242941611684804006528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25206096958520407302630747466352252249","series":"1.2.826.0.1.3680043.8.498.61784024144215772647451629850762732203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25292715244575997322139502072242615319","series":"1.2.826.0.1.3680043.8.498.28604567593800120083068093490375531431","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25440342930186170531037102906687829898","series":"1.2.826.0.1.3680043.8.498.56501586135753301727666816789325474692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25188196199869378521949969245728443004","series":"1.2.826.0.1.3680043.8.498.48302518121791655153369201262648066168","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25487225187305802150215261512092051199","series":"1.2.826.0.1.3680043.8.498.21806478666433648866400815813726815082","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25394815405242666778213609728680620077","series":"1.2.826.0.1.3680043.8.498.67191042378662646758542482404950233588","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25550871519416839807700061317524004550","series":"1.2.826.0.1.3680043.8.498.65852441501044991195546161156135384528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25604547074218147115077644853162295379","series":"1.2.826.0.1.3680043.8.498.99274629643657997628095276398694519140","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25635522141852534580088120697977308505","series":"1.2.826.0.1.3680043.8.498.94750949776873978043441025135794892658","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25663827732039068221184137540713755716","series":"1.2.826.0.1.3680043.8.498.20302883874262945466541880307983057917","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25758766525934930593672922246867704103","series":"1.2.826.0.1.3680043.8.498.69982942818557643410872672636635808827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645","series":"1.2.826.0.1.3680043.8.498.16014775672714479586841217424844144524","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25650384476012633612528314070894186451","series":"1.2.826.0.1.3680043.8.498.58079973226727116191718249042410618176","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25512240159357553134500245590935010104","series":"1.2.826.0.1.3680043.8.498.77771300823080189174043221865246457754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25818541516137783435601858274495597736","series":"1.2.826.0.1.3680043.8.498.62853151767727481067428914076985434900","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25899681172115698926926115441929702502","series":"1.2.826.0.1.3680043.8.498.12050297223602214096498832657180708116","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25909078145370189195643713208718315003","series":"1.2.826.0.1.3680043.8.498.10738289822348881284081946675136600621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25766387275525303860953321689718246405","series":"1.2.826.0.1.3680043.8.498.70930463164067681093740814539475625731","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25792216289349936450377008453709520260","series":"1.2.826.0.1.3680043.8.498.48091815231885820195492095956646344826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25953117039631879376428439063599611036","series":"1.2.826.0.1.3680043.8.498.55205950922253422207601786313881254263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26066524302907849854339129271283426208","series":"1.2.826.0.1.3680043.8.498.92765594638588556968424369815661239560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26006372873438224908989402529530833528","series":"1.2.826.0.1.3680043.8.498.11098294144429090729873382324403176159","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26002972602557466222471260313257965551","series":"1.2.826.0.1.3680043.8.498.84676258201798022827952352407242236870","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26325951519908477683595799515094966069","series":"1.2.826.0.1.3680043.8.498.11594962606190230857651212272662704943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26351577824204863242390892891061594387","series":"1.2.826.0.1.3680043.8.498.94921416018651877531616438339091162755","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26257425287121280175418060407044075705","series":"1.2.826.0.1.3680043.8.498.11453327356994747436277817171909225360","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26150637525321126007832687915610995568","series":"1.2.826.0.1.3680043.8.498.12746956410710418596157813720434191049","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26474456265903975319674669470473661691","series":"1.2.826.0.1.3680043.8.498.93440724921378400887061085261420859876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26408081008482310696738068933718440200","series":"1.2.826.0.1.3680043.8.498.59195579324867609266232704145933812409","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26512333939158074470511885252122191522","series":"1.2.826.0.1.3680043.8.498.88538147950035454399677717489089021523","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26542435427465415008860869796504112727","series":"1.2.826.0.1.3680043.8.498.57928617754493153356603211876634113625","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26725172078893286313351418435377708028","series":"1.2.826.0.1.3680043.8.498.66243263472565801690392415235697276809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26792665054823243997838322477587733719","series":"1.2.826.0.1.3680043.8.498.58247928682956119033091604697458955488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26843088620128166809417361066548447201","series":"1.2.826.0.1.3680043.8.498.23429532435923317882737919863393787881","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26875920364303755911342338961252130512","series":"1.2.826.0.1.3680043.8.498.36299462285730520679739446412090867717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26651323353208729423825552073457972796","series":"1.2.826.0.1.3680043.8.498.65974744329852200008404844678774436037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26946980494141305258556994153410856428","series":"1.2.826.0.1.3680043.8.498.57588476976844048739570421456941616314","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26989584479346775218799121753035575730","series":"1.2.826.0.1.3680043.8.498.94275894786284219518779660898421068326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27157074709305277871045901905120405881","series":"1.2.826.0.1.3680043.8.498.38611065943699038625337996553037009678","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27034189091164693165076510006436405194","series":"1.2.826.0.1.3680043.8.498.11709067061959921301709118093893621555","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27023892731021470476500135946317920771","series":"1.2.826.0.1.3680043.8.498.43346636675574017731147858565145038986","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27184343589256623289925252380526385858","series":"1.2.826.0.1.3680043.8.498.66625912927137345813421358918140064349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27197413293500684424735285039923168926","series":"1.2.826.0.1.3680043.8.498.13017390175403382006093918637762022479","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27191574009501966309382307682020135126","series":"1.2.826.0.1.3680043.8.498.18712030622582066168405613486084424934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27311413427137994435573903541974122355","series":"1.2.826.0.1.3680043.8.498.60991584585177540090554423796179541819","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27324284859290040148888098857880927353","series":"1.2.826.0.1.3680043.8.498.62566401017985565680067495197067306024","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27398034409980350921804398626171135707","series":"1.2.826.0.1.3680043.8.498.10719273385974317401786009147249312120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545","series":"1.2.826.0.1.3680043.8.498.11005397036885713173471007818157207010","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924","series":"1.2.826.0.1.3680043.8.498.40175815168544280787064836169050383117","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27593710915588175306316595263625481601","series":"1.2.826.0.1.3680043.8.498.12114542201813070385566527173103873033","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27655410895168998898506695863000928400","series":"1.2.826.0.1.3680043.8.498.71437527401457581741993499501541792614","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27644321991318957631114727895883331622","series":"1.2.826.0.1.3680043.8.498.52271027499773005902476328956986428597","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27460982183399364904497832933739191826","series":"1.2.826.0.1.3680043.8.498.65283862457973884606747711262385892706","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27671779587445983390840223475173360482","series":"1.2.826.0.1.3680043.8.498.10828432721478958598777944567691579713","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27793455627426963285485957211105588905","series":"1.2.826.0.1.3680043.8.498.92169048048224785932525369156937674695","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27710718386318170070605046744941324421","series":"1.2.826.0.1.3680043.8.498.23608464296858835751346871661030361719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27723573213501259053792522608339812759","series":"1.2.826.0.1.3680043.8.498.95084503134941845884889569853189381775","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27883371281368994280894952749865160316","series":"1.2.826.0.1.3680043.8.498.11258616468380709778286377559672214003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27921111050680927418542588425335218896","series":"1.2.826.0.1.3680043.8.498.16126717924259050409338714730334553370","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27934397251575987048967084987994202973","series":"1.2.826.0.1.3680043.8.498.80288323492836220469622767719309267398","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28023795363827851445172311810918021563","series":"1.2.826.0.1.3680043.8.498.69472479603055682081456120300811349755","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28132506470634542229651932141193521661","series":"1.2.826.0.1.3680043.8.498.55630768274297148095017472869757455369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28040645186226259590182363324146810031","series":"1.2.826.0.1.3680043.8.498.11308811876295785249998157986174767879","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28153286315438977482271934270979508349","series":"1.2.826.0.1.3680043.8.498.58612477617378653043043836679400605260","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27990965288663885936880638302158503894","series":"1.2.826.0.1.3680043.8.498.41253946212521027276401070663408023771","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28269350392124251309834584420564806882","series":"1.2.826.0.1.3680043.8.498.11804609068875260190563982387113933829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28295265939946677460838974253299313798","series":"1.2.826.0.1.3680043.8.498.52258623185019475586514103975709512808","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28220626968612492259133318770141901567","series":"1.2.826.0.1.3680043.8.498.13227919874742439146561979562246858756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28481442608487320644829189127206806471","series":"1.2.826.0.1.3680043.8.498.54087459096440000675947298622850936363","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28490861391717560759667589681288427737","series":"1.2.826.0.1.3680043.8.498.51875938064120641177842533408405915524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28360174615866226538417948042981668998","series":"1.2.826.0.1.3680043.8.498.53274268993873912262853099401412140863","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28634248756653012357434232845788268719","series":"1.2.826.0.1.3680043.8.498.50945933472617748581850471594795663954","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28626044336302232656987963623687293705","series":"1.2.826.0.1.3680043.8.498.57168686297116270200458175628851167952","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28579372595938810820078670002733005121","series":"1.2.826.0.1.3680043.8.498.27461687043074380148263549706995950957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28333127304382930639614800697959296306","series":"1.2.826.0.1.3680043.8.498.54806612110559733802587847747311225541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28646304373057947277669139986982208473","series":"1.2.826.0.1.3680043.8.498.97830809997841763552096385114268940649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28697216332632397717077153882513537456","series":"1.2.826.0.1.3680043.8.498.64198485014916044731082964364641454968","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28770594655815108010532953376271504596","series":"1.2.826.0.1.3680043.8.498.62586375698720443515334672257887439703","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28848511016500781798837621261660844237","series":"1.2.826.0.1.3680043.8.498.94688407362016678917453183135435406606","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28670999234453413675502551916979760344","series":"1.2.826.0.1.3680043.8.498.12426999626391166853024407273242575203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28888941572409703700315590339486528256","series":"1.2.826.0.1.3680043.8.498.50372843936846591946463949251143137060","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28951331578236330522646806432827370713","series":"1.2.826.0.1.3680043.8.498.15993446535734977805925764750137300394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28944156754203202831026290463520445489","series":"1.2.826.0.1.3680043.8.498.13222580064867972960615237533635561703","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29039872266866623711060910233238440882","series":"1.2.826.0.1.3680043.8.498.45633745508258340793769769465674621363","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29294899952044311440141172596030689586","series":"1.2.826.0.1.3680043.8.498.58523674359834850755287860305828291536","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29280448517432444306374036429824425668","series":"1.2.826.0.1.3680043.8.498.91949805242690199102037469908532939344","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29090158364700006304493717187346344597","series":"1.2.826.0.1.3680043.8.498.12065886853379919198667511759378467059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29371692234061436334815897970798625595","series":"1.2.826.0.1.3680043.8.498.91172542512879997184707830584288385693","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29384016102624554484300773136383156153","series":"1.2.826.0.1.3680043.8.498.46419368822169352673062925700448075522","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29354469534263517460113311852821643665","series":"1.2.826.0.1.3680043.8.498.11387144052728381268733197772531059805","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29509443439138257275071354031180789464","series":"1.2.826.0.1.3680043.8.498.12235010397311182973352363212189919171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29443692810539372078922146461173545451","series":"1.2.826.0.1.3680043.8.498.50953208599081059669040533242340574032","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29735852958813434703969490187047665950","series":"1.2.826.0.1.3680043.8.498.10558909789342372313081941930313522121","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29741673719687535294395668150749894623","series":"1.2.826.0.1.3680043.8.498.38296306226557745389634967028542535117","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550","series":"1.2.826.0.1.3680043.8.498.77755509181820937582475082973876194850","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29826129803278661744835136993610646069","series":"1.2.826.0.1.3680043.8.498.70765589626944821951326715248561349022","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29943089198208468441240036404506927883","series":"1.2.826.0.1.3680043.8.498.90496466568651666857198883378568852652","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30066410959464367576375502522994033142","series":"1.2.826.0.1.3680043.8.498.46767896249477842741073392851465641332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30131339052241668380936148362472245259","series":"1.2.826.0.1.3680043.8.498.11418206484416708635755495875682447373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29976611818313906970687136815346573471","series":"1.2.826.0.1.3680043.8.498.13615381895605791518810220403491483883","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30167179890902431464942556467698238073","series":"1.2.826.0.1.3680043.8.498.12798906103197057399985288341668753560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30206987686375669521451035370290737934","series":"1.2.826.0.1.3680043.8.498.12925826706182201648260744986949811831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30233578499383083757206211752001832047","series":"1.2.826.0.1.3680043.8.498.12047646622167765557756960729817305271","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30248682294786626375369374249874501600","series":"1.2.826.0.1.3680043.8.498.33409471063014066720897499555508159257","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30264599677934095429161619981918944098","series":"1.2.826.0.1.3680043.8.498.12839721100603688347292415876204546549","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30330001893437233719772129404417384472","series":"1.2.826.0.1.3680043.8.498.12229653712279685358116641420124270953","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30435446905709054197096833427672604260","series":"1.2.826.0.1.3680043.8.498.13202154906602142048832508927249987859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30517381653283520211188050362394799510","series":"1.2.826.0.1.3680043.8.498.42976605285293134525785050148716244521","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30420245540489522189154634720778032533","series":"1.2.826.0.1.3680043.8.498.88011546352112288419080736262407727224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30613528335797277620843542280889823612","series":"1.2.826.0.1.3680043.8.498.60698472327505392919689467385009926171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30672390215843894382866701432377109444","series":"1.2.826.0.1.3680043.8.498.97580862359175761294206107532310061586","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30537361712821464927469618193893850465","series":"1.2.826.0.1.3680043.8.498.43790559430986159200721075338727728046","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30756534269149851281518502475458067520","series":"1.2.826.0.1.3680043.8.498.13176775658641037067750707124752412085","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30800677497901063887487934675082066316","series":"1.2.826.0.1.3680043.8.498.76205631020323430594776876120165024832","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30836611607735372299752226690388090391","series":"1.2.826.0.1.3680043.8.498.12862742486414290690859287032222106454","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30690232747840004203140392776057147177","series":"1.2.826.0.1.3680043.8.498.78238017247854975109094256729715559503","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30978547345367837000803077743620171961","series":"1.2.826.0.1.3680043.8.498.28224216075672555249442527621565233108","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30942874102560162331831960649006411381","series":"1.2.826.0.1.3680043.8.498.27403545604174100563751898738434969237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31079821574648958955911441626193848961","series":"1.2.826.0.1.3680043.8.498.54545439708447073091623966784611500149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30842406081180392627544336691360658070","series":"1.2.826.0.1.3680043.8.498.10324178443663392447648740746117661057","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31137162827930164367134811210741362416","series":"1.2.826.0.1.3680043.8.498.13390771391295293181997377945234212239","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31171050331393253655760166990008263793","series":"1.2.826.0.1.3680043.8.498.38321923846106835098880872804028350529","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31175617350083221245857813741060042777","series":"1.2.826.0.1.3680043.8.498.43174138506509910117910273813570657499","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31218039240665208717816408042644491715","series":"1.2.826.0.1.3680043.8.498.27270336689033522859786929065744072810","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31282366761674650660824706600044115805","series":"1.2.826.0.1.3680043.8.498.10337595320674826209301041882506482147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31352187408311685267245948214616940728","series":"1.2.826.0.1.3680043.8.498.81467903885132559244409926243868915219","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31257542772176806182696506252925365119","series":"1.2.826.0.1.3680043.8.498.64583368098504208095570023763159350667","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31082592133742929889611857821843859701","series":"1.2.826.0.1.3680043.8.498.78243225249004452720484245056534461366","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31482725509752510642857072190147133461","series":"1.2.826.0.1.3680043.8.498.45564313869439111113290360368037245527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31565705236407722278921437231164204914","series":"1.2.826.0.1.3680043.8.498.10710066472292037512845285857568873930","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31640159014711750808214354395404624815","series":"1.2.826.0.1.3680043.8.498.13241259743717542076295590925847429014","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31421989235441217674752676866539112529","series":"1.2.826.0.1.3680043.8.498.15744409802258555883211217663137150167","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31806829168516955528191705338336428721","series":"1.2.826.0.1.3680043.8.498.11478208645872139676030623479988819669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31790457319528233269194462675995569260","series":"1.2.826.0.1.3680043.8.498.31553749077632991775170013402888000235","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31842732064173292828619477783097732181","series":"1.2.826.0.1.3680043.8.498.12082457636533663464556040745108670268","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31949224274318638796196151105617844024","series":"1.2.826.0.1.3680043.8.498.10281646556830165328535103479196312858","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31898017986762601111404430885432769100","series":"1.2.826.0.1.3680043.8.498.11135750104148915686150411007241606201","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32138869097959653224534642965199855627","series":"1.2.826.0.1.3680043.8.498.10578225146948118089130325618782455337","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32179217289301203414651193280580594771","series":"1.2.826.0.1.3680043.8.498.43108483916565871974555385349800649951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32241219263926672894433308808653719495","series":"1.2.826.0.1.3680043.8.498.11378896832782285876409954787384086007","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32078592787308498852570411454111970699","series":"1.2.826.0.1.3680043.8.498.40610304075575110116890290437520437987","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32299120582647333604770626110392750631","series":"1.2.826.0.1.3680043.8.498.19658184270787063606497668234514537943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32413534416513678586709935453506443461","series":"1.2.826.0.1.3680043.8.498.22725163833998912255899261647148764332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32328699307307661181881949763360098439","series":"1.2.826.0.1.3680043.8.498.38717449866400770234154485012758039449","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32513010438338970418994141548516096399","series":"1.2.826.0.1.3680043.8.498.86286075233878920685722493389502925863","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32571577475600416042179474541176580933","series":"1.2.826.0.1.3680043.8.498.20187026050171240531040218538180218239","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32611545463217490575457831874695918229","series":"1.2.826.0.1.3680043.8.498.20028234198241050550064868368828111409","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32501049887140121289143479773995481982","series":"1.2.826.0.1.3680043.8.498.36822350193653135956500787567492932249","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32723296226443325838892542529446214677","series":"1.2.826.0.1.3680043.8.498.14747369012635818548397816412407756025","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32680606387445562892045224227371304764","series":"1.2.826.0.1.3680043.8.498.83725569270209380550746733386381525626","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32479930837336533083798155873656743007","series":"1.2.826.0.1.3680043.8.498.27956496427049782272588382155221657369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32784816158214124982843838883515640569","series":"1.2.826.0.1.3680043.8.498.11580597474172807872899738816585565542","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32862727544234948830522036721585095753","series":"1.2.826.0.1.3680043.8.498.11323273634523485717366014592768730871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32916544979513354644075485240094539251","series":"1.2.826.0.1.3680043.8.498.96090528657202608689704159438865162168","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33045687849853291066101186962073873042","series":"1.2.826.0.1.3680043.8.498.92246696273993776490824084188909160807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33247377165700300890880607485836825807","series":"1.2.826.0.1.3680043.8.498.85011217264587180597825393349573575359","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33114390259375873203134285352951765534","series":"1.2.826.0.1.3680043.8.498.12207198393595039801838165878510832067","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33284228284209148349668722931307588402","series":"1.2.826.0.1.3680043.8.498.11929605906760438192078258534215181752","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33387430823462011959198830568803117633","series":"1.2.826.0.1.3680043.8.498.78719651402767666867050654355158872478","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33419086668421177906530808458004386826","series":"1.2.826.0.1.3680043.8.498.70148833134630827630507583728590788074","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33432461096351771607862098991677662209","series":"1.2.826.0.1.3680043.8.498.68966406443738206941896981663636637522","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33620268340279901383196638228030589893","series":"1.2.826.0.1.3680043.8.498.13162125548596943515250247237101075217","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33324670630108603340869986205298694836","series":"1.2.826.0.1.3680043.8.498.94123677686662859198706037554484654215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33707148112045883958228656491167647755","series":"1.2.826.0.1.3680043.8.498.87334914289394475155240230995543665431","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33584205642521728950676369503690551550","series":"1.2.826.0.1.3680043.8.498.29082321020196291724188616434097025423","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33734536545775442043827840341408174347","series":"1.2.826.0.1.3680043.8.498.43976749955528341389968560610821012528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33885566709305241744786004177260915650","series":"1.2.826.0.1.3680043.8.498.10941124769739202129077305591748099232","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33841907095190352493353075864604147687","series":"1.2.826.0.1.3680043.8.498.56075878695554593655776090103946115538","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33947148016115721563601115463364787679","series":"1.2.826.0.1.3680043.8.498.98821953962943681539591460653267217068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34002185870196144582348922158474418674","series":"1.2.826.0.1.3680043.8.498.88250399164668502720981945609427504467","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33990091057243328830745904178686745167","series":"1.2.826.0.1.3680043.8.498.62934915480500775715683283796306479026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34058295828142876023659425586740832405","series":"1.2.826.0.1.3680043.8.498.15159088667870563356727108520764569388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34096149829586714035779314398766243514","series":"1.2.826.0.1.3680043.8.498.31790951453913355194276007245431416082","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34262242337778924864334788374037992215","series":"1.2.826.0.1.3680043.8.498.10456946165339793300901879861646007154","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34126843286927697018624309801437341727","series":"1.2.826.0.1.3680043.8.498.67507426011627402233008146716850853423","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34360298211002490108642003575962873984","series":"1.2.826.0.1.3680043.8.498.10411077001789531688334109497563134092","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34441662343660776323013762699640240389","series":"1.2.826.0.1.3680043.8.498.45094184501007874045048903419545282736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34501976934829436985900782482912019868","series":"1.2.826.0.1.3680043.8.498.11955691145509128772246750970400568757","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34404554752241524210538769325381023947","series":"1.2.826.0.1.3680043.8.498.10432337670053391914454588315957073058","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34572444983416782410188788741787280444","series":"1.2.826.0.1.3680043.8.498.12869312195258343906686717518986461178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34636178772986348716257313987840123906","series":"1.2.826.0.1.3680043.8.498.28301611137198126379728613555819909352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34685905030370793639196564723935583035","series":"1.2.826.0.1.3680043.8.498.56454085218863048442970283058276928820","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34717974828302316666446325243570202245","series":"1.2.826.0.1.3680043.8.498.49941599534334616434639962035593279336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34753112960188810416088070814158959510","series":"1.2.826.0.1.3680043.8.498.13201775091685468100150819667596578260","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34858782760614383566891001610985452165","series":"1.2.826.0.1.3680043.8.498.73601668113829278968486786042457459718","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34795373994453585401035013232069587413","series":"1.2.826.0.1.3680043.8.498.96403389863486380985631365727999961266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34895742092137516664772577251243433579","series":"1.2.826.0.1.3680043.8.498.38560643987634716513969999960851520429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35017973257585430544005763350601015208","series":"1.2.826.0.1.3680043.8.498.84548990433543537802840356042864831585","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35127678462764233938878060108411012978","series":"1.2.826.0.1.3680043.8.498.52388826182812669284820554254401271810","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35027750475162395113025853006474122989","series":"1.2.826.0.1.3680043.8.498.78122515320849502665664785222284861988","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35230417208602644083092414161817929841","series":"1.2.826.0.1.3680043.8.498.13377942934303287691559584530462125327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35090276406828326770336478753776261776","series":"1.2.826.0.1.3680043.8.498.44925325763767388953226773430518999393","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35572306796137927247415883445612561056","series":"1.2.826.0.1.3680043.8.498.11192182402082894181596918418092194720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35542843700847404584397532402313561503","series":"1.2.826.0.1.3680043.8.498.43177137609954802554536268858854916050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35338198410583984337908932327787063577","series":"1.2.826.0.1.3680043.8.498.16953619970177145296516943097889514274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35273590880523815297008790288567354883","series":"1.2.826.0.1.3680043.8.498.42019772063033694778110117748528938334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35639250779062198905356481136455419313","series":"1.2.826.0.1.3680043.8.498.12446040501596538172337850952626804658","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35790589718590759254855277551289842944","series":"1.2.826.0.1.3680043.8.498.36401239884729678959653379995625077554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35677515667357567818845364400903668190","series":"1.2.826.0.1.3680043.8.498.10842823413593766374647496354280322448","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35769002792415947424087098083674166777","series":"1.2.826.0.1.3680043.8.498.34244579294687755435313096359765156582","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35871201339095431578743898508613853278","series":"1.2.826.0.1.3680043.8.498.20488981391366192812206731920718895398","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35826402289277331555736683320175912112","series":"1.2.826.0.1.3680043.8.498.81809991629531351650943944916077616009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35932368810339127867306228699757809449","series":"1.2.826.0.1.3680043.8.498.11631679242860424886110968019028173742","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35836928928058091851993989259747528067","series":"1.2.826.0.1.3680043.8.498.12411233573720774329809665851041032419","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35906584224533966870202042352292878668","series":"1.2.826.0.1.3680043.8.498.11706659594350190472209279183619816736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923183633813281697521537744486885300","series":"1.2.826.0.1.3680043.8.498.40598871134969985328143479318284853078","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35971292331621735865925455612813197566","series":"1.2.826.0.1.3680043.8.498.10447046166677745504165769826832255856","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36013620117872095227314165026090752632","series":"1.2.826.0.1.3680043.8.498.74751193398992906298008001777566401169","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36131369236532190338276701919104598475","series":"1.2.826.0.1.3680043.8.498.75125032501215162074160354917205720473","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36180248571044206939894546783755557323","series":"1.2.826.0.1.3680043.8.498.90075483889665866942840254141246307151","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36193046155628980489031133466840036976","series":"1.2.826.0.1.3680043.8.498.25849166793413470068760056737386573917","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36083794505435625841225237415909011547","series":"1.2.826.0.1.3680043.8.498.44388661440380168204188307330606001280","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35961087303991442751747064990497975939","series":"1.2.826.0.1.3680043.8.498.72539331612088678728973496374344689176","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36314809560682514762544085885005264167","series":"1.2.826.0.1.3680043.8.498.55479626162951683760214963011743032789","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36488342368050295711098478501722217415","series":"1.2.826.0.1.3680043.8.498.59204176171437983524540040885674352886","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36235892729383338522021416304853301198","series":"1.2.826.0.1.3680043.8.498.12381944849795102583366879455039382451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36431753977367847587395456605556703040","series":"1.2.826.0.1.3680043.8.498.10029175578742911134222249240104542598","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36588608184578739092409073014150374185","series":"1.2.826.0.1.3680043.8.498.15765981510242737982721107482685330755","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36645945703275956149107297016813148264","series":"1.2.826.0.1.3680043.8.498.11713713757559765722122575351746549694","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36743562653862381001280112332273495985","series":"1.2.826.0.1.3680043.8.498.64768450655749775654947747055578204093","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36812284468308169396147366740331573100","series":"1.2.826.0.1.3680043.8.498.36814988399108615227918826257817731618","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36846391753332367835804699576399553980","series":"1.2.826.0.1.3680043.8.498.15078050270513747680260108971117505458","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36930256628357053046574486650916442047","series":"1.2.826.0.1.3680043.8.498.46080586137859461895748997557243185594","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37005775889857426034942587344620888398","series":"1.2.826.0.1.3680043.8.498.12131461006849384115031773794313569563","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36679002788952511154610176614545192374","series":"1.2.826.0.1.3680043.8.498.68396714913531458989398483526112723113","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37030593568962187244853961673610373876","series":"1.2.826.0.1.3680043.8.498.12626148399414995981643675457969692010","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37046961643473249831189497076699393868","series":"1.2.826.0.1.3680043.8.498.11606651026907404375395827153772720395","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37210322466522537523056852160944888029","series":"1.2.826.0.1.3680043.8.498.35864312498110361568963283857810441099","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37110727485959679337431199930441321268","series":"1.2.826.0.1.3680043.8.498.48555931175296328080076107669819928218","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37270291131952413969044156276264153316","series":"1.2.826.0.1.3680043.8.498.39101202862948576297043050262066193754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37347132459817985668038603215490355489","series":"1.2.826.0.1.3680043.8.498.97597150784141718328513012974471048485","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36914870739904705183135763739850515617","series":"1.2.826.0.1.3680043.8.498.23608714593983564886139917622270182366","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37432328401993499374675847888297416449","series":"1.2.826.0.1.3680043.8.498.23211609184924284084409780750548334989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37447511763482014027579634343525200849","series":"1.2.826.0.1.3680043.8.498.94431211035798469723400141633786236977","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37485075271886447709158414670394650122","series":"1.2.826.0.1.3680043.8.498.74299385694069271701942363390919399297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37510478969810784660124747242051166554","series":"1.2.826.0.1.3680043.8.498.49423328785128486447962607398630807516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37558426555271635608942179071280432530","series":"1.2.826.0.1.3680043.8.498.12365536133209712669262500410493029600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37580209294522316445963449455348371815","series":"1.2.826.0.1.3680043.8.498.70975420155943793442364583176780259866","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37591590536870291237852764995372956522","series":"1.2.826.0.1.3680043.8.498.25183362666949904780488688074693159953","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37390494423735156928337187351900345080","series":"1.2.826.0.1.3680043.8.498.35679516599021040814360595424535159361","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37644754953609976838298110418046213654","series":"1.2.826.0.1.3680043.8.498.66315424724605748154875745886012462186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37413349933694832641902637847629969234","series":"1.2.826.0.1.3680043.8.498.11429153337473666956466996099813459389","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37838990693631869129590283596686992710","series":"1.2.826.0.1.3680043.8.498.74639140234447316010010921661359291327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37713008080610827760634302982073393454","series":"1.2.826.0.1.3680043.8.498.39348354594598785665967877938922887925","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37782918081276484321055690005029111967","series":"1.2.826.0.1.3680043.8.498.77221123034013258041733858238672116361","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37862788793597360072184753016994526122","series":"1.2.826.0.1.3680043.8.498.98541108023240066208951525331910974684","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37885316830093262977685802809269466252","series":"1.2.826.0.1.3680043.8.498.86383484065398833679891167565390164029","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37936061471366103885185528435327650981","series":"1.2.826.0.1.3680043.8.498.34667104890487349493793037424456431064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38005916106577944091708997377832932938","series":"1.2.826.0.1.3680043.8.498.31264376767116642237014611643036420223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38013372055010421273425628837282557767","series":"1.2.826.0.1.3680043.8.498.42129127094102708911169847494475708036","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38104228771638294693272534500734878990","series":"1.2.826.0.1.3680043.8.498.10719605272964681712173995663524518061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38085359363401170553191582038767900323","series":"1.2.826.0.1.3680043.8.498.23636929524529988014865017298984479136","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38056024157303576597111203925091929489","series":"1.2.826.0.1.3680043.8.498.80105575159193880261797234483575077562","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38224750236138877853888930317337541394","series":"1.2.826.0.1.3680043.8.498.46443397946938178297885855174516081875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38333934623775971943734069301776225018","series":"1.2.826.0.1.3680043.8.498.68764078527811705714905544259191522305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38303182087206935163155565441234078389","series":"1.2.826.0.1.3680043.8.498.10460870341351460154982762424110736345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38306262614552497897681886368029860358","series":"1.2.826.0.1.3680043.8.498.78255436042446110609569563505480358063","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38420384692181444061893234153745102059","series":"1.2.826.0.1.3680043.8.498.10816917951596978574877591329696012453","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38365187857616428023213060377488046375","series":"1.2.826.0.1.3680043.8.498.85577956721663519999364120955960763329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38518069950656326194016175319535174299","series":"1.2.826.0.1.3680043.8.498.10960624381838984599523096158190342488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38238963443939714344730201261512911003","series":"1.2.826.0.1.3680043.8.498.10316308766270051334780405903821118184","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38554121831397903041461691318968151267","series":"1.2.826.0.1.3680043.8.498.11792588086631092698865492181810701634","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38457315909459574570206119213798163082","series":"1.2.826.0.1.3680043.8.498.44831340192654938938915373019395826666","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38584486469155131627808759862008577200","series":"1.2.826.0.1.3680043.8.498.44247127955857486447844376514845964808","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38593351637971248239815933638950449587","series":"1.2.826.0.1.3680043.8.498.12319300144991064893410092229888090247","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38628165743951175602803204505216754926","series":"1.2.826.0.1.3680043.8.498.11127908826264139499009460342059673976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38701927697650476092773702704871872598","series":"1.2.826.0.1.3680043.8.498.59021773463332297251111567072270853485","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38774313460931748278644354956408578813","series":"1.2.826.0.1.3680043.8.498.10420914111239958777587518822615335676","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38796519786809225694393617946244012065","series":"1.2.826.0.1.3680043.8.498.10122598437798424763243767994593227680","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38717329577625255474350472405748992219","series":"1.2.826.0.1.3680043.8.498.12587317334470776202576582039061535704","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38862475907405766248096419059100131863","series":"1.2.826.0.1.3680043.8.498.83115594503394841815040184287192451460","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38893522232848525819919092148478879455","series":"1.2.826.0.1.3680043.8.498.12798362081189482047812435954331521486","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38925755872951351471027638563742132144","series":"1.2.826.0.1.3680043.8.498.81072097652111149791698225721457471399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38974023308452365783768081643215200948","series":"1.2.826.0.1.3680043.8.498.52051618150267320257625072634022511215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38935694477580522005505074220300062204","series":"1.2.826.0.1.3680043.8.498.12691262274232911278210568132671685346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38900640030483599614088224967855884221","series":"1.2.826.0.1.3680043.8.498.84550875817793923931598881975616162712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39025433784711757736168236829157583043","series":"1.2.826.0.1.3680043.8.498.63108154505248189452220048875044789288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39068632185013624181565849012182917752","series":"1.2.826.0.1.3680043.8.498.70311844370892917189399097668304574788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39135770176150551756142329948474103964","series":"1.2.826.0.1.3680043.8.498.93456349908692444846137744536468187578","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39020025030013167602991688044932114354","series":"1.2.826.0.1.3680043.8.498.39030510642941718339609532621048133231","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39197976722851762826265579540678924584","series":"1.2.826.0.1.3680043.8.498.53430647148751100189447125585266300881","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39264696726258404531295961801831089064","series":"1.2.826.0.1.3680043.8.498.97460077265085610316311160584601218736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39507184510584030713933330702725991027","series":"1.2.826.0.1.3680043.8.498.17737269200215083051360399650048128182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39394842976048577715164357752399606924","series":"1.2.826.0.1.3680043.8.498.85881496688424102419288136188089641725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39578126940710463296572383286587298912","series":"1.2.826.0.1.3680043.8.498.24502522665928850423336246381624105813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39636073039676814228254715705296535819","series":"1.2.826.0.1.3680043.8.498.96277132044074620533664927393948748602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39787654910948635229369478006027304578","series":"1.2.826.0.1.3680043.8.498.12214277700721663196289043682420621780","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39324076659429494696721784912308233388","series":"1.2.826.0.1.3680043.8.498.12142388506904916627251373074129201848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39845300426582562781355154625767485381","series":"1.2.826.0.1.3680043.8.498.33648341154298041087189242627346888015","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39895705984068714346274689335576595443","series":"1.2.826.0.1.3680043.8.498.46344657797566309878917619998649511168","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39468803878805753474151381590569125402","series":"1.2.826.0.1.3680043.8.498.47949671640090717516778355817114443914","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40131899651518057045444739705301716085","series":"1.2.826.0.1.3680043.8.498.29372621530757263793271039351015592739","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39969071491383654382076256213867635377","series":"1.2.826.0.1.3680043.8.498.11568578179446318364929546272483586573","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40041164441539690764994039690598305675","series":"1.2.826.0.1.3680043.8.498.13383888212045272725159464089845443048","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39954567501513957418422043089587980092","series":"1.2.826.0.1.3680043.8.498.13200238010651788904456138675416778309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40257126480600927758308319831518876543","series":"1.2.826.0.1.3680043.8.498.60944805033318733747937886503342949004","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40196271329411009311273016603705560443","series":"1.2.826.0.1.3680043.8.498.56572247829918444188492919547221857773","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40239104167848425290754606013326180693","series":"1.2.826.0.1.3680043.8.498.56325699121060778474166396458756831229","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40308838286515017547707130085914948230","series":"1.2.826.0.1.3680043.8.498.10254199456670070400172004123551929578","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40372643537313871452054321931253684619","series":"1.2.826.0.1.3680043.8.498.16347612013726692196702901325042948045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40437305053062892226978469238653825745","series":"1.2.826.0.1.3680043.8.498.59826455713762123903855800722525447816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40487147942341753806704765355921322161","series":"1.2.826.0.1.3680043.8.498.26885458322348002108818230958511922873","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40547250250566982489293057895661861722","series":"1.2.826.0.1.3680043.8.498.12320302693215633762757989717078479754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40419759559020255481034237210540871312","series":"1.2.826.0.1.3680043.8.498.86450894868366998453779392033932353495","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40608233778918249560056537030108548037","series":"1.2.826.0.1.3680043.8.498.32294815986241189233910309454012506307","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40654763252951267464957826289984701872","series":"1.2.826.0.1.3680043.8.498.12713949210961280060664432652336945336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40718200801895968234499223301900799376","series":"1.2.826.0.1.3680043.8.498.81184949013469079243232113259745454300","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40765829900885943055669918091033014654","series":"1.2.826.0.1.3680043.8.498.30390299542134168637101661817338570822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40828786094337186697222449293183309391","series":"1.2.826.0.1.3680043.8.498.21680093204431026686370199322714060199","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40618283771317160262473004328692613267","series":"1.2.826.0.1.3680043.8.498.53443019868312618088129029705380846517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40930045208346676628855922755791787949","series":"1.2.826.0.1.3680043.8.498.38897059883511605467107105050359413017","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40987687016832998820664054439678138878","series":"1.2.826.0.1.3680043.8.498.86768610664715369117925054000638424661","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41108421820714108307078614688022098388","series":"1.2.826.0.1.3680043.8.498.86973580053278530037652336327643543521","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41036022211588451079713314564568887994","series":"1.2.826.0.1.3680043.8.498.12807109529955425779017639605948889023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41129090437522031508960438281451672212","series":"1.2.826.0.1.3680043.8.498.12355257592382152554066479770544376507","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40898220266193841898869905714214454387","series":"1.2.826.0.1.3680043.8.498.11313546462622325726674711253567961027","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41240503248178304281405515398211762506","series":"1.2.826.0.1.3680043.8.498.10525398442257191796946969725864113576","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41363979247025384202932112601795525615","series":"1.2.826.0.1.3680043.8.498.10603789953732967200262238533421879075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41382780862853765983216810855531499640","series":"1.2.826.0.1.3680043.8.498.40177602036263520145678442006723143216","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41315359092354940082611381451934245995","series":"1.2.826.0.1.3680043.8.498.12634323663109377602621334390576364346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41440206520769979754790425253631809212","series":"1.2.826.0.1.3680043.8.498.62872618336234477395423936480211549581","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41564229176277158115386256329157870722","series":"1.2.826.0.1.3680043.8.498.60849842447599033644401231175794770934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41619164749467663852546208143469380415","series":"1.2.826.0.1.3680043.8.498.78292404626810484188554767365014499372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125","series":"1.2.826.0.1.3680043.8.498.21445139509346804994276778291771624587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41640481150213029810772014344448781554","series":"1.2.826.0.1.3680043.8.498.64977252560264196576406794494400377175","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41662013720606493874198348776219875496","series":"1.2.826.0.1.3680043.8.498.29359490034090057476790526485274870128","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41696746845418651508748386392327996143","series":"1.2.826.0.1.3680043.8.498.11891517304135789991830741865544490972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41769275806689876192405091865340286535","series":"1.2.826.0.1.3680043.8.498.11010179874744075474240305053217270252","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41810826648175757249770990269365542940","series":"1.2.826.0.1.3680043.8.498.21506402456239666347970198239843660884","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41717437729988689273718049162153247404","series":"1.2.826.0.1.3680043.8.498.13363071576670530227091719801291640575","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41856137734822068230231773222820851271","series":"1.2.826.0.1.3680043.8.498.75841503742241227729746199380326083025","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41852534027066121760361035269460584004","series":"1.2.826.0.1.3680043.8.498.69457237268630238679796427348335488524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41990779875873214154237496750949660480","series":"1.2.826.0.1.3680043.8.498.23637413006274253200504108537779807818","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42018683052544325209168649309206539721","series":"1.2.826.0.1.3680043.8.498.82443844526611229845455566195822167605","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42051385452979123325023608442860201782","series":"1.2.826.0.1.3680043.8.498.45210186801802880554313494770387816541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42120645248710860870761681999665860884","series":"1.2.826.0.1.3680043.8.498.11887492301127134204516552143749622804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42389063782063360163798575455522539633","series":"1.2.826.0.1.3680043.8.498.35708818646916196471660659373798699430","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42414269167296543219685897036829241885","series":"1.2.826.0.1.3680043.8.498.52946074053232527354101596503993592196","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42225165085995728538373532053227662192","series":"1.2.826.0.1.3680043.8.498.32726036629512181451144074708324254530","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42566296562794128096734153388840524012","series":"1.2.826.0.1.3680043.8.498.14575497290780438642976645978272611829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42486421970612099441197418307958561877","series":"1.2.826.0.1.3680043.8.498.13393397033425094661536635226898481556","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42609322631348207494943622999125822017","series":"1.2.826.0.1.3680043.8.498.88211791517364551735400922253661933594","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42162585695624963017992042571207363427","series":"1.2.826.0.1.3680043.8.498.14631238864930521559141343838568823061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42587087495729026068405824960397052997","series":"1.2.826.0.1.3680043.8.498.78749730233474915458705451034845508947","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42776652100045349404152232788825670056","series":"1.2.826.0.1.3680043.8.498.63259173970963051349800113151044329748","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42719592405241698450757802131459694665","series":"1.2.826.0.1.3680043.8.498.84735097096117550849768153143604555496","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42847376403722341195083350288786052637","series":"1.2.826.0.1.3680043.8.498.12396651034358109464405978740812955266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42816339547796375466199938922141987775","series":"1.2.826.0.1.3680043.8.498.62676301741403781528310278260355713290","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42896117106271721157113072562739356381","series":"1.2.826.0.1.3680043.8.498.79731873119829053161001901760751298835","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42942844751304929338846069579289911684","series":"1.2.826.0.1.3680043.8.498.69930957197206099939351756012158318551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43014288535685961874615794778401152033","series":"1.2.826.0.1.3680043.8.498.10457668044773937135374836434090902756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43065391907984821806263204343792008938","series":"1.2.826.0.1.3680043.8.498.11281054690325537524609260740035779280","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43127139928509033152155587499980767364","series":"1.2.826.0.1.3680043.8.498.12570091551922316577825074236587985059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43256282639827216310036033378453939750","series":"1.2.826.0.1.3680043.8.498.15426982644755843911632677943707033557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43339007024446439709455570527734115557","series":"1.2.826.0.1.3680043.8.498.84730090174935075674863083383560109798","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43432499267536761263202158662502957684","series":"1.2.826.0.1.3680043.8.498.45822908321687758650107364105372422016","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43355923025859051425546451466843844346","series":"1.2.826.0.1.3680043.8.498.12489642043470249957562520123750060318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43483537125776347707929354413013709736","series":"1.2.826.0.1.3680043.8.498.52975061361936541871034315054008469295","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43300763182450509440662392516941842389","series":"1.2.826.0.1.3680043.8.498.11399842150034766539251069328425954408","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43442933855807685981061557274166869461","series":"1.2.826.0.1.3680043.8.498.46544196769124529155674466903851797962","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42967970121147528831983176261453235744","series":"1.2.826.0.1.3680043.8.498.12783693695015789030453471765095541485","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43587826365647713044374316837330397375","series":"1.2.826.0.1.3680043.8.498.36116704531079444428408371581101477740","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43709699920591091261137846976272625163","series":"1.2.826.0.1.3680043.8.498.12540085297893141670226554247271220544","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43797037142729611453626414920554246615","series":"1.2.826.0.1.3680043.8.498.79564077268699258003978250387401716862","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44002376543340544410495269060824194475","series":"1.2.826.0.1.3680043.8.498.62075178092521603695024564838463424170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43870319732657730742794291443940225011","series":"1.2.826.0.1.3680043.8.498.12958427127534225999817695089189445002","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44149403030093858908988179411336839587","series":"1.2.826.0.1.3680043.8.498.11896933689973799826162263783862328275","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44259687212868598275713310246316207497","series":"1.2.826.0.1.3680043.8.498.14908519531683702841551261327038130859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44333089398623004498721836325723633455","series":"1.2.826.0.1.3680043.8.498.86434272198724802941548874692973780073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44365814642869065241166313231424697474","series":"1.2.826.0.1.3680043.8.498.72877473025852424650319586862558977606","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44321626715626508427287815810252844889","series":"1.2.826.0.1.3680043.8.498.29356226090723869970082572572408735370","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44525218699266472141419416162979441275","series":"1.2.826.0.1.3680043.8.498.10252120765602374592738038949030698493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44417768181231991989438194369583907493","series":"1.2.826.0.1.3680043.8.498.12382868312113705341614030197573645720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44654558140536370040701171884970036040","series":"1.2.826.0.1.3680043.8.498.14148780559834467997176854326167640543","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44758766790156297849747618145458535803","series":"1.2.826.0.1.3680043.8.498.22460381378101812016089106286662739750","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44863396093454506630473164990623458425","series":"1.2.826.0.1.3680043.8.498.43718193914165285997277605418816690057","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44902036121220687805715010480088379768","series":"1.2.826.0.1.3680043.8.498.33699494451325070701190946128148772060","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44713330360073884105636085566230396530","series":"1.2.826.0.1.3680043.8.498.95860884194640441986138526568251990060","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44441813293962885868294317830142850713","series":"1.2.826.0.1.3680043.8.498.51214921587197073107742526262641733786","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44996883917122941421301655137158093858","series":"1.2.826.0.1.3680043.8.498.71361798960448110450089458803758572159","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45059917085750212913588683242522799083","series":"1.2.826.0.1.3680043.8.498.98841607867643759857905840918379309122","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45082393649917309927267478072569185694","series":"1.2.826.0.1.3680043.8.498.65285442592369383354797615628086880826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45170517567841608067443743961197123093","series":"1.2.826.0.1.3680043.8.498.13310056883669203243232669647401585738","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45286493874121093392171044034263851961","series":"1.2.826.0.1.3680043.8.498.25470227410465704813020697363007658982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45316453179223822297459860764883030492","series":"1.2.826.0.1.3680043.8.498.80936737767794943904471001854957236517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45332185607602495786497298658937575959","series":"1.2.826.0.1.3680043.8.498.60508153328877400477086621469417354872","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45392604471538776666611947492530314159","series":"1.2.826.0.1.3680043.8.498.27773551621963378907307622046873805834","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45368532788186860828867663606651818835","series":"1.2.826.0.1.3680043.8.498.17696489087888680499237352685551955557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45454539422542821586939369440135728144","series":"1.2.826.0.1.3680043.8.498.80397191343031032405306579704242017269","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44879085997210091159117906674172231025","series":"1.2.826.0.1.3680043.8.498.26543928649623342796324443695003832198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45251306032215692953561852960585282987","series":"1.2.826.0.1.3680043.8.498.79833690781863499808608007933236134837","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45463321406810606039579747001339703552","series":"1.2.826.0.1.3680043.8.498.78050800848224905565105639475009781042","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45644703214235636656820126625301033696","series":"1.2.826.0.1.3680043.8.498.51997968163310990164225131086082306642","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45504348071514081636909771148066956410","series":"1.2.826.0.1.3680043.8.498.10932995796154723393614655318975049915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45755198489211010290512228906807170657","series":"1.2.826.0.1.3680043.8.498.11828451213207624405261972371979611206","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45811964949115542204306831813385709206","series":"1.2.826.0.1.3680043.8.498.39863975188523831910221948737770253903","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45865541491386404818767748310365919685","series":"1.2.826.0.1.3680043.8.498.97822902823042260030610084651944252656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45894475516096841241467193775629362381","series":"1.2.826.0.1.3680043.8.498.76393793546909617233425983036192992279","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45625191292968830335971647388746041117","series":"1.2.826.0.1.3680043.8.498.21259113117036499581080551655283673602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46036209147177723970966724006242055397","series":"1.2.826.0.1.3680043.8.498.10842149065587004589520919233498044670","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46172451572836863687073454997158931662","series":"1.2.826.0.1.3680043.8.498.15047501682855032768298074497690380853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46052469424267861803032899866428547156","series":"1.2.826.0.1.3680043.8.498.25290536932163484205591104991004626891","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46268820642949526111807948525363811264","series":"1.2.826.0.1.3680043.8.498.76334460953828722309554712708964969867","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46297226877524853395345229615300658523","series":"1.2.826.0.1.3680043.8.498.10369844332965475649555077068941146428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46228189366201925238527911905072482095","series":"1.2.826.0.1.3680043.8.498.93670261003204004137156665992962419914","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46001596547902477402837300565271742663","series":"1.2.826.0.1.3680043.8.498.42210519213192645850547619797319911443","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46452612704507986055186132736451572677","series":"1.2.826.0.1.3680043.8.498.25690215352133586165092065907077676623","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46700895961751940291334058396794206479","series":"1.2.826.0.1.3680043.8.498.69622581756209071404885264267059035198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46309540159823359540686782651264853894","series":"1.2.826.0.1.3680043.8.498.72104183779196527264713602286002355633","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46407639440476738746130651250276012493","series":"1.2.826.0.1.3680043.8.498.12897773444622584235007697835650348126","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46715670142711601593792518196789347552","series":"1.2.826.0.1.3680043.8.498.27100211009487313627553228568735004485","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46770589129181839292542004762882468541","series":"1.2.826.0.1.3680043.8.498.30827499653737521035361403786649585948","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46797419946872257347118338518757027995","series":"1.2.826.0.1.3680043.8.498.37345539972964028015406501463318188342","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46506161284114284997220751358509505380","series":"1.2.826.0.1.3680043.8.498.37541876840553918608326722372675173126","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46842734152337184482152392783260161845","series":"1.2.826.0.1.3680043.8.498.47711191367401328860379398762983454922","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46925959240037679795735705370016021232","series":"1.2.826.0.1.3680043.8.498.58103528662223727346669910246362454587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46893956584698923808118623450380169212","series":"1.2.826.0.1.3680043.8.498.21621349189556712551159070418990170134","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47063694709201049930726855357000925037","series":"1.2.826.0.1.3680043.8.498.36979995627551181782560535080915965821","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47239511009810375185594384849101241004","series":"1.2.826.0.1.3680043.8.498.10023375006207708943543356180091297293","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47094981922544354899183959858078311972","series":"1.2.826.0.1.3680043.8.498.56537160926328122162193063250728009750","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47336007784831556446570294227732688335","series":"1.2.826.0.1.3680043.8.498.68858693922481485003921734139297543988","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47267174205472073784818634988669693642","series":"1.2.826.0.1.3680043.8.498.13282666580748334629961163840841140497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47391919345230380216864709248400573953","series":"1.2.826.0.1.3680043.8.498.80643072670279557972356597043046284723","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47531034194300620953276025401018051246","series":"1.2.826.0.1.3680043.8.498.13116575153430483635389961146254595213","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47465512474718256570933149642267926825","series":"1.2.826.0.1.3680043.8.498.65617286662135969783597937766337935137","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47655269404919497034805209368250028981","series":"1.2.826.0.1.3680043.8.498.12397566320943383265870338603540082635","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47753549323738763346112850705924843959","series":"1.2.826.0.1.3680043.8.498.10046105027332935089397099816715465807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47678121014268671436574141288012829166","series":"1.2.826.0.1.3680043.8.498.25326084156910558360537652538920970154","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47855330524708374523226349383799712473","series":"1.2.826.0.1.3680043.8.498.61803824255873110166848023759356747224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47870074144891988520855535250618228390","series":"1.2.826.0.1.3680043.8.498.76463723776195899281784860547011633705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47925688677871325890380893763458019552","series":"1.2.826.0.1.3680043.8.498.42897217769815591878922010519102067733","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48033467661219494489168041469309669510","series":"1.2.826.0.1.3680043.8.498.79968495249107284980718393223352025844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47036446751578280498105392189811189787","series":"1.2.826.0.1.3680043.8.498.64355107935782104066266305903711039975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48308485987218296338173512060246238691","series":"1.2.826.0.1.3680043.8.498.11035903844741280695764529240455336555","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48417903971145834309474934717837250776","series":"1.2.826.0.1.3680043.8.498.66494020485767343881767722178613012956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48367523758673249284754003765836877239","series":"1.2.826.0.1.3680043.8.498.12936188184179050131677321288106754777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48141366506505611287110229972010695331","series":"1.2.826.0.1.3680043.8.498.66075560381006507615118757750984370166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48486018503198264262324670490116120612","series":"1.2.826.0.1.3680043.8.498.71428030021713964660397032030647193444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48564780460694033783288540237012068889","series":"1.2.826.0.1.3680043.8.498.72390389383901270101278573800455181031","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48697800450463192944610588533290320545","series":"1.2.826.0.1.3680043.8.498.17471574295776086022990063661313915009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48771187298294322173131630438467163244","series":"1.2.826.0.1.3680043.8.498.11827182563340099784785171494407395081","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48521453765449688946535921546652765399","series":"1.2.826.0.1.3680043.8.498.72893922763862489754853559371973877142","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48784244479029704181755878324450207126","series":"1.2.826.0.1.3680043.8.498.58706026685033161589109237855658480444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48872083192774100826667097552356058756","series":"1.2.826.0.1.3680043.8.498.68089489344636251286485750844209246770","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48829151552094424673693975665028690142","series":"1.2.826.0.1.3680043.8.498.10632133550407059266223628780480925705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48933020429865163393792899740045803587","series":"1.2.826.0.1.3680043.8.498.12768664226383161706301551515474571075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48977699132767849812826572970218325717","series":"1.2.826.0.1.3680043.8.498.12004976236821487486328249981202969009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49047255106144196602523585429683869547","series":"1.2.826.0.1.3680043.8.498.66764005589618487200000098286293344367","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49054454369199540406654153762248931799","series":"1.2.826.0.1.3680043.8.498.12482239242973924496821147551828939336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49163335688297423808344168197551387860","series":"1.2.826.0.1.3680043.8.498.12394696304676294591664815860431403657","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49219586198038249030222134813685536295","series":"1.2.826.0.1.3680043.8.498.96468768646607484664746598872174996245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49347024975371539869871751159907613638","series":"1.2.826.0.1.3680043.8.498.20170762064528969403940517026824374708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49394404966674633152360206835416205330","series":"1.2.826.0.1.3680043.8.498.71741193407811557522932512822075934261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49500752658775616590911569693817273205","series":"1.2.826.0.1.3680043.8.498.11765695481925969652870327858412940336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49492260217998945587168161555623476238","series":"1.2.826.0.1.3680043.8.498.93564524270110483248180515499690701470","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49660659440268002826070581802625582554","series":"1.2.826.0.1.3680043.8.498.10108739743808970566086723402422161936","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49583348226932515919984811293816232513","series":"1.2.826.0.1.3680043.8.498.14075773371069006983230344021399814412","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49847104080351361589960954776757484668","series":"1.2.826.0.1.3680043.8.498.32736232611459023563207529313478386206","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49443962795025989994487968185070137217","series":"1.2.826.0.1.3680043.8.498.47019701799480532270346712503407619521","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49855677166670499939366963036804180912","series":"1.2.826.0.1.3680043.8.498.13062751732114222646435238074739425423","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49887758195164927372912584780941789021","series":"1.2.826.0.1.3680043.8.498.53385578468689069090891305974314156402","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49880369313850883713947031303955494125","series":"1.2.826.0.1.3680043.8.498.41337772693062653546694724364012169726","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49703744074809050221928706774732375079","series":"1.2.826.0.1.3680043.8.498.47528272842021679768905677224521930202","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50073674465735509400964384004695041387","series":"1.2.826.0.1.3680043.8.498.89364310140827645513535396248419905961","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50051845967696843074391018845332480653","series":"1.2.826.0.1.3680043.8.498.11329535386073203259378044302144210248","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49942556484410764813681696157580687854","series":"1.2.826.0.1.3680043.8.498.78657280650051042689079064320386647824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50144715912379960437506930683683218333","series":"1.2.826.0.1.3680043.8.498.37013163087039628146716832751973724827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50259348432783983788186173025555513832","series":"1.2.826.0.1.3680043.8.498.97218232628071695392061446729787340364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50183004658202912273462671442829139110","series":"1.2.826.0.1.3680043.8.498.15794031711526027912087848836970939229","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50220991646207533313828550949660577504","series":"1.2.826.0.1.3680043.8.498.10229886895133741367183750825626139919","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50107479256735739327080544431929221135","series":"1.2.826.0.1.3680043.8.498.90785153339963803410222667720184686223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50342726517205151681842826322659275894","series":"1.2.826.0.1.3680043.8.498.43808573417258018894358191268931199766","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50512021577184367495598050974910055550","series":"1.2.826.0.1.3680043.8.498.51199654503589442400652028358636938628","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50648934951234229265309905009982297515","series":"1.2.826.0.1.3680043.8.498.15751456483322938148110549157732294803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50287512028986928372350279452197816661","series":"1.2.826.0.1.3680043.8.498.11163128320435706544120026200757718337","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50667444467788034101157821576803778774","series":"1.2.826.0.1.3680043.8.498.11824830597690701402680193384703293894","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50487158921933419035809338652990508020","series":"1.2.826.0.1.3680043.8.498.10845328004144452774791248000263824533","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50424421287869498746932271082980000521","series":"1.2.826.0.1.3680043.8.498.91626327872531742978210815590550360442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50687743970686088872471288288960579306","series":"1.2.826.0.1.3680043.8.498.27626938106222655670580463266746668516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50810596378229719761556926464290468652","series":"1.2.826.0.1.3680043.8.498.12313337360294908369655813014752189018","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50694733929204626284366818623993960684","series":"1.2.826.0.1.3680043.8.498.99790198615121742953703282060485576145","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50777252749942894325007417893973309310","series":"1.2.826.0.1.3680043.8.498.59214942495588901180505286008102604766","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50819987120895374623148160166392914169","series":"1.2.826.0.1.3680043.8.498.12970089278217998456447986381932293901","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50880891504121357724276091096406248845","series":"1.2.826.0.1.3680043.8.498.10293906837485274981408725924095307244","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50935247372419775856440294835587827419","series":"1.2.826.0.1.3680043.8.498.10856697109921479776655562906712725198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50844227724977857739867646104945090414","series":"1.2.826.0.1.3680043.8.498.73899431387459813614219417486869494599","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51126807380879212261057967703844943102","series":"1.2.826.0.1.3680043.8.498.81675489457265125536918176595283492622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51054592064718900209123281525185013961","series":"1.2.826.0.1.3680043.8.498.11703339209555192717921762390315575458","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51379389400506715529323532959275254794","series":"1.2.826.0.1.3680043.8.498.15192275962186990985570188487341107997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51179760508587500025973916664101120069","series":"1.2.826.0.1.3680043.8.498.10798931103487775387119405266798889791","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51427159850769972151411336025891779423","series":"1.2.826.0.1.3680043.8.498.54133911083922322854205417330299189418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51469564445249084310991187819410005207","series":"1.2.826.0.1.3680043.8.498.74500484507593574529646160351626089725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50980137249034338585131194965993785201","series":"1.2.826.0.1.3680043.8.498.65959640740308057155386513402569801628","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51676679394093588751371807453793655464","series":"1.2.826.0.1.3680043.8.498.11004080566033948299888469325032089956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51752963441899958127321113249182970131","series":"1.2.826.0.1.3680043.8.498.95461319789349931821015117560734593172","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51536234256866063100891626028999490545","series":"1.2.826.0.1.3680043.8.498.50897876716178267434200535830241309880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51297777726540529860433097009654453742","series":"1.2.826.0.1.3680043.8.498.22092265626493596927174038608858796130","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51770131897828080663537020812721042704","series":"1.2.826.0.1.3680043.8.498.53385873872811950592096712850449928078","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51809092313496476873962641559638828608","series":"1.2.826.0.1.3680043.8.498.97792130861152356581305415611223856460","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51828180600648126726057100018010765613","series":"1.2.826.0.1.3680043.8.498.12076013011025274251971985536268719158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51837740845904075589194479205218060915","series":"1.2.826.0.1.3680043.8.498.78625492607468482888071435386138981514","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51917043703891091519740084517167861965","series":"1.2.826.0.1.3680043.8.498.84140941802277057085205257908249476672","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51886877282987244004912794726043055730","series":"1.2.826.0.1.3680043.8.498.41912173808202959218835772267090213297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51981019698056120061499970572334688555","series":"1.2.826.0.1.3680043.8.498.71138294959276248688894887993943561119","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52046170319252016846878355722404423329","series":"1.2.826.0.1.3680043.8.498.99651601794886223011987800620676234138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52067134682915759489080529708019359378","series":"1.2.826.0.1.3680043.8.498.91354054643588300413169287279035483178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52138967568492774495485454357106269286","series":"1.2.826.0.1.3680043.8.498.12541836355613384584309940648985459160","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52204390785696944340286127542242341202","series":"1.2.826.0.1.3680043.8.498.11721940801082625877550198010398791315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52110163572817528199125320120779141456","series":"1.2.826.0.1.3680043.8.498.11950258240993727331726428176943444043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52010165788889734743850946282043771445","series":"1.2.826.0.1.3680043.8.498.10403040086911086184967378140854174658","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52347547859131274836026480407333415929","series":"1.2.826.0.1.3680043.8.498.10708518326502733190262614248000307580","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52409113683924904563676007800283978062","series":"1.2.826.0.1.3680043.8.498.12040973448820930694278836134910451792","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52475562989926669452280424331791927661","series":"1.2.826.0.1.3680043.8.498.98897557057659719818478350068902205806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52520288400853832269613213204540385687","series":"1.2.826.0.1.3680043.8.498.10676182579683697932936705951369524200","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52599302991259061057950834864441634066","series":"1.2.826.0.1.3680043.8.498.63055776176877043329726876905702714979","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52639459702574533721011212286428487650","series":"1.2.826.0.1.3680043.8.498.10223940419623434837128274200185317155","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52846023545870016653608198538437574931","series":"1.2.826.0.1.3680043.8.498.37485573813927669393227740920399862428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52985748674486376309978868177895755694","series":"1.2.826.0.1.3680043.8.498.10039168824072025519905826717252750953","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52785983814702054185130109942936184688","series":"1.2.826.0.1.3680043.8.498.12352280451371900578390254652406558764","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52849154011665864497632297320961015159","series":"1.2.826.0.1.3680043.8.498.10819715869536630944699043768311196336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52437948508356178175558637950043479411","series":"1.2.826.0.1.3680043.8.498.57096356037224843152411398714471986004","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53076472372145933727990308381815269368","series":"1.2.826.0.1.3680043.8.498.55913245911721058156404982584816656439","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53114802640679819441521055029024037716","series":"1.2.826.0.1.3680043.8.498.99010765620654796828970410213076929803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53173579822475854424126184867040523127","series":"1.2.826.0.1.3680043.8.498.10764277235615039143796620062399265550","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53137953094305471774552883960361922832","series":"1.2.826.0.1.3680043.8.498.21973256924729948936083383644075213228","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53221818460567216467597044606040911702","series":"1.2.826.0.1.3680043.8.498.92485476378911074411201590712415921554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53271710112621372496324855809391580130","series":"1.2.826.0.1.3680043.8.498.69648500523933419823813871850860245813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53337684515627962545396495541158882262","series":"1.2.826.0.1.3680043.8.498.12201311195815242035806065399358139405","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53355961816756477601843499985089590263","series":"1.2.826.0.1.3680043.8.498.12979136123941104523472023603848418037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53080172184945962593713560671746700445","series":"1.2.826.0.1.3680043.8.498.56155619375622351100515831265656314330","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53370670740736715669115043403494610200","series":"1.2.826.0.1.3680043.8.498.11333764429624897251170212745760528847","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53408696789721413873026963095773050643","series":"1.2.826.0.1.3680043.8.498.11211470412542842776378466183692669102","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53528011450161116437223332448942321596","series":"1.2.826.0.1.3680043.8.498.45771618087214481407380502706149798981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53587920256280027683523152568832253423","series":"1.2.826.0.1.3680043.8.498.27211273684520777651232861760249295372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53649895712172408476570770924813501118","series":"1.2.826.0.1.3680043.8.498.61311869154323740886641131873364001944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53745429232602455103594227782517963027","series":"1.2.826.0.1.3680043.8.498.11711966572541975014501800354878170276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53893586200869613140666806653041177558","series":"1.2.826.0.1.3680043.8.498.11062063881300773140600251441362629614","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53797236075631231042400235117445767744","series":"1.2.826.0.1.3680043.8.498.15089327665133691289446302818579253170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54000782656400326230944526201670909133","series":"1.2.826.0.1.3680043.8.498.77205003339380736610629178864696808114","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54100786607660749952814251186461344633","series":"1.2.826.0.1.3680043.8.498.27535556969074131662602286712164913788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54132814503202891434380002378663763655","series":"1.2.826.0.1.3680043.8.498.64214117001205001738028680331180712795","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53963759852017695930808272438606800422","series":"1.2.826.0.1.3680043.8.498.97188670163617692947868485591502416737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54142170846176833568961426171739782214","series":"1.2.826.0.1.3680043.8.498.10436180569919265505217102173661907226","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54049224487950229496345484554581532711","series":"1.2.826.0.1.3680043.8.498.23675788620906555960395439801935368105","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54202117421897784469771192940597638508","series":"1.2.826.0.1.3680043.8.498.15032128807527060399238510110268177919","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54316764984549863227549326352573342232","series":"1.2.826.0.1.3680043.8.498.82060053410905304762823187190718293346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54451515695681765924346479727204451807","series":"1.2.826.0.1.3680043.8.498.11893474983607630246492077004834951070","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54374735423703474541190464716308417985","series":"1.2.826.0.1.3680043.8.498.60698982273882425815091569917317119242","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54427039222108917725902677899081931510","series":"1.2.826.0.1.3680043.8.498.75390850793130246399522147558085343358","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54280954106925918796002359856408530043","series":"1.2.826.0.1.3680043.8.498.12477517151032482785183817213864630484","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54628782412849994943720963251113534589","series":"1.2.826.0.1.3680043.8.498.77703395972488245430539939249235360602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54483132338261866032495309977418409299","series":"1.2.826.0.1.3680043.8.498.27564047367701790860685862281846953358","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54669065204808162914386927559244866557","series":"1.2.826.0.1.3680043.8.498.83255167036451047380807424342946827529","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54711709751232783991855540573189433446","series":"1.2.826.0.1.3680043.8.498.12363892816962229043867070649283981364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54833384793167149128371137537527194757","series":"1.2.826.0.1.3680043.8.498.96361693368014829043404445733500274724","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54589880111295324708950851057241927191","series":"1.2.826.0.1.3680043.8.498.13369574978407820866505646710490351961","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54693572065777563403567480644050627025","series":"1.2.826.0.1.3680043.8.498.12335728891098575620494764680946902374","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55015061060346597564205577461227720715","series":"1.2.826.0.1.3680043.8.498.11373042917592221482230445052927276953","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450","series":"1.2.826.0.1.3680043.8.498.71042574996097068332053264058857876735","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55061039227197546099231336400032878562","series":"1.2.826.0.1.3680043.8.498.85949270804303246471522267332321650533","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55300053252365037422745072128951402593","series":"1.2.826.0.1.3680043.8.498.72958460580784894984309449111587233340","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55183153701332792661581741475876132309","series":"1.2.826.0.1.3680043.8.498.78723237032349693052240786943491428731","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54761306437713414745748829128854749469","series":"1.2.826.0.1.3680043.8.498.78955468634322882016539358406025075068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55346033326903331672579723063380636237","series":"1.2.826.0.1.3680043.8.498.81226349878641456467506609393529462840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55067956175569778132589128971795053431","series":"1.2.826.0.1.3680043.8.498.91768480914830996184247683674201344883","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55344075316305815718253234447285669886","series":"1.2.826.0.1.3680043.8.498.97966351191853976799288431535751454215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55415479446253262335685494509470808516","series":"1.2.826.0.1.3680043.8.498.49455215868295981867535241913054550712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55653770127824574123236841437998533498","series":"1.2.826.0.1.3680043.8.498.12468390277939284902684474947679982763","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55583505147281684100202399148099429721","series":"1.2.826.0.1.3680043.8.498.49439582244962016660591612867758346527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55700532609077428304194916132759352518","series":"1.2.826.0.1.3680043.8.498.11615634807475575753590476939580398294","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55729193057949125029301812320609764642","series":"1.2.826.0.1.3680043.8.498.37682614742081303136775134978308830841","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55887747667305814945719986974538814207","series":"1.2.826.0.1.3680043.8.498.32776836092155192850597157195372382334","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55856308312040330141050707531982127791","series":"1.2.826.0.1.3680043.8.498.39509301321493578881133587245514172052","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55808772152667135571939417929724526296","series":"1.2.826.0.1.3680043.8.498.10406532699986191922719700009507171043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55930031395651221279170435293900769500","series":"1.2.826.0.1.3680043.8.498.56537216409438351368489711785653711807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55971241672517044174019654998425819738","series":"1.2.826.0.1.3680043.8.498.12932223535328246665431462492864827152","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56091473614612257999053529249190958828","series":"1.2.826.0.1.3680043.8.498.49508869598642421020617210536308261242","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56054334296747411394367010227992359191","series":"1.2.826.0.1.3680043.8.498.90882956097461595909996473169055546253","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56188634971310070481285525830793231849","series":"1.2.826.0.1.3680043.8.498.55795007966187408596366271748156382386","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56156923369480999484275191793049573490","series":"1.2.826.0.1.3680043.8.498.13349207646731350258418128048604175142","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56266280744044680626884459696038109616","series":"1.2.826.0.1.3680043.8.498.65823399854402610888254161321388024107","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56016606949603829995818303105991224499","series":"1.2.826.0.1.3680043.8.498.35549201450330258442774903768222589872","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56305445410093481072159375249067050124","series":"1.2.826.0.1.3680043.8.498.89109588047233916689468694504286160845","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56428314239264997930645427543149285770","series":"1.2.826.0.1.3680043.8.498.13288978200965881058758545848262205769","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56497268548664917337130798701015887461","series":"1.2.826.0.1.3680043.8.498.72580628806359217317695776871533064141","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363","series":"1.2.826.0.1.3680043.8.498.10759295590848082080643815881145133309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56628956454784173189908113425348977923","series":"1.2.826.0.1.3680043.8.498.52101065048674183881218589539254003684","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56564801707939049324472369774251260607","series":"1.2.826.0.1.3680043.8.498.11504200242638161995527365372579991333","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56762592839620371588520593573846414860","series":"1.2.826.0.1.3680043.8.498.98129622460894334389266520383126182831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56715311294415711253166312546942356882","series":"1.2.826.0.1.3680043.8.498.57548785673612115230367881293424243076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56814987912934301686833132930648230999","series":"1.2.826.0.1.3680043.8.498.59519819550842477630478850913043922287","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56796622077427019119540733986070534126","series":"1.2.826.0.1.3680043.8.498.53404842845227960875554584164370760344","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56880070152758706555016296832842012883","series":"1.2.826.0.1.3680043.8.498.12893559518309477836913128591067991076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56893426177755960410197244250599230068","series":"1.2.826.0.1.3680043.8.498.11566271729318380433790148930971731541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56915079210789596803972894081756325026","series":"1.2.826.0.1.3680043.8.498.17765505690924022616453303633583234022","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56945039212126810522718038626526579591","series":"1.2.826.0.1.3680043.8.498.78861758373352579779134855111487885489","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56991590802231436015843619633497607680","series":"1.2.826.0.1.3680043.8.498.29430991290614692861088244400199589725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57091203959244495164081406481825915856","series":"1.2.826.0.1.3680043.8.498.11344476848250480715777112475837310266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56995522547527264862064392266001620445","series":"1.2.826.0.1.3680043.8.498.10225645168234371900481848884381842564","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57074072196714325300896734167161517369","series":"1.2.826.0.1.3680043.8.498.63859238357870423900638537225378715365","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57342523828166309133982222435655662667","series":"1.2.826.0.1.3680043.8.498.11656694227605366396371234438259038118","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57537915810250993237120953960404180449","series":"1.2.826.0.1.3680043.8.498.97888428325663166219710081540113069574","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57239450717434070615919278253974598533","series":"1.2.826.0.1.3680043.8.498.12425103791033660864267096357337973588","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57703608122587911055931213603182679454","series":"1.2.826.0.1.3680043.8.498.17884987448122685564081849213202580825","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57406640473235989459885313797729459197","series":"1.2.826.0.1.3680043.8.498.84515381480834074167423382829499866161","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57775389384191131322519457853517177210","series":"1.2.826.0.1.3680043.8.498.49145413527911932598141448540567088020","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57829173797358025677240411871600928097","series":"1.2.826.0.1.3680043.8.498.11770568837352199161100194162081145078","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57882532838957577543516026386452055977","series":"1.2.826.0.1.3680043.8.498.13393645696795033569922380286393570826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57845032486422442404222046144765204905","series":"1.2.826.0.1.3680043.8.498.88678952986475668148840226745717484665","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57802887045514681373294932749517914959","series":"1.2.826.0.1.3680043.8.498.96659803019429395702797838960371829307","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58167256197067081436345136244032069163","series":"1.2.826.0.1.3680043.8.498.74901081164233228295547547442551178195","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57951268100539350409054562078476694936","series":"1.2.826.0.1.3680043.8.498.10809972020949643838794011273707905332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58261214953729037661916565473101636690","series":"1.2.826.0.1.3680043.8.498.46690649377474529723346381461716660495","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58280349821647977336008109443058641195","series":"1.2.826.0.1.3680043.8.498.12233699475559550234659210766367010166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58051085126748057986491277367254533832","series":"1.2.826.0.1.3680043.8.498.12116894638967635301719896808551382787","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58482152742263188503728990365891896621","series":"1.2.826.0.1.3680043.8.498.78753042808860765623659810259172766352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58381033890285289577066078050126251787","series":"1.2.826.0.1.3680043.8.498.72475494349549473826672621327439436014","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58346459085553680879151588730895136332","series":"1.2.826.0.1.3680043.8.498.74015290732616651105381860337431974269","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58522891982092784622515590012748600644","series":"1.2.826.0.1.3680043.8.498.41572010764905387486695812564918888245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58585441284202143792468863402617212167","series":"1.2.826.0.1.3680043.8.498.80162545614266206616457646586560794457","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58626635384305289939700801774393536415","series":"1.2.826.0.1.3680043.8.498.76657371903103751887730549652782362032","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58654915518882480245974544469656013058","series":"1.2.826.0.1.3680043.8.498.53852189443443770500219789964623930488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58710813820037448352310278234947865485","series":"1.2.826.0.1.3680043.8.498.37058949112761323714316869501787233473","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58768083117076186082335984587845964130","series":"1.2.826.0.1.3680043.8.498.64666016672288767190448702271449581035","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58857477075095368361468658035369975572","series":"1.2.826.0.1.3680043.8.498.13151516325441759705143035396754331929","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58811528407321581206563412079643094538","series":"1.2.826.0.1.3680043.8.498.61230263310011490307854581684787496955","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59021913092219789232890327780064145948","series":"1.2.826.0.1.3680043.8.498.52814722371678159620722888660893320092","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58416798660375193103814365795347286292","series":"1.2.826.0.1.3680043.8.498.48838688731348020369341886146521225637","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58739535411760425418941874065260802744","series":"1.2.826.0.1.3680043.8.498.13608514272649787035088891518891540052","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59048646289883992375610835845636263714","series":"1.2.826.0.1.3680043.8.498.50293526512650854191940766979335925267","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59098825208018364061073394433706021769","series":"1.2.826.0.1.3680043.8.498.10332524335240845250267029318957223075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58925843611945635442423587804169242644","series":"1.2.826.0.1.3680043.8.498.49644762073616068325471594713988007052","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59394856717216352192037635065925607933","series":"1.2.826.0.1.3680043.8.498.18775492952396399000409497075257066451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59140342594072117638979402278277248612","series":"1.2.826.0.1.3680043.8.498.65854637589917775737077028005421768122","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59262240088132401543519466539885561578","series":"1.2.826.0.1.3680043.8.498.40557953963090001861616677923502826086","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59475691595504590121311599301170361869","series":"1.2.826.0.1.3680043.8.498.69175689528989799640157202128746512982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59426134288436219215525445839849588348","series":"1.2.826.0.1.3680043.8.498.74732251480752121892491794331601677338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710","series":"1.2.826.0.1.3680043.8.498.10375380880125329105761253769836090010","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59527896591755229337596505685198957637","series":"1.2.826.0.1.3680043.8.498.10823647123457650324015139368706970328","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59646951619367295963072805933618947027","series":"1.2.826.0.1.3680043.8.498.19350923149440641309299102142520984321","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59826277112503440743678968423276737254","series":"1.2.826.0.1.3680043.8.498.11750945083725657768194881469654632286","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59923507410103964137819195926587517759","series":"1.2.826.0.1.3680043.8.498.67562553441066946185285764639779819729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59938475193950583662125713924156951751","series":"1.2.826.0.1.3680043.8.498.76023056879550289975367782135123219541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59948559729295284519115137429127740378","series":"1.2.826.0.1.3680043.8.498.49433480296080733651564500317569066008","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59835677513721120155185666132462195647","series":"1.2.826.0.1.3680043.8.498.20799225460429305477333996307478640448","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60088605286413557861235417321985538357","series":"1.2.826.0.1.3680043.8.498.58242982644016380335454541196960213906","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59973351395253648445741322652773810641","series":"1.2.826.0.1.3680043.8.498.12434568205361203672125848894553366771","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60163612617486071760358965933890738162","series":"1.2.826.0.1.3680043.8.498.10278857190604912362726848501313261316","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59708410902079108538978687294461753235","series":"1.2.826.0.1.3680043.8.498.31370101161880814134258684005828582052","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60227430483444032531479844365716485643","series":"1.2.826.0.1.3680043.8.498.10732649912441277434464807857531029203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60259828181878278590614805828898992241","series":"1.2.826.0.1.3680043.8.498.70538157703514937956523260363544558504","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60439571175176992075370053747045234796","series":"1.2.826.0.1.3680043.8.498.10825946149839758813080120209661235373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60465293103574559546982440125572490190","series":"1.2.826.0.1.3680043.8.498.12714310266665605619269022694808522530","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60548477793215320533051194155539307724","series":"1.2.826.0.1.3680043.8.498.63058901296363262835605473389537022762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60430475172382878092821699001847797153","series":"1.2.826.0.1.3680043.8.498.85119702344140055742843597993486816171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60347768295306794807599672836772410567","series":"1.2.826.0.1.3680043.8.498.47880188565725651175810909334924754168","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60633008040453481523409852849000823589","series":"1.2.826.0.1.3680043.8.498.83902973031821689027383758344235084570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60611146141493700985009985723275645474","series":"1.2.826.0.1.3680043.8.498.87406579760676680645864924040006335087","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60477027832057450004004942199870813886","series":"1.2.826.0.1.3680043.8.498.54282762281956425318188232401394621049","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60931716414402855657852057585917605786","series":"1.2.826.0.1.3680043.8.498.65841185427729253304347742369091464205","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60840220723824894619895264673401129352","series":"1.2.826.0.1.3680043.8.498.88199497859146747052763172835898912365","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61148578767041261085865406067025346106","series":"1.2.826.0.1.3680043.8.498.29941615005230393154937263850750503979","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61024356179284390979230045743542661223","series":"1.2.826.0.1.3680043.8.498.99093840903677846149990163021816925952","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61072566792285434404885416160728089641","series":"1.2.826.0.1.3680043.8.498.27502239812376656802939626249915143906","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61220609061901632500124675031032586841","series":"1.2.826.0.1.3680043.8.498.22730006317809985769212127610977277379","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61333047594780344399535845209036589563","series":"1.2.826.0.1.3680043.8.498.32420913085631547269285815155747635868","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61238657014751597736608285205361041675","series":"1.2.826.0.1.3680043.8.498.79778606564515576839386076101251570622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60723378470101669902158229125336455520","series":"1.2.826.0.1.3680043.8.498.12895628426521618643656578239031759540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61435096069921474761812731150911059491","series":"1.2.826.0.1.3680043.8.498.19159852753525900249365441169863961022","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61529451559250327125552112075119161933","series":"1.2.826.0.1.3680043.8.498.12100431991368403403251599410433448311","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61490475141024845836646569993549106628","series":"1.2.826.0.1.3680043.8.498.13201401834416240781255539328821254442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61570904948128887877542043572765432417","series":"1.2.826.0.1.3680043.8.498.12898653598437372012868887151090998072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61797111379749329977228618245440305623","series":"1.2.826.0.1.3680043.8.498.58128851960500229959240677988505168351","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61864939963243675140060416958055914981","series":"1.2.826.0.1.3680043.8.498.58443329222919708249348834255270486013","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61934170443339188744573234190617483017","series":"1.2.826.0.1.3680043.8.498.53976057252559359796406579886874674582","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61981891095235024266625598999854298547","series":"1.2.826.0.1.3680043.8.498.12501969674232689316530859136871639026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62030472630193758412620945408364583008","series":"1.2.826.0.1.3680043.8.498.52994735922254178104975890879530303254","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61630894340791204767579628554826500984","series":"1.2.826.0.1.3680043.8.498.97813721403111171221636974778554912547","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62154969360158054310282857093665911261","series":"1.2.826.0.1.3680043.8.498.72760808286795599851789198915925987610","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62218684089117864632544493263539598403","series":"1.2.826.0.1.3680043.8.498.29943074035297606205411793935681392842","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62205799963299902160268565319924399775","series":"1.2.826.0.1.3680043.8.498.45438885713280828772539117880859940164","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62400625562782184774973349983498962743","series":"1.2.826.0.1.3680043.8.498.60740417675349165837376054103159318081","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62252176347004905787746821113523889523","series":"1.2.826.0.1.3680043.8.498.61373585378384952770238533505455631475","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395","series":"1.2.826.0.1.3680043.8.498.80088218471450943607980716401104656630","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62420632149863295140175740643640707423","series":"1.2.826.0.1.3680043.8.498.42088692264623788581840990186297141548","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62383135836496954297324866452887364097","series":"1.2.826.0.1.3680043.8.498.10337498269211232457136986979853287425","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62752755711205966800746994120425231855","series":"1.2.826.0.1.3680043.8.498.43522616608569126042739376397513507279","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62830031603985608774418534497520663269","series":"1.2.826.0.1.3680043.8.498.79544725457045386451690787340403677852","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62776347582771469289738234607517546504","series":"1.2.826.0.1.3680043.8.498.96143722249441523004920054532195972592","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62662430565901539554882496013173225758","series":"1.2.826.0.1.3680043.8.498.13661107449936337438489450319591037762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62948920639798990757395955917890550512","series":"1.2.826.0.1.3680043.8.498.10290619310256642453385343296036943217","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63002559681883387414876870926128159682","series":"1.2.826.0.1.3680043.8.498.12341001813756513122021015753947523339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62924835279557724526138406447993672103","series":"1.2.826.0.1.3680043.8.498.12516759746287279901070648519614942854","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63096952391833769395204944538276970114","series":"1.2.826.0.1.3680043.8.498.48107437754133795650969258208753249498","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63068687840516335034631353955761036465","series":"1.2.826.0.1.3680043.8.498.70384734067274388996581629815121305961","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63126604639589769235241216875934773992","series":"1.2.826.0.1.3680043.8.498.11720254543214861396482080747287375317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63195088465290711580668024587233364997","series":"1.2.826.0.1.3680043.8.498.49571409934962580420248704176346652338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63204735160524958513470115761010950181","series":"1.2.826.0.1.3680043.8.498.70390963942313053085444371858429659553","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63223220827342079068491460847713457238","series":"1.2.826.0.1.3680043.8.498.58579003983047361359207186452522693956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63360204391136938955220146755331421179","series":"1.2.826.0.1.3680043.8.498.12387475574003558924641201281893895517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63280826430574305683941548157934410641","series":"1.2.826.0.1.3680043.8.498.12204684374709449411474891529190491098","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63424633611847022112809173449690688015","series":"1.2.826.0.1.3680043.8.498.39724352224865987155260818969890829035","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63439188800533762543950349022148547855","series":"1.2.826.0.1.3680043.8.498.19794916388901752541514102805104313976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63519003265636561965405570006747767668","series":"1.2.826.0.1.3680043.8.498.11707117303091576895140578317291765989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63564058504808107209753434989270686941","series":"1.2.826.0.1.3680043.8.498.10240304017136997230649540201191109782","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63521726454879203860288713455843702564","series":"1.2.826.0.1.3680043.8.498.77769413682205982448586291066563195825","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63550258873504696667112631869753870779","series":"1.2.826.0.1.3680043.8.498.11003454443317349041966651408688306880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63242029756172180508167933519465902827","series":"1.2.826.0.1.3680043.8.498.97673778748061976777762235901257537123","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63584266247144369196126771996126173497","series":"1.2.826.0.1.3680043.8.498.50709740657818752906394548730705786927","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63683477631564232018930819747238111809","series":"1.2.826.0.1.3680043.8.498.72670097160866711668146661490063373489","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63884838982826315257233262184190435906","series":"1.2.826.0.1.3680043.8.498.47464808993720632953507743637146344320","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64022949342538714157641824612944253302","series":"1.2.826.0.1.3680043.8.498.36983293267458786187579569717074502992","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63797588614544518196205411126336672334","series":"1.2.826.0.1.3680043.8.498.23993228475285857664708431307294110227","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63986652686300452077410299429386420400","series":"1.2.826.0.1.3680043.8.498.10910872679082391056451747535599138220","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64240637244650029205175793729578897734","series":"1.2.826.0.1.3680043.8.498.24081141244360290417573951901596605956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64172939203926034064618447795131041647","series":"1.2.826.0.1.3680043.8.498.94284373427192175022504978348793882084","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64109231850522212435708176045583779359","series":"1.2.826.0.1.3680043.8.498.43320620226749204901419630826990430488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910","series":"1.2.826.0.1.3680043.8.498.40051854759647629267374762012274626762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64034149345689391514924451724232196567","series":"1.2.826.0.1.3680043.8.498.58828232864393461135515052189720229850","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64421299492271548595446260681698412384","series":"1.2.826.0.1.3680043.8.498.35532073972503426890195038388931541011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64487230511522940587342058274736571996","series":"1.2.826.0.1.3680043.8.498.10667791811983677043451028153180018691","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64527059976573114931020581328924993069","series":"1.2.826.0.1.3680043.8.498.10178052926196150995450702638798580832","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64286017646659573966584053755363026715","series":"1.2.826.0.1.3680043.8.498.13169754981090483540437046811517453047","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64381606512330581256470279413868645789","series":"1.2.826.0.1.3680043.8.498.95000231071137877695427189800962335800","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64606794953061143951972249525094490665","series":"1.2.826.0.1.3680043.8.498.99986085542110348438606173663301745221","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64652085172505930317660036583152917472","series":"1.2.826.0.1.3680043.8.498.36109601059677201324325838963111595855","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64673069370410118387363952155758553567","series":"1.2.826.0.1.3680043.8.498.72858587233297410696360822025048253330","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64897285598346101578442697515294908341","series":"1.2.826.0.1.3680043.8.498.90343010537349440631397592642611423117","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65085954283273471512936799274064949557","series":"1.2.826.0.1.3680043.8.498.12617301568371449723865613657487176036","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65013994414976103275014874284064145461","series":"1.2.826.0.1.3680043.8.498.60157805138449432723373313847701816770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64752580249608101443758975861213065048","series":"1.2.826.0.1.3680043.8.498.76823121465955116839506873082436080995","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64946212153370539647301308676689632525","series":"1.2.826.0.1.3680043.8.498.42996426682579964313562627579248394191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65187850119346448625449915281726338455","series":"1.2.826.0.1.3680043.8.498.11542889668785498578959161568332259641","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65312506382127486486519184894274326380","series":"1.2.826.0.1.3680043.8.498.11558158055648918667614264224461765556","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65124952546314712352837396207287384405","series":"1.2.826.0.1.3680043.8.498.59678900321011366653765254083672756444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65356758967883114612797250487949537460","series":"1.2.826.0.1.3680043.8.498.11997728472089884988641551811079158462","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65224808465738083245406235000047993647","series":"1.2.826.0.1.3680043.8.498.11270823136130063217834066754751106458","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65438121008780366054026567981577676554","series":"1.2.826.0.1.3680043.8.498.90349736851508037752026531968055715860","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65524487168129064460609645228098420719","series":"1.2.826.0.1.3680043.8.498.59736659138300364563932910405592989144","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65543472488214519069910341591668399186","series":"1.2.826.0.1.3680043.8.498.24902717187682773408092756765946695218","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65597375829884314152221851871550349856","series":"1.2.826.0.1.3680043.8.498.22358615316746570343697539079675270815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65667678378249093118816281743956646856","series":"1.2.826.0.1.3680043.8.498.57677348651595775508988578990248724929","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65756786850574169314934708606155033765","series":"1.2.826.0.1.3680043.8.498.18110391809995942915703978303756044448","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65783500506568702195262661666345236666","series":"1.2.826.0.1.3680043.8.498.55036766442151372168230621150068993941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327","series":"1.2.826.0.1.3680043.8.498.51563509636661462327570594061418684481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65939700606136711089565313916789162980","series":"1.2.826.0.1.3680043.8.498.12593148731639857200651022975267414247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65834774574423727070722226151408235786","series":"1.2.826.0.1.3680043.8.498.11055767268994596483900171149654699916","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65792625478083832901309715174678358900","series":"1.2.826.0.1.3680043.8.498.94242893551854623212066998398562573216","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65977711270942843864266358352913175672","series":"1.2.826.0.1.3680043.8.498.11845701396401829270308980769058514482","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66065279867975557954375204776779208827","series":"1.2.826.0.1.3680043.8.498.32888058490090603929684691773239827379","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66083855405250624607521810369199218887","series":"1.2.826.0.1.3680043.8.498.10545270718535988613361899072404971224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65896962360688978581417748865609785541","series":"1.2.826.0.1.3680043.8.498.28697002973153351537054665711678875505","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66264392187277583747691643146920431469","series":"1.2.826.0.1.3680043.8.498.62955814298634355217806239702930673825","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66199248385068685539872840826351152652","series":"1.2.826.0.1.3680043.8.498.59628367200743960349158159409928546544","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66296018797614563698471347079960740755","series":"1.2.826.0.1.3680043.8.498.13286999593213201771519362983187649200","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66308195780676793917989368423009849637","series":"1.2.826.0.1.3680043.8.498.89199423581099883939653881729242877802","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66022464665791424345622894674375661089","series":"1.2.826.0.1.3680043.8.498.11597928343357499411504869798572802224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66364462115470823900176770413527217503","series":"1.2.826.0.1.3680043.8.498.11109102802066388476251810890449005609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66510565421813721258189101326477318907","series":"1.2.826.0.1.3680043.8.498.75985414576459072681213562249698438603","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66327751482076404294387513492434577338","series":"1.2.826.0.1.3680043.8.498.64257312219074977882701684995620287859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66640124635222298792902508374800337726","series":"1.2.826.0.1.3680043.8.498.87667636113043989845428422332220686924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66694519343321893230666089565893349853","series":"1.2.826.0.1.3680043.8.498.86383461942619363712945462397217100021","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66865810098959732285731429066569782524","series":"1.2.826.0.1.3680043.8.498.29588118672458311208461757292685479011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66973089869591928349547584554030589647","series":"1.2.826.0.1.3680043.8.498.23819599092070536531545666508388775243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66805139646641294853509958688742554525","series":"1.2.826.0.1.3680043.8.498.71683072507979244740919467394610509002","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66404725797392899950170138361835649917","series":"1.2.826.0.1.3680043.8.498.39221159311378653658500978478839410362","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67001329390153622359718581388215312710","series":"1.2.826.0.1.3680043.8.498.23623335207215169089012942842621861696","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67157948144868269956661697924341234924","series":"1.2.826.0.1.3680043.8.498.73994163153160474750415445103506974696","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774","series":"1.2.826.0.1.3680043.8.498.45473142305062485127856603480622246355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67281816624401881515444587844116136597","series":"1.2.826.0.1.3680043.8.498.31168475846276210690002160150425968889","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67208608901061914528631080268819069203","series":"1.2.826.0.1.3680043.8.498.64115589406349254885377271313332221784","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67374476726034608591738910363595695123","series":"1.2.826.0.1.3680043.8.498.87039166220147038118219057767712240609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67397756161011684282778294948252031539","series":"1.2.826.0.1.3680043.8.498.10652117894528513687286244520313386625","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67490059263503953166467514121525025332","series":"1.2.826.0.1.3680043.8.498.65705855554747006863757229415795734355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67527486841399075490744039058207332725","series":"1.2.826.0.1.3680043.8.498.10772926725835587405920276049667707864","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67601368724828563564325678719126190797","series":"1.2.826.0.1.3680043.8.498.12777703394049323719266862185808775006","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67716063997837501476742746033172868748","series":"1.2.826.0.1.3680043.8.498.61576452162813973122318055257971363756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67740000270419228186274186410889719154","series":"1.2.826.0.1.3680043.8.498.11411795102312839914883636755555253826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67456402757136973484427155324664966595","series":"1.2.826.0.1.3680043.8.498.52251973666622218390842154477477131526","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67775301856843451215029610725114809378","series":"1.2.826.0.1.3680043.8.498.78355323882069722074603465638357956674","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67690090498828405347380696553164435363","series":"1.2.826.0.1.3680043.8.498.53864293796081830276975033758827650128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67875980316053973061252844698884229168","series":"1.2.826.0.1.3680043.8.498.90511870677017494015026156722126457977","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67902658411525109462089381503197501728","series":"1.2.826.0.1.3680043.8.498.34156081006180887137660133923856861767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67620512607520186685421323991166816451","series":"1.2.826.0.1.3680043.8.498.81817014253824809099659255117322699085","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67941678374744711267840655419606994796","series":"1.2.826.0.1.3680043.8.498.71243481241549497858525638697599209569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68042422993686716523227090971667037309","series":"1.2.826.0.1.3680043.8.498.31865269751891000539558442692190619399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67981157392064878495682078197161506255","series":"1.2.826.0.1.3680043.8.498.32727954268656833201637387239855314981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68115220080063728900711167735074276147","series":"1.2.826.0.1.3680043.8.498.12561323352343097281936458019933968243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68098431505945078581285587720599762771","series":"1.2.826.0.1.3680043.8.498.23634252064212322310775716267219415559","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68166103964518457268906879600985959436","series":"1.2.826.0.1.3680043.8.498.22929371502457108005488900868697611074","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67817378785531423675449308236680249899","series":"1.2.826.0.1.3680043.8.498.12940666663736385653209917275151402426","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68142054543837964766571710567214307830","series":"1.2.826.0.1.3680043.8.498.49124506608656023756731738126141069915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68263925314265871578642679583159577645","series":"1.2.826.0.1.3680043.8.498.35450028079776013116651987917484251169","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68176786698913278568384612123261229708","series":"1.2.826.0.1.3680043.8.498.80208780905112574949078788330795491964","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68304853223764870957854101604223809218","series":"1.2.826.0.1.3680043.8.498.10691364585849580554421330988389048401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68442883836038453562548037358230921748","series":"1.2.826.0.1.3680043.8.498.71307200769544253955025994514135264089","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68434323590918408241182252996136001891","series":"1.2.826.0.1.3680043.8.498.32329844717798544029526480486792857623","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68328948893840135511720724977892905019","series":"1.2.826.0.1.3680043.8.498.21092723223052181748842858665356605007","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68504220056564878621959394439671931747","series":"1.2.826.0.1.3680043.8.498.73540242461191250812646859309355703332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68570907895010847368926943092385685462","series":"1.2.826.0.1.3680043.8.498.72177283734970100713122927680390625627","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68578659816221770558506780445608768600","series":"1.2.826.0.1.3680043.8.498.42143806600151682776736264021170372455","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68744926735490285461495285547312308522","series":"1.2.826.0.1.3680043.8.498.35475698516127388191488706984816169817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68236154531625015457518732674625182872","series":"1.2.826.0.1.3680043.8.498.10185698269969452545400148059412363717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68707023438918237611039069622720041636","series":"1.2.826.0.1.3680043.8.498.89769237556569103798046862884521644675","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68806105227826942108149090508198910591","series":"1.2.826.0.1.3680043.8.498.57229178542028969917243658742484525578","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68777783869253673804970857981124324335","series":"1.2.826.0.1.3680043.8.498.76123865232882119139470063579377656630","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68919687595587004665658610968739852899","series":"1.2.826.0.1.3680043.8.498.32190567950349623837066950409634787793","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68969263295435881669720950631709959471","series":"1.2.826.0.1.3680043.8.498.44362139256399187771112010137323059864","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69077351372476086930472412978307038819","series":"1.2.826.0.1.3680043.8.498.74767954159147731439215137785186844702","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69002590469755721401448924002989552840","series":"1.2.826.0.1.3680043.8.498.74250432972767945533719602327377155888","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68637419175382604890136839259344633105","series":"1.2.826.0.1.3680043.8.498.33091597369275269387110982780938444853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69212713525853754921729012449897924087","series":"1.2.826.0.1.3680043.8.498.22649963636314885855841533749973907352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69260280362382086729793018979234418276","series":"1.2.826.0.1.3680043.8.498.10754437641104765150972899083881140720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69311065409015951283858403030798356475","series":"1.2.826.0.1.3680043.8.498.40627012780931465060014714376567314025","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69397705155385306818511499752200941968","series":"1.2.826.0.1.3680043.8.498.85528417357662351465916243568924561591","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69141410772327587821159027820329185006","series":"1.2.826.0.1.3680043.8.498.83010609572991820787942248212184777020","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69417319234704516520173921900954337376","series":"1.2.826.0.1.3680043.8.498.76556614335468144138661712369279700794","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69470130842235635628917024269527211321","series":"1.2.826.0.1.3680043.8.498.27136064574534708333735313359225056132","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69596668032633976121623939445795750384","series":"1.2.826.0.1.3680043.8.498.91686343460974739197446999362698620410","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69635429172929809539574727887787303282","series":"1.2.826.0.1.3680043.8.498.30690359493179815366862046577303596072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69514709065182531331987458626496800737","series":"1.2.826.0.1.3680043.8.498.11666903709362045940410034924642096171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69730556680449158538173708009262014812","series":"1.2.826.0.1.3680043.8.498.12172868593979401142662219675042291519","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69717507929047897681405659943474469018","series":"1.2.826.0.1.3680043.8.498.11410384038779882544308736148860057808","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69828573582708185393942443598783108800","series":"1.2.826.0.1.3680043.8.498.10426908051287807982200701421718276110","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69665129458102359604411189244405850661","series":"1.2.826.0.1.3680043.8.498.78731161609309833956232774594172501999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69954533310505596128507234525019777805","series":"1.2.826.0.1.3680043.8.498.42609308783325168912262005480559981524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69917156795941398965793962392683899162","series":"1.2.826.0.1.3680043.8.498.80702477113993294343358436578225501190","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70125113032990356507120856030706104508","series":"1.2.826.0.1.3680043.8.498.11685883134884604094773641436692425161","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70310162525433830894324495692973986063","series":"1.2.826.0.1.3680043.8.498.19658365047044220895317338395663891174","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70204818275951821519024007466988533363","series":"1.2.826.0.1.3680043.8.498.68520821237690082358275623770504060747","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70271042033022764422915852922035168712","series":"1.2.826.0.1.3680043.8.498.11821718848531664461302333922404623156","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70358024114589255428192032298530338260","series":"1.2.826.0.1.3680043.8.498.37940543021741917944472922106974684817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70435722790459313233345196619727661231","series":"1.2.826.0.1.3680043.8.498.16608119398116369513514521574289680536","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70409266114961050236894235887802126808","series":"1.2.826.0.1.3680043.8.498.56793131162914323862365815497089211432","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70471859687546886677145171619638581757","series":"1.2.826.0.1.3680043.8.498.79068165799542246793429719834708229639","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70674155597869824035214737999044110827","series":"1.2.826.0.1.3680043.8.498.59683791864458662298971075220349060262","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70507536279875475164400849670959748314","series":"1.2.826.0.1.3680043.8.498.80178940651868606143975629151968168803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70865607713855272648070757909359669526","series":"1.2.826.0.1.3680043.8.498.12025167786811786178111467757424555896","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70705483212961922828901596159131697717","series":"1.2.826.0.1.3680043.8.498.11375955924383882332929317024166731976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70877157427595288430686609728186100169","series":"1.2.826.0.1.3680043.8.498.98169263119226015763635866270400722622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70829434408808897601630953382239533220","series":"1.2.826.0.1.3680043.8.498.46522087221326180385606196870572067616","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70909154434106251910698870110315237266","series":"1.2.826.0.1.3680043.8.498.32982335365115421486618609927463742190","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70532653867267627708152426358320452518","series":"1.2.826.0.1.3680043.8.498.10086196160360308376842789661115907599","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70955936086115471632446529979621948261","series":"1.2.826.0.1.3680043.8.498.41284183441965523748098439401831984550","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315","series":"1.2.826.0.1.3680043.8.498.21047964874169368078957202496975868526","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71047916312801688293181464016099792199","series":"1.2.826.0.1.3680043.8.498.10728585248599406152561805742662711101","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70918221697435256352088727172904502068","series":"1.2.826.0.1.3680043.8.498.22565264868327494191343521591345966467","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71180975285730441100082825359032677931","series":"1.2.826.0.1.3680043.8.498.64843580064388389357901554603129313269","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71311031275189592818091380416268522846","series":"1.2.826.0.1.3680043.8.498.18855543083213269127436205972402936846","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70984418257970716669463829979819981861","series":"1.2.826.0.1.3680043.8.498.11147238314616592227093300539889961895","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71388532549899713881913113571627425347","series":"1.2.826.0.1.3680043.8.498.15592574025702726982526406135348522751","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71084400653875583977681616665093993652","series":"1.2.826.0.1.3680043.8.498.65027240114429236388834400575683429276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71671013221669628598634284971059228267","series":"1.2.826.0.1.3680043.8.498.63296971792528229050450397384140253876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71636001534145439773297498405674521629","series":"1.2.826.0.1.3680043.8.498.10564260073333319074527028308778863600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71688651674041466985305836912343513997","series":"1.2.826.0.1.3680043.8.498.12720228687087771317756972125582026592","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71507059468810869004142324822642900237","series":"1.2.826.0.1.3680043.8.498.80390739089621736300540570821604481932","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71736046309441396676679830874326797437","series":"1.2.826.0.1.3680043.8.498.10917918545499235402618243917661516019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71766161038805688022754893134126528749","series":"1.2.826.0.1.3680043.8.498.67517822333122963955596108044931189166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71749507378205349451480318735471291366","series":"1.2.826.0.1.3680043.8.498.10167003155712742297910157198632228210","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71707317731065645278250956442336515504","series":"1.2.826.0.1.3680043.8.498.46416143782455904406027101419217423469","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71898908031968158312013185809213669768","series":"1.2.826.0.1.3680043.8.498.32088876397023987488368984850544262077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71814919885777053270239861047114794237","series":"1.2.826.0.1.3680043.8.498.74996571911924254090781447340271535840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71901039762584748047172479049347609017","series":"1.2.826.0.1.3680043.8.498.70143849195063434288598176752677265444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72008869367139958325145888733651022033","series":"1.2.826.0.1.3680043.8.498.14892620533626703566705784424481723074","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72018602547183896629903902899903468287","series":"1.2.826.0.1.3680043.8.498.47938211459751283209526971914452766492","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72095214441332094906401331319197122529","series":"1.2.826.0.1.3680043.8.498.40974010997607966049465799972463032132","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72172265826902387784016098389863005666","series":"1.2.826.0.1.3680043.8.498.47838263261178653738294054895422287121","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72185504245484151161044972024469667666","series":"1.2.826.0.1.3680043.8.498.94464430362301428792262815353871283285","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72237065473467452226073540784070938342","series":"1.2.826.0.1.3680043.8.498.71484940280680118581870767636112327156","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72252601583562621620422058467969183753","series":"1.2.826.0.1.3680043.8.498.75592874523411618524960516719660941035","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72313325634455670216423864504103231505","series":"1.2.826.0.1.3680043.8.498.63740759365614787422712546066998551906","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71836684300308056297405632410209248276","series":"1.2.826.0.1.3680043.8.498.10162282523284664796736828096063362714","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72368645606329000923061796699431437924","series":"1.2.826.0.1.3680043.8.498.62980807387063805994149296100832927369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72395496925770440630088720587484704971","series":"1.2.826.0.1.3680043.8.498.11512474179977479839876478366099586152","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72494404099796908482534991590660770670","series":"1.2.826.0.1.3680043.8.498.42848460349023699564490343025141674968","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72495677804641720406765749568114476096","series":"1.2.826.0.1.3680043.8.498.47040471882580686534514566245449927699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72697172162233919540021967137047646179","series":"1.2.826.0.1.3680043.8.498.98759837861547376952136290053614119317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72576069358772864331810785663326683534","series":"1.2.826.0.1.3680043.8.498.11578804148297149694999195892827777788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72653666963736486779382314125782779004","series":"1.2.826.0.1.3680043.8.498.12992442475096075529653431889712849072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72270915206075693056344475438055463113","series":"1.2.826.0.1.3680043.8.498.72460583210015210982753088915945698687","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72771360268592017354044657882090174639","series":"1.2.826.0.1.3680043.8.498.81535593488754812895248952859908800503","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72814437733065652700292896915085936230","series":"1.2.826.0.1.3680043.8.498.57442377412749132815569252274164505064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72786773761211751423503785260617678955","series":"1.2.826.0.1.3680043.8.498.12694973477861653878363933345783597848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72880034930990967339546757487847803284","series":"1.2.826.0.1.3680043.8.498.61756497944551572164374627976025597688","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72937134140407212536042212109839895294","series":"1.2.826.0.1.3680043.8.498.22049131065523944034823574692548395318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72896586188735703676995894830815493004","series":"1.2.826.0.1.3680043.8.498.78491143578046910050987135676859825930","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73033521237848260228708170280586043408","series":"1.2.826.0.1.3680043.8.498.12566228508645398260356755233154671260","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73083786204651635096412213233334934941","series":"1.2.826.0.1.3680043.8.498.39397416221880642811163735307409677183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73070182506490199787361298167985971577","series":"1.2.826.0.1.3680043.8.498.11343531028236723298620561336997812508","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73170541731840260065335942048010387599","series":"1.2.826.0.1.3680043.8.498.79963081900954972499529584603164252329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73198335328613992289471885012751338458","series":"1.2.826.0.1.3680043.8.498.19050437970459173803333887827699687691","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73254365064404586162477089636000427751","series":"1.2.826.0.1.3680043.8.498.12451453468534525403298376943605595893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73215265387578908341609746559916587894","series":"1.2.826.0.1.3680043.8.498.35872496882990268188839732034985087446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72977302282583186913749000016284027331","series":"1.2.826.0.1.3680043.8.498.43839962888177349190898371887133964640","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73275019425494043461771574462523171618","series":"1.2.826.0.1.3680043.8.498.70019509183808914333147981933827602279","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73241519025825862373621891294281253158","series":"1.2.826.0.1.3680043.8.498.80476316067957261550828909209531854321","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73394992092536626285935384324191560182","series":"1.2.826.0.1.3680043.8.498.15177292587922323891478166634967543253","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73480422982315825699847838826691577320","series":"1.2.826.0.1.3680043.8.498.56057075236971471705834021278331574316","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73477249559479870799427355988138990349","series":"1.2.826.0.1.3680043.8.498.12716864306687729082231900706655632989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73325959289998105909842627876607163467","series":"1.2.826.0.1.3680043.8.498.17078855879534762167850651561340977489","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73539750992809874106013052513162732951","series":"1.2.826.0.1.3680043.8.498.10564227397032601955635252087972423928","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73567807253694728043552196136704821457","series":"1.2.826.0.1.3680043.8.498.41252421516459328577811663671803574560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73716752081143767997507440278231972668","series":"1.2.826.0.1.3680043.8.498.80395533797716021114166858775570728384","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73792361507751648386401969571717557389","series":"1.2.826.0.1.3680043.8.498.13232701903135303231508484751718388875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73869775636181013203432845292462040922","series":"1.2.826.0.1.3680043.8.498.20355093255619127488408277607375871287","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73616237718955147476241731466078978343","series":"1.2.826.0.1.3680043.8.498.10487607617705764916387674383403085856","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73522298807198784022883320957252530465","series":"1.2.826.0.1.3680043.8.498.26978367953043749138353898958390668040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73988798714023285209290281409329527637","series":"1.2.826.0.1.3680043.8.498.90817222390760464652453753973799098831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74078340346189210015785395961373387782","series":"1.2.826.0.1.3680043.8.498.12125971196918230236440101661319625904","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74160604730133050784449689952695940558","series":"1.2.826.0.1.3680043.8.498.73731190660949263264746606312141548539","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995","series":"1.2.826.0.1.3680043.8.498.13310307807795355391194629563231191821","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.74206939384612669977629625310277500302","series":"1.2.826.0.1.3680043.8.498.32788100201788624117375266718008261274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74127742661271965093205833471645893520","series":"1.2.826.0.1.3680043.8.498.83031198839614711225854399683062733756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74215611637606164186239511504256137871","series":"1.2.826.0.1.3680043.8.498.28691843511559094309674351168942202930","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74274855596893595390880617896069524465","series":"1.2.826.0.1.3680043.8.498.10235178261166447072383464548264334309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74243564762473331581882860440560337725","series":"1.2.826.0.1.3680043.8.498.59861856384888347378785224750971650480","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74332475620045145049628676751586693385","series":"1.2.826.0.1.3680043.8.498.31947675238495457665511911486744885636","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74423888576992387013857540016155410962","series":"1.2.826.0.1.3680043.8.498.10881018634597403115821286599850681970","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74388661917632804671669952747214084502","series":"1.2.826.0.1.3680043.8.498.10497946473576029025196430471622254922","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74688733778107958348746767768064374883","series":"1.2.826.0.1.3680043.8.498.30985189160377439852460148369056564679","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74725806565437665548056410415178280121","series":"1.2.826.0.1.3680043.8.498.62717272638122185386771193605854467247","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74622031396885197580453304563088863451","series":"1.2.826.0.1.3680043.8.498.52174831923192157931785623117334548724","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74471396469132211069109289423495141062","series":"1.2.826.0.1.3680043.8.498.79177423805534441113379435017475507527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74855168460956358728708865599401983005","series":"1.2.826.0.1.3680043.8.498.79696672815999858543815934385007091319","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74939269621045823332837094516056211606","series":"1.2.826.0.1.3680043.8.498.84451953536882733845980212997028601104","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74872419726042767543822107291861307866","series":"1.2.826.0.1.3680043.8.498.10483248699494899396446913794271889551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74952769952682175467667812958859010329","series":"1.2.826.0.1.3680043.8.498.57476531181249737132050505431957281821","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75186274888329123179129240384960518861","series":"1.2.826.0.1.3680043.8.498.73107234960996327959399955356481788257","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74222294235046321147381553078000820604","series":"1.2.826.0.1.3680043.8.498.10514910777181618647057702399296875999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75268429892526831752484062674699274309","series":"1.2.826.0.1.3680043.8.498.89908887188589585530809196305518156663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75282577107802752393765861202996650638","series":"1.2.826.0.1.3680043.8.498.19374493949415168209516540663169341568","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75384600248630012126841890313149674691","series":"1.2.826.0.1.3680043.8.498.12553942338912997929719814954568539303","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75419034794059601820999512200601915669","series":"1.2.826.0.1.3680043.8.498.25770033293028324871335077940837533642","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75126165133796914746114117844697263697","series":"1.2.826.0.1.3680043.8.498.10368974596329490707916041506609409737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75427432788337288702609576920198810880","series":"1.2.826.0.1.3680043.8.498.10028715832304659785576602700454067426","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75627574440860942023321071382766359827","series":"1.2.826.0.1.3680043.8.498.58224143122402950059678672786789650732","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75577846246542841539613557336025271434","series":"1.2.826.0.1.3680043.8.498.33397285285681571410107359215679000860","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75670996863043899245632198549551296292","series":"1.2.826.0.1.3680043.8.498.11890256300245105367200479057897138345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75739561647115166735277767560308263446","series":"1.2.826.0.1.3680043.8.498.37190975950287841292825156225241046568","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75760698509105388883811155432691667447","series":"1.2.826.0.1.3680043.8.498.37613206790496647960241733886130432151","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75547515925579172661099437338637082554","series":"1.2.826.0.1.3680043.8.498.91969324374405375664209390327261180587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75522002047190283617659083787085821064","series":"1.2.826.0.1.3680043.8.498.12014004182240137250266732872077343355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75800418125597325281807386716279941068","series":"1.2.826.0.1.3680043.8.498.58072050334557303365317000149596152536","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75976388060969166980873144500539735332","series":"1.2.826.0.1.3680043.8.498.48756803122147446236748447542810975719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75924957745868212343758911856809118662","series":"1.2.826.0.1.3680043.8.498.61301359659134156228060182148829633039","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76065434429437758192282562351089010088","series":"1.2.826.0.1.3680043.8.498.39124831840138085556877842282580348730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75746520601616488682662195284941583868","series":"1.2.826.0.1.3680043.8.498.11875357488150611640589994641026847480","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76107330932071966465869847071480517541","series":"1.2.826.0.1.3680043.8.498.46877833053431753523369149387922571283","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76190704217737835083438638928129775625","series":"1.2.826.0.1.3680043.8.498.92186576388252464642730710005291675218","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76242245340157474968574926079920787199","series":"1.2.826.0.1.3680043.8.498.34140873614326231265540908649901212717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76125027090596913549224250979076533606","series":"1.2.826.0.1.3680043.8.498.34137810978806684750086250488378772523","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76012896432318288623173593280895772876","series":"1.2.826.0.1.3680043.8.498.56662426216366192756750020544040690715","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76344826994380951498651448092980303697","series":"1.2.826.0.1.3680043.8.498.41366565336496253841847902256289848873","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76514087677277144682713362609812138150","series":"1.2.826.0.1.3680043.8.498.91261676975645843839589532329872252512","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76525107909962775502207829870873494024","series":"1.2.826.0.1.3680043.8.498.79857394204870519968316101574025525079","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76662146598924492573212477881527658730","series":"1.2.826.0.1.3680043.8.498.70748244153625227727448008666820535585","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76742750470853901575934097885382242952","series":"1.2.826.0.1.3680043.8.498.64206509253670872568366460891964871107","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76772148369531098768751348676329945789","series":"1.2.826.0.1.3680043.8.498.58429749608254551285943695407972767345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76543241221133987658552440969465372120","series":"1.2.826.0.1.3680043.8.498.11205802914533485705694502802275967349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76909970028125318280995002129970165723","series":"1.2.826.0.1.3680043.8.498.12873607961255672617573093985275367042","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76950154099427419631807259601983648254","series":"1.2.826.0.1.3680043.8.498.94763594962751055616661757677768562278","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76862911430418639277808760660745763387","series":"1.2.826.0.1.3680043.8.498.12034997945035092597086684404071936315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77018355356797290743788712046815540223","series":"1.2.826.0.1.3680043.8.498.10217951647180921103082467208704037849","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77061877382110597010150770981465581536","series":"1.2.826.0.1.3680043.8.498.84809111339989816745464547197244919024","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76620177268652814425535081607587091903","series":"1.2.826.0.1.3680043.8.498.14642773838188812082441412831203216478","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77155817997584903787597001328868124855","series":"1.2.826.0.1.3680043.8.498.41967779045880377858903429545241222744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77183368292104509335580564463404842046","series":"1.2.826.0.1.3680043.8.498.99436810570834680261801979132941597558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77244685031928454051409193776790348219","series":"1.2.826.0.1.3680043.8.498.38055324680995007577016982409668386428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77274115607096351160296319533007121044","series":"1.2.826.0.1.3680043.8.498.84425048733978792865277225689037278443","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77058587552329832558354544874120592826","series":"1.2.826.0.1.3680043.8.498.48724426424617453265001083265492992809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77319661699409997424800377883567027839","series":"1.2.826.0.1.3680043.8.498.11857305198558868450449700077444812865","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77370119801633353780677378448761665685","series":"1.2.826.0.1.3680043.8.498.12387137925947687829668763899872254897","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77391801789875882109036004511558148658","series":"1.2.826.0.1.3680043.8.498.89115047894747930772708315835100271372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77442647074650043391818472068835112806","series":"1.2.826.0.1.3680043.8.498.11771465904671161876702110989737784474","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77283634197289382453511269950488893300","series":"1.2.826.0.1.3680043.8.498.93529039172626705624673530388827819455","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77508762938574470575737016745357126587","series":"1.2.826.0.1.3680043.8.498.12933816065411170793131384601807893204","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77299727102400166279263320763707637485","series":"1.2.826.0.1.3680043.8.498.11535866189105373187953251583112414231","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77421709644087006134426085284321515809","series":"1.2.826.0.1.3680043.8.498.17371650001680434225521518992657718103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77524507552350448792970399203131998592","series":"1.2.826.0.1.3680043.8.498.10870027068023756688896462425861470588","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77546769004367585830455474984769277713","series":"1.2.826.0.1.3680043.8.498.15576758637798824773691030673453186535","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77611073647079414772859496272387571059","series":"1.2.826.0.1.3680043.8.498.21499065257996154631340582415902196665","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77584925163022886785284486387921312146","series":"1.2.826.0.1.3680043.8.498.83805377628572834188220324518407415752","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77735690782343511872544703464260892488","series":"1.2.826.0.1.3680043.8.498.32944815998225255003854545848179981008","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77628479664122209492622263818114972176","series":"1.2.826.0.1.3680043.8.498.90477640557049233745497205499082901854","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77800740811302686332282410929816056283","series":"1.2.826.0.1.3680043.8.498.66400113188286749771125710601208241672","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77753753872601674668756777373761660873","series":"1.2.826.0.1.3680043.8.498.11256005678466724080630564734780676931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77823946901144772346251781172043480398","series":"1.2.826.0.1.3680043.8.498.10813122722764339267399893719529029985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77911973519830915742655727368286737079","series":"1.2.826.0.1.3680043.8.498.10381516266852869060729203694142371144","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78003548255041391467292087330671945247","series":"1.2.826.0.1.3680043.8.498.29051352178410620123621364021789672366","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78072765790191594685273447833272737429","series":"1.2.826.0.1.3680043.8.498.31382515594248335495451580611829725569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78083057157844664959843364584162587026","series":"1.2.826.0.1.3680043.8.498.12036841367029866713301380221939682736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78119223228067919465880137763593966456","series":"1.2.826.0.1.3680043.8.498.80694240921235522334851360396284760777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77889800516853576877948593628420228178","series":"1.2.826.0.1.3680043.8.498.98883309803403426274220973789638571924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78198993378521219839327914679211902952","series":"1.2.826.0.1.3680043.8.498.13002441693694896192886151270419050050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78286395605671977872377352305890466464","series":"1.2.826.0.1.3680043.8.498.12610999205387048179705761496902257671","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78238394512811614973689011297363399053","series":"1.2.826.0.1.3680043.8.498.46866434814627734312041468298138007877","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78475145549489837635803968565380247517","series":"1.2.826.0.1.3680043.8.498.91848469928166529380049430919830755902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828","series":"1.2.826.0.1.3680043.8.498.51594414677993339088394514174201163950","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78420763928408680447797586163977368190","series":"1.2.826.0.1.3680043.8.498.34155853566233321122789194875024803059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78657055734742806687703707756153505756","series":"1.2.826.0.1.3680043.8.498.99155640537243022567834334047768018126","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78674598286586525734293563394409166190","series":"1.2.826.0.1.3680043.8.498.97231708962369715276414468850402358303","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78757011872917563830995996191850541453","series":"1.2.826.0.1.3680043.8.498.41638077056982146688098193784487146357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78574437345603052437892513165832297960","series":"1.2.826.0.1.3680043.8.498.12984100461844299299126160979121635428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78968420394708302953391711477016139949","series":"1.2.826.0.1.3680043.8.498.63032214662108564869992168801402428741","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79021499290691125513421477204330532755","series":"1.2.826.0.1.3680043.8.498.40587048563462161528460859595699287917","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78812661021494224637586904573732040856","series":"1.2.826.0.1.3680043.8.498.89240427498659264185023617810117177572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78800983923143929298359366572020370031","series":"1.2.826.0.1.3680043.8.498.50506775968118078475103908405616311906","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79087227780630716132147615846840128474","series":"1.2.826.0.1.3680043.8.498.70308429783838754326143554815706486727","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79122189963582241618519333445055548347","series":"1.2.826.0.1.3680043.8.498.10747597510837331028563090319082969570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79077219037094025366585138112899760966","series":"1.2.826.0.1.3680043.8.498.11255993703470581697219105731116371159","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79224785668205420138951103045759609475","series":"1.2.826.0.1.3680043.8.498.10453960821455576170662981061633486281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79258497507223824227830886693749460348","series":"1.2.826.0.1.3680043.8.498.31513439208721624768338510064569761243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79245771997885598399203314109623928718","series":"1.2.826.0.1.3680043.8.498.12106531351508401939655788910742401973","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79292862882980756453890778996691993351","series":"1.2.826.0.1.3680043.8.498.53932479537683080674857728300167263629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79341955397030543255969676347554165142","series":"1.2.826.0.1.3680043.8.498.12806100301143525879172973084948648255","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79560170727391465844958390474997798803","series":"1.2.826.0.1.3680043.8.498.22156610816936127482082482017199624803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79582457162604674356422857305405255763","series":"1.2.826.0.1.3680043.8.498.47306027039454630242223140771880260351","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79524312100988116256514335064057385347","series":"1.2.826.0.1.3680043.8.498.13432222038487980413959590848715446452","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79274948549030940175536603063980168192","series":"1.2.826.0.1.3680043.8.498.30526556458197364057368353610203229222","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79741629769872115526056483024289600492","series":"1.2.826.0.1.3680043.8.498.11268175773650872982106552618964857562","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79672421957132993896448375311641998688","series":"1.2.826.0.1.3680043.8.498.15454163485933290186202657924300566922","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79829033430619597320769579608450393290","series":"1.2.826.0.1.3680043.8.498.11976741738234981705222413406140234569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79991838737793096019496013540529642900","series":"1.2.826.0.1.3680043.8.498.13305142937147722533218851966185826669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80100785345864355815698147663964005920","series":"1.2.826.0.1.3680043.8.498.67353218273364926842653450342673585275","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80161127293548940216292842707683784128","series":"1.2.826.0.1.3680043.8.498.64096639544016150809824005424725822501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80247974926646352443043954025450139721","series":"1.2.826.0.1.3680043.8.498.83026362127414036756153611857852660177","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80284846390295834578273590480193235798","series":"1.2.826.0.1.3680043.8.498.12406947255418396228215356174649417643","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79636443202100988025435536780567446582","series":"1.2.826.0.1.3680043.8.498.92596996672773279242556192419627281044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80292580287849976694993628109721422478","series":"1.2.826.0.1.3680043.8.498.30730551313623314158110057665146594650","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80313580693550912024103221258669390487","series":"1.2.826.0.1.3680043.8.498.13204774805688968926221487550895640935","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80192355832283305029770183988436644845","series":"1.2.826.0.1.3680043.8.498.56231733920314236776874346116352538212","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80433141990021472112645841723991584182","series":"1.2.826.0.1.3680043.8.498.61740846590793311333271423749763403162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80492324225032367918842505447056622561","series":"1.2.826.0.1.3680043.8.498.41981435082793748982017432010696610394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80589796918823112368758999678369292754","series":"1.2.826.0.1.3680043.8.498.71771781562165030728502111848997741421","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80556515221794602430701573588498144286","series":"1.2.826.0.1.3680043.8.498.75455580669087394534986825955949580812","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80627605463642274852766102874822833365","series":"1.2.826.0.1.3680043.8.498.10803465797340384763261554022207315158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80642348849062865125509773837278900750","series":"1.2.826.0.1.3680043.8.498.50294070369230764267516502770601902355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80671597537348112977780954910198682272","series":"1.2.826.0.1.3680043.8.498.14940123451397990101460119404617793770","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80726797831040227285328789648680519664","series":"1.2.826.0.1.3680043.8.498.44324972524014279490754731266472011904","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80325205657974094483016625702390834774","series":"1.2.826.0.1.3680043.8.498.13440113385742052438825080774223005042","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80651977424179089141346174053751183506","series":"1.2.826.0.1.3680043.8.498.29057132806344319307792324125947222187","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80876235930771882271600826613593202364","series":"1.2.826.0.1.3680043.8.498.33798630489622841077203441105432437352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80886525412417948114478368374120704834","series":"1.2.826.0.1.3680043.8.498.12777310391604420793409362296300277900","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80741358365199721720335324482078866565","series":"1.2.826.0.1.3680043.8.498.80887075976294105173413099311427340875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81024897463656728358703040759074583549","series":"1.2.826.0.1.3680043.8.498.10172698676313450564843831567700354357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81096597169632636454986988944050298701","series":"1.2.826.0.1.3680043.8.498.46655028409163498946698317825352557601","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81012548926053584036386949658594985125","series":"1.2.826.0.1.3680043.8.498.31984394771429029900535822163102973711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81109405367395078564887780335570099673","series":"1.2.826.0.1.3680043.8.498.61919096350110482046866623422951548669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81248820934296461385324385812581970340","series":"1.2.826.0.1.3680043.8.498.34864942614907937080611669471033274473","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80851365368114686105558347899990395309","series":"1.2.826.0.1.3680043.8.498.68350745597980085469636118400254059198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81407504276375197122586620633759174421","series":"1.2.826.0.1.3680043.8.498.18223757901898769428630212394867134831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81327951282145199298572980264557740321","series":"1.2.826.0.1.3680043.8.498.11784120971296859878745621717472322699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81464322239381998685473520085495957135","series":"1.2.826.0.1.3680043.8.498.55596251753981674405469877182707014407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81540134120507822899514611863725372563","series":"1.2.826.0.1.3680043.8.498.82844793591978671980982988716722535811","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81475058647553697318837239968139077272","series":"1.2.826.0.1.3680043.8.498.35141551134973678980109740417044236693","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81631925197682464058124413934958934086","series":"1.2.826.0.1.3680043.8.498.27836604411955606648504559764951218908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81590494282169434250974061642267472166","series":"1.2.826.0.1.3680043.8.498.56865376584453129369250882021340154021","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81651455914141365028015556083080638757","series":"1.2.826.0.1.3680043.8.498.26154891202521382052388114276610699798","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81740348974208459068937862799489495782","series":"1.2.826.0.1.3680043.8.498.64987341565764184499333932580230291930","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81698411330214788362326470979160431674","series":"1.2.826.0.1.3680043.8.498.33054077816776989943174336308426289656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81347841116019279082226143308588516787","series":"1.2.826.0.1.3680043.8.498.12506317361825291063377427175139944064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81797437102217585216278133168092577952","series":"1.2.826.0.1.3680043.8.498.60986070724986878747292491220412669859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81822709772232914298414700148454511661","series":"1.2.826.0.1.3680043.8.498.26038324734295957107691023969158263998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81957447654112577010261210572986132211","series":"1.2.826.0.1.3680043.8.498.13846155260108346840943074548917483128","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81892104897856100357169774214718573167","series":"1.2.826.0.1.3680043.8.498.79736417116121052394545923557223536196","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81793454171031578492222004629265937656","series":"1.2.826.0.1.3680043.8.498.13003604122200589680553178314198619441","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81974732979506931694079719601760393277","series":"1.2.826.0.1.3680043.8.498.13683882263483402923431768639824060927","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82143276307898132314496735515204481508","series":"1.2.826.0.1.3680043.8.498.12245033751430663621352193615911379295","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82053853036886620816821481139413594407","series":"1.2.826.0.1.3680043.8.498.13009462421912372861600089610124267368","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82355456395534835058857499033494900716","series":"1.2.826.0.1.3680043.8.498.36526472783457824113553564979287229867","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82157293611630605740641187215639204974","series":"1.2.826.0.1.3680043.8.498.47472875284866725240563787840541871791","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81984200870681776955493738987078746894","series":"1.2.826.0.1.3680043.8.498.98337017554593669490287906558880597400","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82169173362972793731001247413129649263","series":"1.2.826.0.1.3680043.8.498.83033617594458607930224960165436346736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82481056562260046738834822905709207128","series":"1.2.826.0.1.3680043.8.498.72178874561594188578353682293925773817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82523073559792523830827952754346276051","series":"1.2.826.0.1.3680043.8.498.13360151738991191576777566606514314077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82582558624896662901163827145277490232","series":"1.2.826.0.1.3680043.8.498.88701579963800154657591307763685773902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82591599611540849522609405219902904134","series":"1.2.826.0.1.3680043.8.498.12725312486952325042086842699865776140","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82548952153746644951736669119607083230","series":"1.2.826.0.1.3680043.8.498.56914546378247429308371634559916131839","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82757616216788919181323176147396770758","series":"1.2.826.0.1.3680043.8.498.12114242047632352161110349831180358647","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82440268452598207215458064865523988222","series":"1.2.826.0.1.3680043.8.498.31841346645254465864900276143259536766","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82967086493280898295669930903240677729","series":"1.2.826.0.1.3680043.8.498.40768004389812241063040009905672738092","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82731606763525898032678680051515732914","series":"1.2.826.0.1.3680043.8.498.87061561210064226483526437767965219045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83054899303871276184985881638619543362","series":"1.2.826.0.1.3680043.8.498.90223429336380060566037339615262794359","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82902769135373284012762237744774967400","series":"1.2.826.0.1.3680043.8.498.24755288293387446799442918852366781811","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83061811791652434833797599835671351097","series":"1.2.826.0.1.3680043.8.498.21923234497670458220478909305392464967","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83028939839767027342482365275441078777","series":"1.2.826.0.1.3680043.8.498.79356857167205549309691631478131407949","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83220896521302064187926748035734200732","series":"1.2.826.0.1.3680043.8.498.28075865845810342330875541451813300585","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83121378128521985713113311344685112825","series":"1.2.826.0.1.3680043.8.498.63481814017671062977679875258610111672","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83237706274929081175032033288272528919","series":"1.2.826.0.1.3680043.8.498.72009143853219762873024941491737381268","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83271280040777227673434567660749430744","series":"1.2.826.0.1.3680043.8.498.93685361276163851700888519241647140609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83465081013175989426506371406463660936","series":"1.2.826.0.1.3680043.8.498.14189489827170224455552289119462436334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83384018761005700404957421316476952190","series":"1.2.826.0.1.3680043.8.498.14388800218805720232713302598894917918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83509478059288902959149676250911654986","series":"1.2.826.0.1.3680043.8.498.59651919514803119235562910774512519035","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83602205818477323486169938182117122488","series":"1.2.826.0.1.3680043.8.498.12258709458724372070339877561870438167","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84059077072076289248966490696872414783","series":"1.2.826.0.1.3680043.8.498.17167873424015958502570965392350793105","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84007436672323587547274248257117331532","series":"1.2.826.0.1.3680043.8.498.10588546498448668497349091846024917497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83847285247209768917850605454140194088","series":"1.2.826.0.1.3680043.8.498.12942976515779894448618196251090530408","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84123495463663584148839831457665741488","series":"1.2.826.0.1.3680043.8.498.17417139243766922187077317601963670620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83740112248863750162633901687556705057","series":"1.2.826.0.1.3680043.8.498.54798376956940265712217125126507108306","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84062980592611347665739105270731842220","series":"1.2.826.0.1.3680043.8.498.13272753569811958454325270657484696177","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84238814030979264404768702945290704724","series":"1.2.826.0.1.3680043.8.498.38838269267205260452475158216100742891","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84347051904829814300575579670995018991","series":"1.2.826.0.1.3680043.8.498.65367090315555047706272263715481004183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84486787871353595019134028899317143475","series":"1.2.826.0.1.3680043.8.498.13461120940106505397309001424037179551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84661341697154104963062177064495044195","series":"1.2.826.0.1.3680043.8.498.10384735195657086082856589318374515404","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84827027113422423894439539536275453132","series":"1.2.826.0.1.3680043.8.498.22510106959775324133556985118007172626","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84720476187201404764508451536959273676","series":"1.2.826.0.1.3680043.8.498.12499792713723071724283800118901038670","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84630493975910180537831353884518764364","series":"1.2.826.0.1.3680043.8.498.10679748410982165210695509901216372079","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84477198486144701928775090913243250655","series":"1.2.826.0.1.3680043.8.498.84010797302991033339124737752307977285","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84982310626395569925821771375535259394","series":"1.2.826.0.1.3680043.8.498.10633639238632449885042620581338265097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85031127696558971105465423134104223548","series":"1.2.826.0.1.3680043.8.498.11034416011659829634697160705454654194","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85095389649021071892478435298763928925","series":"1.2.826.0.1.3680043.8.498.53711151650956827684990404765233378905","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85137704768728261642637391657270006448","series":"1.2.826.0.1.3680043.8.498.12241796296299270377961475643224951514","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85212353344613461454490252928274988058","series":"1.2.826.0.1.3680043.8.498.82863786625674075025942039024138875585","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85391052076760686301971262244561025345","series":"1.2.826.0.1.3680043.8.498.11691918524010640030201239805226601908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85341019006931436802294963713199308935","series":"1.2.826.0.1.3680043.8.498.90323184314042386603002697590647890733","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84857368518703570476541203284655412396","series":"1.2.826.0.1.3680043.8.498.52515450380720666931306091516022078055","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85454884397223412543050876117416146750","series":"1.2.826.0.1.3680043.8.498.23284035277276366928446377205346723215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85416107585703628549041626609272707431","series":"1.2.826.0.1.3680043.8.498.70622940029345323879676343141861519504","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85326542895076582985200613540648336047","series":"1.2.826.0.1.3680043.8.498.10117149485828956978712834545182567722","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85545102350084131396266710667808696561","series":"1.2.826.0.1.3680043.8.498.11811577071107894011284784025545897855","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85602291032341007550376696695283794665","series":"1.2.826.0.1.3680043.8.498.16211971418722339730330875117514034196","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85646781189339988389190891868001374533","series":"1.2.826.0.1.3680043.8.498.11570035849725280318667755964077048776","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85475892602877115812059264719695919613","series":"1.2.826.0.1.3680043.8.498.18455026283868130134431137564260104125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85715956158218184426400151797364834203","series":"1.2.826.0.1.3680043.8.498.10750588656187024737581075556139815730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85748882309364598407865143890978767674","series":"1.2.826.0.1.3680043.8.498.99209622434210007665938056867612098334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85849908712977285006159496192673293501","series":"1.2.826.0.1.3680043.8.498.41198028731243247341332859373802016711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85663162480758065951465617999516770647","series":"1.2.826.0.1.3680043.8.498.18854112789212226545254189514038359680","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85932495800565018166769303205378781707","series":"1.2.826.0.1.3680043.8.498.53552553136672438956195281836524906995","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85728170022176096748897349447671687164","series":"1.2.826.0.1.3680043.8.498.52531624137755952414937933304804251414","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85879572979834912642178551098379145607","series":"1.2.826.0.1.3680043.8.498.71042960014401175895372968143634215648","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85970222049966384156120176704985355996","series":"1.2.826.0.1.3680043.8.498.12870940226966535909071161426819652481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85992037603186775437749979929724514304","series":"1.2.826.0.1.3680043.8.498.10356179508725065161913200358763069080","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86020966816754359840982061252012700091","series":"1.2.826.0.1.3680043.8.498.65498082934391112963382551120066712456","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85883562680953255833567595509408888410","series":"1.2.826.0.1.3680043.8.498.79157154267215787714719578714290430833","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86042942187650835661744645315369706668","series":"1.2.826.0.1.3680043.8.498.85339241854355273939031164265419688659","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86142930956005716169649168306759482915","series":"1.2.826.0.1.3680043.8.498.83794030371944605025920254095076202966","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86263264208309369957750915520115390972","series":"1.2.826.0.1.3680043.8.498.62017634797215100238942388727374499946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86239793008693746155478763153154040128","series":"1.2.826.0.1.3680043.8.498.89676638518543935801069871010178592571","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86280033511814602594654628677479528508","series":"1.2.826.0.1.3680043.8.498.92831160101233309784561836469873233259","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86082522186055358984181640226529351801","series":"1.2.826.0.1.3680043.8.498.51126420848068003420787427895146087374","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86341139705884690862568425859303870601","series":"1.2.826.0.1.3680043.8.498.87978571727854479380488292593977506768","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86315835011470465506843590014408831373","series":"1.2.826.0.1.3680043.8.498.52212099609381404802343675396547535574","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86405969333062807387894608315897106350","series":"1.2.826.0.1.3680043.8.498.17527290973280363045583523023345587173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86416984885007871128151024515780322767","series":"1.2.826.0.1.3680043.8.498.12501363222599823505936923664284435705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86435271359723045834079326733478538148","series":"1.2.826.0.1.3680043.8.498.12684827032404236097081537840562991711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86516555757334047240474191101770683543","series":"1.2.826.0.1.3680043.8.498.25774891197524614289953868553917798613","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86625876521226465945114246972542881955","series":"1.2.826.0.1.3680043.8.498.12789962617254909158403478522760123792","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86876540394528304678398111135600894869","series":"1.2.826.0.1.3680043.8.498.51014123950531366643053650953473460710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037","series":"1.2.826.0.1.3680043.8.498.73414308736751658627435120649630352251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86715997235054599024919893656486735623","series":"1.2.826.0.1.3680043.8.498.96065538613520261350531162200293494075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86566584776908143877603164704838586097","series":"1.2.826.0.1.3680043.8.498.86333985595868679389564964188968199074","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87101276883427329522365304528449003589","series":"1.2.826.0.1.3680043.8.498.94814108642837836489523030666201682817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87046380372347615219261898797043742937","series":"1.2.826.0.1.3680043.8.498.21789120799125842983040971155713494549","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87208714421237491643915928765740747633","series":"1.2.826.0.1.3680043.8.498.13284441792275819356078225966390755392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87185817772180109135714642184874643715","series":"1.2.826.0.1.3680043.8.498.28440420063788826557436379434608534669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87222653734797522858506220026427900177","series":"1.2.826.0.1.3680043.8.498.84458604955858330872401067031901226569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87241650441242408799533998101943024283","series":"1.2.826.0.1.3680043.8.498.50550535683621716865662203550772608204","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87009029107490862793898457518597022930","series":"1.2.826.0.1.3680043.8.498.11586762459550408282642312338187559471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87413924318793242918916384453755054267","series":"1.2.826.0.1.3680043.8.498.21213931178430617126624737498227066645","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87465202523430400780596885230492167155","series":"1.2.826.0.1.3680043.8.498.10312812987713129926690854187060355401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87262205711949358943515326577886508585","series":"1.2.826.0.1.3680043.8.498.88011176326902278878709234194790236929","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87338030934656808032044071206920470285","series":"1.2.826.0.1.3680043.8.498.23085767839502787034854072653400681106","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87516159856486535118712846576943170757","series":"1.2.826.0.1.3680043.8.498.56728681281470037950500691913928954372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87560916423885591029120178291827033868","series":"1.2.826.0.1.3680043.8.498.11943900250842666704550057813255762090","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87645071186049036230936726742129446188","series":"1.2.826.0.1.3680043.8.498.46954409707922528566053615608493213629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87742737485142445536023876862256284839","series":"1.2.826.0.1.3680043.8.498.52338604701379144885107827805369131742","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87769640167791104899516835133527936885","series":"1.2.826.0.1.3680043.8.498.81743164005311638273377586096457122103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87679606849449562627719676100773778015","series":"1.2.826.0.1.3680043.8.498.12123357274662206610238365099419747853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87799106106199445436621174506197684035","series":"1.2.826.0.1.3680043.8.498.92800150362212319400591917341664884021","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87998832540756872371878613339628615880","series":"1.2.826.0.1.3680043.8.498.33447832840203674996795211373294285212","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87876800169766592603600598167358214111","series":"1.2.826.0.1.3680043.8.498.11818628183363456674948937482326731288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88064086448060280325724227239585797761","series":"1.2.826.0.1.3680043.8.498.97996301605593518037989249810257134053","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88074618626471004030644935418777151905","series":"1.2.826.0.1.3680043.8.498.90748258187827318940466571803245668808","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88114918358665488269761388157480523350","series":"1.2.826.0.1.3680043.8.498.74214991672099715583115840310740556144","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88368930449171804348215887291291274615","series":"1.2.826.0.1.3680043.8.498.97614516343707857416927166626523895037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88457076333783710706782784856816112502","series":"1.2.826.0.1.3680043.8.498.82243369746653169584712340632577255468","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88523737187772411911621088333701306485","series":"1.2.826.0.1.3680043.8.498.97331787897712277532523664883674920655","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88480827968231439729532202780922145132","series":"1.2.826.0.1.3680043.8.498.99485201781626960465166910972383150690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88211355128040927831407007101167868685","series":"1.2.826.0.1.3680043.8.498.30427265655095811817783086382011334830","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88620225126852321191690360547272730103","series":"1.2.826.0.1.3680043.8.498.10670850676655143343840942446742195479","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88570871026022705696415758052703394453","series":"1.2.826.0.1.3680043.8.498.36432071232842492156394547475394482217","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88655485247489827200671636011965281184","series":"1.2.826.0.1.3680043.8.498.25614776094994526107957643987068693469","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88682169866098509464628948503453852928","series":"1.2.826.0.1.3680043.8.498.31097559556870627364456223996564484660","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88711356400820562340756013729440417015","series":"1.2.826.0.1.3680043.8.498.30401069781573447338735050481517299208","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88845193477394064535233691626377077328","series":"1.2.826.0.1.3680043.8.498.57003972167205548728029464218626643540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88801151487634957068752641200383474212","series":"1.2.826.0.1.3680043.8.498.54786454784329308975353713632612087837","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88752039058074948027322975455607258941","series":"1.2.826.0.1.3680043.8.498.25144031130465716271421752500481849064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88574487918084130981335189023288704909","series":"1.2.826.0.1.3680043.8.498.60274406271931223709165295499070253775","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88899519473612920868099376939780201067","series":"1.2.826.0.1.3680043.8.498.11138742229861951560726191525492710819","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88874647535130028956725641820977918615","series":"1.2.826.0.1.3680043.8.498.87858698680797690133049270629817908173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88964371740619947193957393682095448256","series":"1.2.826.0.1.3680043.8.498.15714740765615489035764771553664236990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89045629523485194086986785222960864883","series":"1.2.826.0.1.3680043.8.498.93278670497240598555358645844642480811","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89074359486713340585710357399403393294","series":"1.2.826.0.1.3680043.8.498.12251357981901236972978462462010487796","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89204062904097086723738217311755833877","series":"1.2.826.0.1.3680043.8.498.46029976613990903959725260524085010681","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89247482662611898098489676481780144186","series":"1.2.826.0.1.3680043.8.498.38238576165806717141425022462844311975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89332108074590559691047812196991368746","series":"1.2.826.0.1.3680043.8.498.82186661214852096374188137414829429353","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89350648596674998203116057439419132072","series":"1.2.826.0.1.3680043.8.498.91766341146474400735482430464857374955","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89416235692252764852783739140428958560","series":"1.2.826.0.1.3680043.8.498.14687984466393339406119483652600676388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89445342796957655182687412965056270903","series":"1.2.826.0.1.3680043.8.498.52825105371101731934652615087487003927","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89569762088162031469299314053759758927","series":"1.2.826.0.1.3680043.8.498.13351211023082680684452529380740566399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89584627944947813918297925813120247226","series":"1.2.826.0.1.3680043.8.498.42784882035384744939614619697591500902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89123035099342878362576960682354791116","series":"1.2.826.0.1.3680043.8.498.10753929022446935678976745291111177503","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89307235192406983202868051697765144920","series":"1.2.826.0.1.3680043.8.498.11687745101811297942848788408891991668","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89720428711655200442262503508409163392","series":"1.2.826.0.1.3680043.8.498.76649860299291434218840645088847633554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89935384074170571164295050203650829722","series":"1.2.826.0.1.3680043.8.498.12691221776421169935886600136487252813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89953731420879076053772747816096629231","series":"1.2.826.0.1.3680043.8.498.11888452102647518118909138118854946706","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89746004948304570186940283501211828475","series":"1.2.826.0.1.3680043.8.498.40211637723016291350787021567373652094","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90037507233074511524602897439257102032","series":"1.2.826.0.1.3680043.8.498.40067434011026199339239205594745973862","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90103210546658057701715546323552886913","series":"1.2.826.0.1.3680043.8.498.12799211814512492453352821040357224974","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90170179734833731758923964833173536172","series":"1.2.826.0.1.3680043.8.498.70311608318332669864100680569233079597","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89987866402625167201267936974120824712","series":"1.2.826.0.1.3680043.8.498.99834282341953182522407089600251511020","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89842390998700046088751402253903762827","series":"1.2.826.0.1.3680043.8.498.28414348392018103388950625444790903857","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90268709354762000249148795667759483876","series":"1.2.826.0.1.3680043.8.498.49837816785490281283378595689461665488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90223492515368389331845681759355261030","series":"1.2.826.0.1.3680043.8.498.25979118345076186530075703456947785070","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90251533154603802045666102095577144628","series":"1.2.826.0.1.3680043.8.498.12350649463900546412746618008596652073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767","series":"1.2.826.0.1.3680043.8.498.40976673889922888495356759133819064735","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90331082101665002495739626395247831768","series":"1.2.826.0.1.3680043.8.498.10287904273310059382196983779019849923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90342120331229511020327052213821820086","series":"1.2.826.0.1.3680043.8.498.35365498214717111637261417961359460480","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90446842983860370183814559926724769347","series":"1.2.826.0.1.3680043.8.498.12419100868443003679285720520592208288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90299615112155255774026824400461236845","series":"1.2.826.0.1.3680043.8.498.97041952024098868465970151486043842613","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90483774039833567637730989859822082272","series":"1.2.826.0.1.3680043.8.498.28025583952216818499436381048030807046","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90193654806390401129887447441469319421","series":"1.2.826.0.1.3680043.8.498.36684691011830782009263861672691117753","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90508461713737190527021531642732695353","series":"1.2.826.0.1.3680043.8.498.48617377907026015563814719607602447455","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90551187439998209641370107095424494085","series":"1.2.826.0.1.3680043.8.498.13092633807955913618703377046465649122","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90629760164319310375909187696899268080","series":"1.2.826.0.1.3680043.8.498.89271542740740140178465940260136110908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90758414832541790868268922794274257839","series":"1.2.826.0.1.3680043.8.498.83054072780499203673047146985247596565","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90771070773751767226565553195436971014","series":"1.2.826.0.1.3680043.8.498.12862469747177209185590022989175937982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90893882071188409362296099956462540506","series":"1.2.826.0.1.3680043.8.498.61658218918329993013102909516813325451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90985037792656682958281230441654772668","series":"1.2.826.0.1.3680043.8.498.44206448801945329553245596126829355968","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90679199017361780574384378857945322240","series":"1.2.826.0.1.3680043.8.498.66046249012207619685469994081232368704","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91124979914013108948829704350601987380","series":"1.2.826.0.1.3680043.8.498.13393014261082142218569655651758251815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91106275065657355304239097295699332135","series":"1.2.826.0.1.3680043.8.498.45243338998934927183483652207271442288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91139126799820974665417903552066103454","series":"1.2.826.0.1.3680043.8.498.12879771537175282761006813996424871776","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91262503278090288446189575519187653125","series":"1.2.826.0.1.3680043.8.498.46561780341968307745572741679387467385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91206973544724011503639050238118945528","series":"1.2.826.0.1.3680043.8.498.98349989687034340694145529838922342775","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91230059913252604666971822137553690811","series":"1.2.826.0.1.3680043.8.498.51898352210738833278421662278178219804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91385694666101904780012335954322545042","series":"1.2.826.0.1.3680043.8.498.69810513871734635256892439443116413838","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91328985601265188961626433080766431089","series":"1.2.826.0.1.3680043.8.498.60520935787421443064340229288714174191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365","series":"1.2.826.0.1.3680043.8.498.11435161521929192543149412670580143466","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91497176426480836856097456438014037999","series":"1.2.826.0.1.3680043.8.498.12664787797674809843666198835185372352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91611403539216247154562810076063878924","series":"1.2.826.0.1.3680043.8.498.47643916648162020260428587232275230255","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91686669332744972709173289374472094988","series":"1.2.826.0.1.3680043.8.498.19724613169950207175560691049346171894","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91742054227008690570256869200116133922","series":"1.2.826.0.1.3680043.8.498.56827380780001508988526833983053752590","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91581717063042044543371480238854954692","series":"1.2.826.0.1.3680043.8.498.66710002167764379168595820005561732878","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91839696756303525362262251587171486273","series":"1.2.826.0.1.3680043.8.498.64104806459673864498051173908595386301","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91883829404782805058767342767479130421","series":"1.2.826.0.1.3680043.8.498.45596264959713117306665854340586903629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91431012289716035498822977054491790785","series":"1.2.826.0.1.3680043.8.498.12617065960727660092202448434238927620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91940000162714863613139612302783091395","series":"1.2.826.0.1.3680043.8.498.69684383699796665852377059310338342970","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92063841381005962285592592672311044745","series":"1.2.826.0.1.3680043.8.498.10188583795836242313580054811560915546","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92126358259118823397932994812520542494","series":"1.2.826.0.1.3680043.8.498.90638015107373900168428245428289321145","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92222956609345084635188178384706987675","series":"1.2.826.0.1.3680043.8.498.73758050659621838380657253423368695892","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92285140961160393284537044926797259671","series":"1.2.826.0.1.3680043.8.498.11560961549203372192371354832108194781","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91910585599088493175245713843727208369","series":"1.2.826.0.1.3680043.8.498.86977750249422633435884400426298809611","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91991985280680856830060431668414014465","series":"1.2.826.0.1.3680043.8.498.89946991196444802849938805122990022129","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92455257597708674644536154732642099964","series":"1.2.826.0.1.3680043.8.498.16140878781500950046208391210783243381","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92407834172018110142589158281575319154","series":"1.2.826.0.1.3680043.8.498.86421273711917246726993816513365063433","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92326467051364762529119362944559591665","series":"1.2.826.0.1.3680043.8.498.72221254096582024026943543519167294543","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92621720172216795135762712303124932801","series":"1.2.826.0.1.3680043.8.498.10601904997268302114806430517578049330","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92445779773398371452908052305534240654","series":"1.2.826.0.1.3680043.8.498.11717758822329038527283326907609126309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92650426727723557280117118587545413917","series":"1.2.826.0.1.3680043.8.498.11754928391125387454556677525657641157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92682287574388138675492893420192905780","series":"1.2.826.0.1.3680043.8.498.78413390359460676641323045800946869095","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92839397144213404004277739877191283401","series":"1.2.826.0.1.3680043.8.498.24418143209545565410439845299592443114","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92597065276819676534178545455668171085","series":"1.2.826.0.1.3680043.8.498.84530932792481213677732918796127123105","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92902692647544828045768164058889405369","series":"1.2.826.0.1.3680043.8.498.10642965759110860280654205173748389373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93062749017302143105207135454940053142","series":"1.2.826.0.1.3680043.8.498.41701181508443960878501721200563391277","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92722393862174380736027353110424417588","series":"1.2.826.0.1.3680043.8.498.49747070518691990233225016892273807229","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93104685852024334015651558514195715956","series":"1.2.826.0.1.3680043.8.498.23018397931728158376889231807046167421","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93026010065162534828966791702007067882","series":"1.2.826.0.1.3680043.8.498.80975820817963017410377771614202394025","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93188773180403601524488403580611534053","series":"1.2.826.0.1.3680043.8.498.80762101453077991639350335848305776483","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93230313557251171119475173996942942855","series":"1.2.826.0.1.3680043.8.498.51611137538319854326864290108043801788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93318293884671598055223778443354732793","series":"1.2.826.0.1.3680043.8.498.12198695651650740893635385424108083336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93177444420568709574758402836371200945","series":"1.2.826.0.1.3680043.8.498.22783580283484824925167187725047582409","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93359387652821116106937882388838959209","series":"1.2.826.0.1.3680043.8.498.46066982789403183951797581146886365682","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93478919245462741704531324128061533679","series":"1.2.826.0.1.3680043.8.498.56513509043748142409976562923986909327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93623421864452751381778712192879121271","series":"1.2.826.0.1.3680043.8.498.11433906251449407241842407953085675364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93655619137163635289231840294252859840","series":"1.2.826.0.1.3680043.8.498.24024923226226294642373075936253282669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93719148934772792523193842156217224487","series":"1.2.826.0.1.3680043.8.498.10909415160385104524399857124944945957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93774667493652492119129601342627338680","series":"1.2.826.0.1.3680043.8.498.93768937867332880545862590444657449090","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93976819306514730278115540117878688308","series":"1.2.826.0.1.3680043.8.498.92326256981209525105850367169487903897","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93902628119199416703899320994283313812","series":"1.2.826.0.1.3680043.8.498.20417575944064446496275915763026594189","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94002633501154835018549300667479350401","series":"1.2.826.0.1.3680043.8.498.40641392621838956864387744252102582154","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93728117237268000947508816107096388086","series":"1.2.826.0.1.3680043.8.498.21019285569066037405691124338811870024","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94274898137310893945218731717226819541","series":"1.2.826.0.1.3680043.8.498.97021433364468673150298035320637751406","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94030816415316308210133175314873817171","series":"1.2.826.0.1.3680043.8.498.98681845275725277924564118051795709602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94117540970936035429846288243624086693","series":"1.2.826.0.1.3680043.8.498.22118248635648893752981143165881977818","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94328216642444410827846344286227627425","series":"1.2.826.0.1.3680043.8.498.84420115376825978347075234927792475548","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94381282809153270156987123429739805206","series":"1.2.826.0.1.3680043.8.498.48623507801326297080721776794137413282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312","series":"1.2.826.0.1.3680043.8.498.81238513949482991110216640426071715261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94448720962054029575571654250994769402","series":"1.2.826.0.1.3680043.8.498.27708621160035305002517660383748422115","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94601583997736688762302974573903982304","series":"1.2.826.0.1.3680043.8.498.97733180703175461238247399429703196558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94590838906158914501563687877014778914","series":"1.2.826.0.1.3680043.8.498.11884459390999248285375843143460248711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94658317716039594639343437780115140531","series":"1.2.826.0.1.3680043.8.498.25909945610099659963463302260548449575","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94625454484290996254750481989036812516","series":"1.2.826.0.1.3680043.8.498.11721550529197630725102074533687154813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94252884278314596961426629318911909075","series":"1.2.826.0.1.3680043.8.498.83800650880078783193342720503385776025","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94722363227977785887442194094888460046","series":"1.2.826.0.1.3680043.8.498.80255358491007732464688896616990144000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94761755148413531477206243804832613511","series":"1.2.826.0.1.3680043.8.498.10545411945115490057201637360889921343","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94747403767627336827372096100476526821","series":"1.2.826.0.1.3680043.8.498.15948882860391265334553633455338718725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94798723424795346945893598929864335472","series":"1.2.826.0.1.3680043.8.498.87310061282441705205910885328394403442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94921929374845793599006362190119159537","series":"1.2.826.0.1.3680043.8.498.21978189593939718765390300892765894419","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94945670991495145462605178471390637325","series":"1.2.826.0.1.3680043.8.498.11246787968691220542644345658321036350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94955017957653373798729896600270551224","series":"1.2.826.0.1.3680043.8.498.87159653406138401833165972154176467671","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95030266788619257057351879256576338162","series":"1.2.826.0.1.3680043.8.498.15228082650758146979970012381023361974","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95113114010658169441709314948308683982","series":"1.2.826.0.1.3680043.8.498.82170306501491432359492283202183074007","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95080449640993916851075185393444497308","series":"1.2.826.0.1.3680043.8.498.29140728774502280602335429582215835064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95262231945171099723559613554350591139","series":"1.2.826.0.1.3680043.8.498.60630256675037930625027205978416899264","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95193282490279394602564266574551123732","series":"1.2.826.0.1.3680043.8.498.48784453866755145298489508587427686942","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95324318295506605557933083155792587297","series":"1.2.826.0.1.3680043.8.498.91907687764799591889428845744135442364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95351925765761617117057799846763403963","series":"1.2.826.0.1.3680043.8.498.36236344117030528678128162382538381578","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95433650759419050864053270236035947981","series":"1.2.826.0.1.3680043.8.498.10984947269636505605889762116206618013","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95490888360439852685783476266753304657","series":"1.2.826.0.1.3680043.8.498.43922387477864921634576437308922915827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95500234888498554736637315199668317948","series":"1.2.826.0.1.3680043.8.498.61460747123326739291416140748773135970","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95586477246200285088746818821808017494","series":"1.2.826.0.1.3680043.8.498.12400856454728899729788270854036261680","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310","series":"1.2.826.0.1.3680043.8.498.10083266779904038810871198421942987350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95223549428749744711279985897241419512","series":"1.2.826.0.1.3680043.8.498.30415673915416959687544702139793304346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95638601079527306640829424921455011698","series":"1.2.826.0.1.3680043.8.498.52724725573185441368489078557707456000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95601371640669355936785637162110963091","series":"1.2.826.0.1.3680043.8.498.11840091397064461153576700724937879703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95694598580744163428934772566469311180","series":"1.2.826.0.1.3680043.8.498.12360071282752382268132591041045036928","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95720534093121754146413060811354489247","series":"1.2.826.0.1.3680043.8.498.79797119996879893771626825369470988124","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95796009319232592137564145023096016872","series":"1.2.826.0.1.3680043.8.498.91535539374765371519019778653044879605","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95912798959818676102590029534562878721","series":"1.2.826.0.1.3680043.8.498.10813873122177068362271807875568066551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95864678405973312042815189771902289608","series":"1.2.826.0.1.3680043.8.498.66186372318043181447660758632742702302","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95728724960383464194568391302477328450","series":"1.2.826.0.1.3680043.8.498.12875235954099742080676128469286875183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716","series":"1.2.826.0.1.3680043.8.498.59284431425955191432887398972281858418","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96054287426681123062488435620352206601","series":"1.2.826.0.1.3680043.8.498.85947819447886582244691863490635344275","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466","series":"1.2.826.0.1.3680043.8.498.11594307268832409644859362083140791178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96152894373838410779390571000523933753","series":"1.2.826.0.1.3680043.8.498.12524046139486476803740432065693520062","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96174494699876332875347380499122120925","series":"1.2.826.0.1.3680043.8.498.10162798772737166441562271795844360686","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96316229168827165128604998083731481211","series":"1.2.826.0.1.3680043.8.498.74967836545631035344732225210587926253","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96425049974772570897047688166336162674","series":"1.2.826.0.1.3680043.8.498.42494337424243151736819852958626134171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96559819562317669526803257835398470484","series":"1.2.826.0.1.3680043.8.498.11771964033775692989829371057701639364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96587029379314618196381307882235433083","series":"1.2.826.0.1.3680043.8.498.93708708835451841177483184780917113759","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96483513352751590579274577125356217169","series":"1.2.826.0.1.3680043.8.498.23704487503074026760356291542562750203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96562865861483086790615358719391771217","series":"1.2.826.0.1.3680043.8.498.84680945299599438679895350240053682799","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96621557839848412555023112236627663339","series":"1.2.826.0.1.3680043.8.498.12237747273148367452322201436765693941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96791010280967772394523567484937888186","series":"1.2.826.0.1.3680043.8.498.75738321577460165994754105481540776103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96943946564115188893954628924857155670","series":"1.2.826.0.1.3680043.8.498.50814910290590350596337772983494519669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96846796615932368166905347054245013501","series":"1.2.826.0.1.3680043.8.498.43072334231375784974622543881256743862","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96947237995559790704304892745226362883","series":"1.2.826.0.1.3680043.8.498.30694089578625983397169606170273856477","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97003157807630958179276831877591411623","series":"1.2.826.0.1.3680043.8.498.70090773720439964590420889071497856339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97055845147740306780908051236353219644","series":"1.2.826.0.1.3680043.8.498.16249121177238417008267739907938555632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97234348688223610934999923021953923903","series":"1.2.826.0.1.3680043.8.498.13030826860141949143201384510936206638","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97107472223824971578324157203273471706","series":"1.2.826.0.1.3680043.8.498.38537516654266349046383951071259436224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97020520267554584684160936968569132254","series":"1.2.826.0.1.3680043.8.498.55257447526692761559371316787077701273","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97259766233250760343109933426776941754","series":"1.2.826.0.1.3680043.8.498.98468686598996674524849524337928915087","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97156518925820521351080132062400795929","series":"1.2.826.0.1.3680043.8.498.78147533198968392493604340551936679011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97269744864161337523188705262974234442","series":"1.2.826.0.1.3680043.8.498.12871648191974633893651549257412844086","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97274847570547462056218828853253311223","series":"1.2.826.0.1.3680043.8.498.62831547519063846873638985755324873717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97353980946516912216109392482235435606","series":"1.2.826.0.1.3680043.8.498.28558413399290447454378870118165835912","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97464332941382646555656038793589898504","series":"1.2.826.0.1.3680043.8.498.54167095945326954985687702914881517845","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97303893645051596044169857585815133097","series":"1.2.826.0.1.3680043.8.498.17318459089992955894661739796611956498","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97500632594529766269369977508352975364","series":"1.2.826.0.1.3680043.8.498.13115249201544999166524459450098729467","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97536230315600409983600607358927775480","series":"1.2.826.0.1.3680043.8.498.24848051499273874677843037512413445902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97658595748192892503053963872409769829","series":"1.2.826.0.1.3680043.8.498.46376039048482306312882374427669185509","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97579738268290461886850261405719501384","series":"1.2.826.0.1.3680043.8.498.86612978937797429850646575476186348723","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97694479664965569386338738579573728752","series":"1.2.826.0.1.3680043.8.498.26845490368277485306507464450489178609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97705920250069816684861966612302390614","series":"1.2.826.0.1.3680043.8.498.44817545497803689736438377387228762222","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97759980486481979519068526243106370997","series":"1.2.826.0.1.3680043.8.498.10028090115069501801858560715672119975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97830923192341890652952842494018952920","series":"1.2.826.0.1.3680043.8.498.12949366467981045900243975360840451907","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97488501634729541923597957717971376373","series":"1.2.826.0.1.3680043.8.498.26828897492236540057121610853830513339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97964867333695657200856428936254426086","series":"1.2.826.0.1.3680043.8.498.26615630349827384167064411047173410153","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98039072306618106705944452906395982381","series":"1.2.826.0.1.3680043.8.498.13157344127524195577913855068918142676","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98027050915885651900992366068887550187","series":"1.2.826.0.1.3680043.8.498.11666100137350802296683566507752451848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97909881548577311929907613359449243075","series":"1.2.826.0.1.3680043.8.498.56965554777032067277083643721872941331","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98081486779628580946589119341754353354","series":"1.2.826.0.1.3680043.8.498.35240239757380081791604122065164270058","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98151095279541278201351784048265686573","series":"1.2.826.0.1.3680043.8.498.91963478824768702319618334153956929993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98107112397588450345153336193341117589","series":"1.2.826.0.1.3680043.8.498.17794462715156055975994646758574924012","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98143909986671737130865120569673002676","series":"1.2.826.0.1.3680043.8.498.40449769170768949325318643683678731262","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98239758242192048511912905088391835938","series":"1.2.826.0.1.3680043.8.498.21369274140073378305196974113902999112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98315778542448798493600485116505234132","series":"1.2.826.0.1.3680043.8.498.81212289294690585293990567610053283976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98343185597651455245996667389303915779","series":"1.2.826.0.1.3680043.8.498.10649946530349420302652320954747006492","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98541661035340329000785741171799559104","series":"1.2.826.0.1.3680043.8.498.10305911785606743174477254639064677682","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98599322992880180540967962062521615070","series":"1.2.826.0.1.3680043.8.498.22105365422429735919285731188087132336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98752901295449528283322564256444025625","series":"1.2.826.0.1.3680043.8.498.13001918693714788391370641863004653673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98795677271204502833293873138219040186","series":"1.2.826.0.1.3680043.8.498.26363771796184774509366499780335946951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98828753472895685940998888332947119084","series":"1.2.826.0.1.3680043.8.498.20620456124233467817883834013681702636","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98914528123216365374152816424181510781","series":"1.2.826.0.1.3680043.8.498.11635002551691881354524444251146559471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99052905406751383674032587504235261874","series":"1.2.826.0.1.3680043.8.498.88885378327392424537627274974646887651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98400072082366198211931728015692551635","series":"1.2.826.0.1.3680043.8.498.80381214158624030220302746809668405924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99088247184917460283293617180083591028","series":"1.2.826.0.1.3680043.8.498.92252780641237231414742159579199816328","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98883718766501717651651876978607853803","series":"1.2.826.0.1.3680043.8.498.66239374265853329797447949310205204101","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99155175553749543768821558247962844052","series":"1.2.826.0.1.3680043.8.498.76935021942064442947235829291758509169","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99257151613846084492342083136797222759","series":"1.2.826.0.1.3680043.8.498.10482024555490761490529417956932095632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99309981642816644502709148844743995902","series":"1.2.826.0.1.3680043.8.498.11787479686216007522970084237151694286","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99237666579371266866432031468578336182","series":"1.2.826.0.1.3680043.8.498.13123125414369112094379605522982417395","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99441773159476751698123795913150660766","series":"1.2.826.0.1.3680043.8.498.11899368256839738621553100899022016458","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99458163491875066451936719024359566928","series":"1.2.826.0.1.3680043.8.498.11820254473623635640942895920851010315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99448612525108835020450967944392129536","series":"1.2.826.0.1.3680043.8.498.63045660500409978193936897248497748366","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99414163787650487720542840540379270468","series":"1.2.826.0.1.3680043.8.498.76513865385287705529742743578415943576","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99578894341571394220300473118855007095","series":"1.2.826.0.1.3680043.8.498.87290613282201687267019846861744769872","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99526998617163591447503530185291775961","series":"1.2.826.0.1.3680043.8.498.34646978966919327007874770608641230938","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99726731706411379395165830480890405115","series":"1.2.826.0.1.3680043.8.498.10731060894454049508761911820195549305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99813269006669714758887909212386431934","series":"1.2.826.0.1.3680043.8.498.55075381196663522697202704870355310884","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99651725851820083142440965238449655973","series":"1.2.826.0.1.3680043.8.498.38135480996575725720494083119227716539","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99864558747514747686151806909794288262","series":"1.2.826.0.1.3680043.8.498.10651063225290069440597914902724437752","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99827460918477409069820779642161448917","series":"1.2.826.0.1.3680043.8.498.84111703816917031730631015620429748318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99710592666731203419714499168506345263","series":"1.2.826.0.1.3680043.8.498.13335220307995571884694301831854539909","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99881620336512626242732899010111041290","series":"1.2.826.0.1.3680043.8.498.99238949500125352248111605622199971709","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99939100657291954890286900729983287973","series":"1.2.826.0.1.3680043.8.498.46699396562036490964108080683282499372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"}]')
print('series a convertir (UID figes depuis crop130 224) :', len(jobs))
assert len(jobs) == 2202
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    # pad/crop spatial identique : si shapes divergent (rare), resize au mode
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w)

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    band = vol[a:b + 1]
    idx = np.linspace(0, band.shape[0] - 1, N_SLICES).round().astype(int)
    band = band[idx]

    out = np.empty((N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for i in range(N_SLICES):
        sl = cv2.resize(band[i], (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        out[i] = np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    # fallback : parfois series a la racine de train_series
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_320_t1_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / "manifest_t1_part1.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "note": "1 serie T1/non-fluide extra, UID figes depuis crop130 224. Crop 130 mm (PixelSpacing) sinon centre 70 %. Bande profondeur 20-80 %, linspace 24, resize 320, uint8.",
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

# ZipFile n'est pas thread-safe en ecriture : workers convertissent, le
# thread principal ecrit le zip.
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

# Sanity : recharger 3 NPZ, shapes, moments
import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    means = []
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        print(name, vol.shape, vol.dtype, "min", int(vol.min()), "max", int(vol.max()),
              "mean", float(vol.mean()), "std", float(vol.std()))
        assert vol.shape == (N_SLICES, IMG_SIZE, IMG_SIZE), vol.shape
        means.append(float(vol.mean()))
print("mean uint8 (3 samples)", means)
print(f"attendu : shape ({N_SLICES}, {IMG_SIZE}, {IMG_SIZE}) uint8")
print("cible historique NPZ communautaires : mean~42 std~49.5 (crop 130 mm => moments differents, c'est normal)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
